# DiT Generalization and Physical-Validity Diagnostics

Reader-facing analysis of the Fig.2-style DiT depth sweep. The notebook keeps three questions separate:

1. **Optimization:** did the denoising objective decrease?
2. **Novelty:** are generated maps unlike their nearest training examples?
3. **Physical agreement:** do generated maps reproduce the one-point distribution and power spectrum?

A model is useful only when all three checks are interpreted together.


## tl;dr

- DiT-L8 and DiT-L12 show a recognizable shift from training-neighbor behavior toward novel samples as the training set grows.
- The original DiT-L16 curve is anomalous at small data sizes. Some samples receive high PCA or SSCD novelty scores while visibly failing the power-spectrum check. Those points are **novel but not physically valid**.
- Training loss alone does not resolve this: DiT-L16 can fit the denoising objective well while generating statistically incorrect fields.
- The black one-point and power-spectrum references come from the **exact training subset configured for each model**, not the full CAMELS collection.
- The clean DiT-L16 300k sweep is the appropriate replacement experiment. Until it is complete, the notebook does not claim a depth-capacity scaling law.


## Nick review checklist

This notebook answers the requested review questions directly:

1. **Nearest training examples:** generated maps are compared with nearest slices from the complete training subset configured for each model, in both pixel and SSCD spaces.
2. **Black physical-statistics reference:** every one-point and $P(k)$ black curve is rebuilt from every slice in that model's exact training subset. It is not the full CAMELS collection and is not capped for plotting.
3. **In-distribution check:** SSCD Fréchet distance is normalized by a real-vs-real split baseline, so novel but out-of-distribution samples can be separated from useful generalization. This is an FID-style distribution check in domain-relevant SSCD features, not literal ImageNet Inception FID.
4. **Full data-size coverage:** generated maps, one-point PDFs, and $P(k)$ errors are shown for $2^6$ through $2^{15}$ for DiT-L8, DiT-L12, and DiT-L16.
5. **Conditional-input provenance:** the calibration appendix verifies all six CAMELS parameters, held-out manifest alignment, and disjoint training/test simulation indices before displaying recovery results.


## Setup


In [ ]:
from __future__ import annotations

import json
import math
import os
import re
import sys
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
from matplotlib.lines import Line2D
from IPython.display import Image, Markdown, display


def parse_csv_env(name: str, default: str) -> list[str]:
    return [x.strip() for x in os.environ.get(name, default).split(',') if x.strip()]


def find_project_dir() -> Path:
    env = os.environ.get('DIFFUSION_PROJECT_DIR')
    if env:
        return Path(env).expanduser().resolve()
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / 'scripts').exists() and (candidate / 'notebooks').exists():
            return candidate
    return here


PROJECT_DIR = find_project_dir()
SWEEP_NAME = 'nf_generalize_fig2_dit'
RESULTS_DIR = PROJECT_DIR / 'results' / SWEEP_NAME
TABLE_DIR = RESULTS_DIR / 'tables'
QUICKCHECK_DIR = RESULTS_DIR / 'quickcheck'
SAMPLE_DIR = RESULTS_DIR / 'samples'
MANIFEST_PATH = PROJECT_DIR / 'local' / SWEEP_NAME / 'manifest.json'
SAMPLE_LABEL = os.environ.get('SAMPLE_LABEL', 'dpm50')
SEED = int(os.environ.get('SEED', '123'))

ALL_DATA_TAGS = [f'd2p{i:02d}' for i in range(6, 16)]
LOW_DATA_TAGS = ALL_DATA_TAGS[:5]
HIGH_DATA_TAGS = ALL_DATA_TAGS[5:]
DATA_TAG_BLOCKS = {
    'low_transition': LOW_DATA_TAGS,
    'high_data': HIGH_DATA_TAGS,
}

DIT_ARCH_ORDER = ['dit_l8', 'dit_base', 'dit_l16']
DIT_ARCH_LABELS = {
    'dit_l8': 'DiT-L8',
    'dit_base': 'DiT-L12 / base',
    'dit': 'DiT-L12 / base',
    'dit_l16': 'DiT-L16',
}
DIT_ARCH_COLORS = {
    'dit_l8': '#009E73',
    'dit_base': '#0072B2',
    'dit': '#0072B2',
    'dit_l16': '#CC79A7',
}
DIT_ARCH_MARKERS = {'dit_l8': 'P', 'dit_base': 'D', 'dit': 'D', 'dit_l16': 'X'}
UNET_ARCH_ORDER = ['u64', 'u128', 'u256']
UNET_ARCH_LABELS = {'u64': 'UNet-64', 'u128': 'UNet-128', 'u256': 'UNet-256'}
UNET_ARCH_COLORS = {'u64': '#009E73', 'u128': '#D55E00', 'u256': '#0072B2'}
UNET_ARCH_MARKERS = {'u64': '^', 'u128': 'o', 'u256': 's'}

DETAIL_ARCH = os.environ.get('DIT_DETAIL_ARCH', 'dit_base')
IMAGE_ARCH = os.environ.get('DIT_IMAGE_ARCH', DETAIL_ARCH)
LOSS_ARCHES = parse_csv_env('DIT_LOSS_ARCHES', 'dit_l8,dit_base,dit_l16')
LOSS_TAGS = parse_csv_env('DIT_LOSS_TAGS', 'd2p06,d2p10,d2p15')
DETAIL_TAGS = parse_csv_env('DIT_DETAIL_TAGS', 'd2p06,d2p08,d2p10,d2p12,d2p15')
IMAGE_TAGS = parse_csv_env('DIT_IMAGE_TAGS', 'd2p06,d2p08,d2p10,d2p12,d2p15')

plt.rcParams.update({
    'figure.dpi': 130,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'font.family': 'sans-serif',
    'font.sans-serif': ['DejaVu Sans'],
    'mathtext.fontset': 'dejavusans',
    'savefig.dpi': 300,
    'font.size': 14,
    'axes.labelsize': 15,
    'axes.titlesize': 16,
    'legend.fontsize': 12,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
})

print('PROJECT_DIR =', PROJECT_DIR)
print('RESULTS_DIR =', RESULTS_DIR)
print('SAMPLE_LABEL =', SAMPLE_LABEL)
print('SEED =', SEED)
print('DETAIL_ARCH =', DETAIL_ARCH, 'IMAGE_ARCH =', IMAGE_ARCH)

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

try:
    from simdiff_eval.io import (
        as_nchw, configured_training_reference_info, iter_real_reference_batches_from_config,
        load_real_from_config, load_real_reference_from_config,
    )
    from simdiff_eval.metrics import batch_power_spectra, field_histogram, frechet_feature_distance, nearest_training_matches, real_split_frechet_baseline
    SIMDIFF_EVAL_AVAILABLE = True
except Exception as exc:
    SIMDIFF_EVAL_AVAILABLE = False
    SIMDIFF_EVAL_ERROR = repr(exc)
    print('simdiff_eval unavailable; image/P(k) diagnostics will be skipped:', SIMDIFF_EVAL_ERROR)

PK_NBINS = int(os.environ.get('DIT_PK_NBINS', '30'))
MAX_GENERATED = int(os.environ.get('DIT_MAX_GENERATED', '512'))
MAX_REAL_REFERENCE_SLICES = int(os.environ.get('DIT_MAX_REAL_REFERENCE_SLICES', '2048'))

## Context & Methods

These runs use Hugging Face diffusers `DiTTransformer2DModel` with a single null class label (`class_labels=0`) because the DiT adaLN path requires labels even for unconditional training. The label is constant, so this is still an unconditional Fig.2-style sweep.

The depth sweep is:

- `DiT-L8`: 8 transformer blocks, width 768.
- `DiT-L12/base`: the original DiT-base run, 12 transformer blocks, width 768.
- `DiT-L16`: 16 transformer blocks, width 768.

Each model is trained at the same 2D training-set sizes as the UNet Fig.2 sweep, then sampled with DPM-Solver 50 steps. The diagnostics below ask whether deeper DiTs need more data before generated fields stop looking like nearest training slices.

### How to read the figures

| Diagnostic | Question | Good result | Important limitation |
|---|---|---|---|
| Training loss | Is the optimizer fitting the denoising objective? | Loss decreases and remains finite | Low loss can coexist with memorization or bad physical statistics |
| Nearest training image | Is a generated field a near-copy? | Difference image is structured rather than nearly blank | Pixel distance is sensitive to shifts and does not measure distributional validity |
| PCA / SSCD q95 novelty | Is the generated set unusually close to training examples? | Score increases away from zero | A high score can also come from out-of-distribution noise |
| SSCD Fréchet ratio | Is the generated distribution close to heldout real maps? | Ratio near the real-vs-real baseline of one | Representation-dependent; not a replacement for physical statistics |
| One-point PDF | Are pixel values distributed correctly? | Generated and black curves overlap | Ignores spatial arrangement |
| Power spectrum | Is spatial structure correct across scale? | Generated/real ratio stays near one | Does not test every higher-order statistic |

The notebook therefore reads from **optimization**, to **visual copy checks**, to **distribution and physical-statistics checks**, and only then to the generalization curves.

## Data Audit


In [ ]:
def read_json(path: Path) -> Any | None:
    if not path.exists():
        return None
    with path.open() as f:
        return json.load(f)


def rel(path: Path | str | None) -> str:
    if path is None:
        return ''
    path = Path(path)
    try:
        return str(path.relative_to(PROJECT_DIR))
    except ValueError:
        return str(path)


def dataset_tag_from_name(name: str) -> str | None:
    m = re.search(r'd2p(\d+)', str(name))
    if not m:
        return None
    return 'd2p' + m.group(1)


def dataset_size_from_tag(tag: str | None) -> int | None:
    if not tag:
        return None
    m = re.match(r'd2p(\d+)', tag)
    if not m:
        return None
    return 2 ** int(m.group(1))


def arch_from_run_name(name: str) -> str:
    text = str(name)
    for arch in ['dit_l16', 'dit_l8', 'dit_base', 'u256', 'u128', 'u64']:
        if arch in text:
            return arch
    if re.search(r'\bdit\b', text):
        return 'dit_base'
    return 'unknown'


def arch_label(raw: Any) -> str:
    text = str(raw)
    if text in DIT_ARCH_LABELS:
        return DIT_ARCH_LABELS[text]
    if text in UNET_ARCH_LABELS:
        return UNET_ARCH_LABELS[text]
    return text


def ensure_arch_columns(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    out = df.copy()
    if 'arch' not in out.columns:
        name_col = 'run_name' if 'run_name' in out.columns else ('name' if 'name' in out.columns else None)
        out['arch'] = out[name_col].map(arch_from_run_name) if name_col else 'unknown'
    else:
        out['arch'] = out['arch'].astype(str).replace({'dit': 'dit_base'})
    if 'arch_label' not in out.columns:
        out['arch_label'] = out['arch'].map(arch_label)
    return out


def dataset_size_label(n: int | float) -> str:
    n = int(n)
    log2n = np.log2(n)
    if np.isfinite(log2n) and abs(log2n - round(log2n)) < 1e-9:
        return rf'$2^{{{int(round(log2n))}}}$'
    return f'{n:,}'


def sample_path_for(row: pd.Series) -> Path:
    raw = str(row.get('sample_path', '') or '')
    if raw:
        raw = raw.format(seed=SEED, sample_label=SAMPLE_LABEL)
        path = Path(raw)
        return path if path.is_absolute() else PROJECT_DIR / path
    run_name = row.get('run_name') or row.get('name')
    arch = row.get('arch') or arch_from_run_name(str(run_name))
    tag = row.get('dataset_tag') or dataset_tag_from_name(str(run_name))
    if tag is None:
        return SAMPLE_DIR / f'unknown_seed{SEED}_{SAMPLE_LABEL}.npz'
    return SAMPLE_DIR / f'nf_fig2_{arch}_{tag}_noaug_200k_seed{SEED}_{SAMPLE_LABEL}.npz'


manifest_obj = read_json(MANIFEST_PATH)
if manifest_obj is None:
    display(Markdown(f'**Missing manifest:** `{rel(MANIFEST_PATH)}`'))
    manifest_df = pd.DataFrame()
else:
    if isinstance(manifest_obj, dict):
        rows = manifest_obj.get('runs', [])
    elif isinstance(manifest_obj, list):
        rows = manifest_obj
    else:
        rows = []
    manifest_df = pd.DataFrame(rows)
    if 'run_name' not in manifest_df.columns and 'name' in manifest_df.columns:
        manifest_df['run_name'] = manifest_df['name']
    manifest_df = ensure_arch_columns(manifest_df)
    if 'dataset_tag' not in manifest_df.columns:
        manifest_df['dataset_tag'] = manifest_df['run_name'].map(dataset_tag_from_name)
    if 'dataset_size' not in manifest_df.columns:
        manifest_df['dataset_size'] = manifest_df['dataset_tag'].map(dataset_size_from_tag)
    manifest_df['sample_path_resolved'] = manifest_df.apply(sample_path_for, axis=1)
    manifest_df['sample_exists'] = manifest_df['sample_path_resolved'].map(Path.exists)
    manifest_df['sample_size_mb'] = manifest_df['sample_path_resolved'].map(lambda p: p.stat().st_size / 1024**2 if p.exists() else np.nan)

    show_cols = [c for c in [
        'arch', 'arch_label', 'run_name', 'dataset_tag', 'dataset_size', 'sample_exists', 'sample_size_mb',
        'config_path', 'output_dir', 'sample_path_resolved'
    ] if c in manifest_df.columns]
    display(manifest_df.sort_values(['arch', 'dataset_size'])[show_cols])
    print(f"sample files present: {manifest_df['sample_exists'].sum()} / {len(manifest_df)}")

expected_tables = [
    TABLE_DIR / 'nf_generalize_fig2_dit_pca_full_nn_metrics.csv',
    TABLE_DIR / 'nf_generalize_fig2_dit_pca_full_nn_mode_norms.csv',
    TABLE_DIR / 'nf_generalize_fig2_dit_pca_full_nn_similarity_histograms.csv',
    TABLE_DIR / 'nf_generalize_fig2_dit_sscd_full_nn_metrics.csv',
]
expected_figures = [
    QUICKCHECK_DIR / 'nf_generalize_fig2_dit_pca_full_nn_paper_style_gl_curves.png',
    QUICKCHECK_DIR / 'nf_generalize_fig2_dit_sscd_full_nn_paper_style_gl_curves.png',
    QUICKCHECK_DIR / 'nf_generalize_fig2_dit_pca_full_nn_similarity_curves.png',
    QUICKCHECK_DIR / 'nf_generalize_fig2_dit_sscd_full_nn_similarity_curves.png',
    QUICKCHECK_DIR / 'nf_generalize_fig2_dit_pca_full_nn_copy_fraction_curves.png',
    QUICKCHECK_DIR / 'nf_generalize_fig2_dit_sscd_full_nn_copy_fraction_curves.png',
]

audit = pd.DataFrame({
    'path': [rel(p) for p in expected_tables + expected_figures],
    'kind': ['table'] * len(expected_tables) + ['figure'] * len(expected_figures),
    'exists': [p.exists() for p in expected_tables + expected_figures],
    'size_mb': [p.stat().st_size / 1024**2 if p.exists() else np.nan for p in expected_tables + expected_figures],
})
display(audit)

## Load Metrics


In [ ]:
def read_csv_if_exists(path: Path) -> pd.DataFrame:
    if not path.exists():
        display(Markdown(f'**Missing table:** `{rel(path)}`'))
        return pd.DataFrame()
    df = pd.read_csv(path)
    print(f'loaded {rel(path)}: {len(df)} rows, {len(df.columns)} columns')
    return df


def add_generalization_columns(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    df = ensure_arch_columns(df)
    for q in ['q50', 'q68', 'q90', 'q95', 'q99']:
        copy_col = f'gen_copy_fraction_{q}'
        gl_col = f'gen_gl_{q}'
        if gl_col not in df.columns and copy_col in df.columns:
            df[gl_col] = 1.0 - df[copy_col]
    # Some older tables used shorter names.
    for q in ['q90', 'q95', 'q99']:
        if f'gen_gl_{q}' not in df.columns and f'copy_fraction_{q}' in df.columns:
            df[f'gen_gl_{q}'] = 1.0 - df[f'copy_fraction_{q}']
    if 'dataset_tag' not in df.columns:
        name_col = 'run_name' if 'run_name' in df.columns else df.columns[0]
        df['dataset_tag'] = df[name_col].map(dataset_tag_from_name)
    if 'dataset_size' not in df.columns:
        df['dataset_size'] = df['dataset_tag'].map(dataset_size_from_tag)
    return df


pca_metrics = add_generalization_columns(read_csv_if_exists(TABLE_DIR / 'nf_generalize_fig2_dit_pca_full_nn_metrics.csv'))
sscd_metrics = add_generalization_columns(read_csv_if_exists(TABLE_DIR / 'nf_generalize_fig2_dit_sscd_full_nn_metrics.csv'))

for feature_name, df in [('PCA', pca_metrics), ('SSCD', sscd_metrics)]:
    display(Markdown(f'### {feature_name} metrics'))
    if df.empty:
        continue
    print('architectures:', sorted(df['arch'].dropna().unique()) if 'arch' in df.columns else 'missing')
    print('generalization columns:', [c for c in df.columns if c.startswith('gen_gl')])
    preferred = [
        'arch_label', 'run_name', 'dataset_tag', 'dataset_size', 'n_generated', 'n_train',
        'gen_gl_q90', 'gen_gl_q95', 'gen_gl_q99',
        'gen_copy_fraction_q95', 'threshold_q95', 'gen_nn_median', 'gen_nn_q95'
    ]
    cols = [c for c in preferred if c in df.columns]
    display(df.sort_values(['arch', 'dataset_size'])[cols] if cols else df.head())

## Training Loss Curves

This compares model depths at the same training-set size. DiT-L8 and DiT-L12 use their original 200k-update histories. For DiT-L16 at $N_{2D}=2^6,\ldots,2^{10}$, the plot uses the verified stage-4 continuation history through 300k updates; DiT-L16 falls back to 200k where no continuation exists. The curves are averaged over one 4,000-update learning-rate restart cycle for readability. Lower training loss can indicate easier fitting or memorization; it does not by itself establish sample quality or physical fidelity.


In [ ]:
def checkpoint_epoch(path: Path) -> int | None:
    m = re.search(r'checkpoint-epoch-(\d+)', str(path))
    return int(m.group(1)) if m else None


def metric_candidates(row: pd.Series) -> list[Path]:
    root = Path(str(row.get('checkpoint_dir', '') or ''))
    paths: list[Path] = []
    if root.exists():
        paths.extend(sorted(root.glob('metrics_epoch_*.json')))
        metrics_json = root / 'metrics.json'
        if metrics_json.exists():
            paths.append(metrics_json)
        for ckpt in sorted(root.glob('checkpoint-epoch-*')):
            paths.extend(sorted(ckpt.glob('metrics*.json')))
    return paths


def flatten_numeric(values: Any) -> np.ndarray:
    if values is None:
        return np.asarray([], dtype=float)
    out: list[float] = []

    def visit(x: Any) -> None:
        if x is None:
            return
        if isinstance(x, dict):
            for key in ('loss', 'value', 'mean', 'avg'):
                if key in x:
                    visit(x[key])
                    return
            return
        if isinstance(x, (list, tuple, np.ndarray)):
            for item in x:
                visit(item)
            return
        try:
            out.append(float(x))
        except (TypeError, ValueError):
            return

    visit(values)
    return np.asarray(out, dtype=float)


def read_latest_metrics(row: pd.Series) -> tuple[dict[str, Any], Path | None]:
    paths = metric_candidates(row)
    if not paths:
        return {}, None

    def score(path: Path) -> tuple[int, float]:
        epoch = checkpoint_epoch(path)
        return (epoch if epoch is not None else -1, path.stat().st_mtime)

    latest = max(paths, key=score)
    try:
        with latest.open() as f:
            return json.load(f), latest
    except Exception as exc:
        print('failed reading metrics:', latest, exc)
        return {}, latest


def downsample_xy(x: np.ndarray, y: np.ndarray, max_points: int = 1200) -> tuple[np.ndarray, np.ndarray]:
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    finite = np.isfinite(x) & np.isfinite(y)
    x = x[finite]
    y = y[finite]
    if len(x) <= max_points:
        return x, y
    idx = np.linspace(0, len(x) - 1, max_points, dtype=int)
    return x[idx], y[idx]


def cycle_average_epoch_loss(
    epoch_loss: np.ndarray,
    steps_per_epoch: int,
    restart_updates: int = 4000,
) -> tuple[np.ndarray, np.ndarray]:
    """Average roughly one LR-restart cycle without changing the update axis."""
    y = np.asarray(epoch_loss, dtype=float)
    if not len(y):
        return np.asarray([], dtype=float), np.asarray([], dtype=float)
    steps_per_epoch = max(1, int(steps_per_epoch))
    window = max(1, min(len(y), int(round(restart_updates / steps_per_epoch))))
    if window == 1:
        x = (np.arange(len(y), dtype=float) + 1.0) * steps_per_epoch
        return downsample_xy(x, y)
    kernel = np.ones(window, dtype=float) / window
    smooth = np.convolve(y, kernel, mode='valid')
    x = (np.arange(len(smooth), dtype=float) + 0.5 * (window + 1)) * steps_per_epoch
    return downsample_xy(x, smooth)


loss_by_run: dict[str, dict[str, Any]] = {}
loss_rows = []
if manifest_df.empty:
    display(Markdown('No manifest available; skipping loss audit.'))
else:
    for _, row in manifest_df.sort_values(['arch', 'dataset_size']).iterrows():
        metrics, metrics_path = read_latest_metrics(row)
        epoch_loss = flatten_numeric(metrics.get('epoch_loss'))
        batch_loss = flatten_numeric(metrics.get('loss', metrics.get('batch_loss')))
        epoch_lr = flatten_numeric(metrics.get('epoch_lr', metrics.get('lr')))
        run_name = str(row.get('run_name'))
        loss_by_run[run_name] = {
            'metrics': metrics,
            'metrics_path': metrics_path,
            'epoch_loss': epoch_loss,
            'batch_loss': batch_loss,
            'epoch_lr': epoch_lr,
        }
        steps_per_epoch = int(row.get('steps_per_epoch', 1) or 1)
        tail_window = max(1, min(len(epoch_loss), max(5, len(epoch_loss) // 20)))
        loss_rows.append({
            'arch': row.get('arch'),
            'arch_label': row.get('arch_label'),
            'run_name': run_name,
            'dataset_tag': row.get('dataset_tag'),
            'dataset_size': int(row.get('dataset_size')) if pd.notna(row.get('dataset_size')) else np.nan,
            'steps_per_epoch': steps_per_epoch,
            'gradient_accumulation_steps': int(row.get('gradient_accumulation_steps', 1) or 1),
            'metrics_path': rel(metrics_path) if metrics_path else None,
            'n_epoch_loss': len(epoch_loss),
            'final_epoch_loss': float(epoch_loss[-1]) if len(epoch_loss) else np.nan,
            'best_epoch_loss': float(np.nanmin(epoch_loss)) if len(epoch_loss) else np.nan,
            'tail_median_epoch_loss': float(np.nanmedian(epoch_loss[-tail_window:])) if len(epoch_loss) else np.nan,
            'epochs_completed': len(epoch_loss),
            'optimizer_updates_recorded': len(epoch_loss) * steps_per_epoch,
            'n_batch_loss': len(batch_loss),
            'final_batch_loss': float(batch_loss[-1]) if len(batch_loss) else np.nan,
        })

    loss_df = pd.DataFrame(loss_rows).sort_values(['arch', 'dataset_size'])

    # Load the independent L16 replacement sweep. A history is used only
    # when its recorded optimizer-update count reaches the requested budget.
    FRESH_LOSS_SWEEP_NAME = 'nf_generalize_fig2_dit_l16_fresh300k_v2'
    fresh_loss_manifest_path = (
        PROJECT_DIR / 'local' / FRESH_LOSS_SWEEP_NAME / 'manifest.json'
    )
    fresh_loss_obj = read_json(fresh_loss_manifest_path)
    if isinstance(fresh_loss_obj, dict):
        fresh_loss_rows = fresh_loss_obj.get('runs', [])
    elif isinstance(fresh_loss_obj, list):
        fresh_loss_rows = fresh_loss_obj
    else:
        fresh_loss_rows = []

    def explicit_update_count(metrics: dict[str, Any]) -> int | None:
        for key in ('optimizer_updates', 'optimizer_step', 'global_step', 'num_updates', 'updates'):
            if key in metrics:
                values = flatten_numeric(metrics.get(key))
                if len(values):
                    return int(round(values[-1]))
        return None

    fresh_loss_by_tag: dict[str, dict[str, Any]] = {}
    fresh_loss_audit_rows: list[dict[str, Any]] = []
    fresh_loss_df = pd.DataFrame(fresh_loss_rows)
    if not fresh_loss_df.empty:
        if 'run_name' not in fresh_loss_df.columns and 'name' in fresh_loss_df.columns:
            fresh_loss_df['run_name'] = fresh_loss_df['name']
        if 'dataset_tag' not in fresh_loss_df.columns:
            fresh_loss_df['dataset_tag'] = fresh_loss_df['run_name'].map(dataset_tag_from_name)
        if 'dataset_size' not in fresh_loss_df.columns:
            fresh_loss_df['dataset_size'] = fresh_loss_df['dataset_tag'].map(dataset_size_from_tag)

        for _, fresh_row in fresh_loss_df.sort_values('dataset_size').iterrows():
            dataset_tag = str(fresh_row.get('dataset_tag', '') or '')
            steps_per_epoch = max(1, int(fresh_row.get('steps_per_epoch', 1) or 1))
            target_updates = int(fresh_row.get('target_total_updates', 300000) or 300000)
            metrics, metrics_path = read_latest_metrics(fresh_row)
            epoch_loss = flatten_numeric(metrics.get('epoch_loss'))
            computed_updates = len(epoch_loss) * steps_per_epoch
            recorded_updates = explicit_update_count(metrics) or computed_updates
            use_fresh_300k = recorded_updates >= 0.98 * target_updates
            if use_fresh_300k:
                fresh_loss_by_tag[dataset_tag] = {
                    'metrics': metrics,
                    'metrics_path': metrics_path,
                    'epoch_loss': epoch_loss,
                    'batch_loss': flatten_numeric(metrics.get('loss', metrics.get('batch_loss'))),
                    'epoch_lr': flatten_numeric(metrics.get('epoch_lr', metrics.get('lr'))),
                    'steps_per_epoch': steps_per_epoch,
                    'optimizer_updates_recorded': recorded_updates,
                    'source_label': 'fresh 300k v2',
                }
            fresh_loss_audit_rows.append({
                'run_name': fresh_row.get('run_name'),
                'dataset_tag': dataset_tag,
                'target_updates': target_updates,
                'optimizer_updates_recorded': recorded_updates,
                'using_fresh_300k_history': use_fresh_300k,
                'metrics_path': rel(metrics_path) if metrics_path else None,
            })

    fresh_loss_audit_df = pd.DataFrame(fresh_loss_audit_rows)
    display(Markdown('### Fresh 300k v2 loss-source audit'))
    if fresh_loss_audit_df.empty:
        display(Markdown(
            f'**Missing fresh loss manifest:** `{rel(fresh_loss_manifest_path)}`. '
            'The plot will retain verified 200k histories and will not claim 300k.'
        ))
    else:
        display(fresh_loss_audit_df)

    loss_plot_summary_df = loss_df.copy()
    loss_plot_summary_df['loss_source'] = '200k original'
    for dataset_tag, fresh_source in fresh_loss_by_tag.items():
        mask = (
            (loss_plot_summary_df['arch'].astype(str) == 'dit_l16')
            & (loss_plot_summary_df['dataset_tag'].astype(str) == dataset_tag)
        )
        epoch_loss = np.asarray(fresh_source['epoch_loss'], dtype=float)
        tail_window = max(1, min(len(epoch_loss), max(5, len(epoch_loss) // 20)))
        loss_plot_summary_df.loc[mask, 'epochs_completed'] = len(epoch_loss)
        loss_plot_summary_df.loc[mask, 'optimizer_updates_recorded'] = fresh_source[
            'optimizer_updates_recorded'
        ]
        loss_plot_summary_df.loc[mask, 'tail_median_epoch_loss'] = float(
            np.nanmedian(epoch_loss[-tail_window:])
        )
        loss_plot_summary_df.loc[mask, 'best_epoch_loss'] = float(np.nanmin(epoch_loss))
        loss_plot_summary_df.loc[mask, 'loss_source'] = 'fresh 300k v2'
    loss_summary_cols = [
        'arch_label', 'dataset_tag', 'dataset_size', 'epochs_completed',
        'optimizer_updates_recorded', 'tail_median_epoch_loss', 'best_epoch_loss',
    ]
    display(loss_df[loss_summary_cols])

    plot_df = manifest_df[manifest_df['arch'].isin(LOSS_ARCHES) & manifest_df['dataset_tag'].isin(LOSS_TAGS)].copy()
    if plot_df.empty:
        plot_df = manifest_df.copy()

    if loss_df['n_epoch_loss'].sum() == 0 and loss_df['n_batch_loss'].sum() == 0:
        print('No training metrics JSON found yet.')
    else:
        selected_tags = [tag for tag in LOSS_TAGS if tag in set(plot_df['dataset_tag'])]
        fig, axes = plt.subplots(1, len(selected_tags), figsize=(18.0, 6.8), sharex=False, sharey=True)
        if len(selected_tags) == 1:
            axes = np.asarray([axes])
        for ax, dataset_tag in zip(axes, selected_tags):
            tag_df = plot_df[plot_df['dataset_tag'] == dataset_tag].copy()
            tag_df['arch_order'] = tag_df['arch'].map({name: i for i, name in enumerate(DIT_ARCH_ORDER)})
            panel_curves = []
            panel_sources = []
            for _, row in tag_df.sort_values('arch_order').iterrows():
                arch = str(row.get('arch'))
                run_name = str(row.get('run_name'))
                loss_source = loss_by_run.get(run_name, {})
                if arch == 'dit_l16' and dataset_tag in fresh_loss_by_tag:
                    loss_source = fresh_loss_by_tag[dataset_tag]
                epoch_loss = np.asarray(loss_source.get('epoch_loss', []), dtype=float)
                if not len(epoch_loss):
                    continue
                steps_per_epoch = max(1, int(
                    loss_source.get('steps_per_epoch', row.get('steps_per_epoch', 1)) or 1
                ))
                x, y = cycle_average_epoch_loss(epoch_loss, steps_per_epoch)
                source_label = str(loss_source.get('source_label', '200k original'))
                panel_sources.append((arch, source_label, len(epoch_loss)))
                panel_curves.append((arch, x / 1000.0, y))
                ax.plot(
                    x / 1000.0, y, lw=2.8,
                    color=DIT_ARCH_COLORS.get(arch, '0.2'),
                    label=arch_label(arch),
                )
            dataset_size = dataset_size_from_tag(dataset_tag)
            dataset_exponent = int(round(np.log2(dataset_size)))
            l16_source = fresh_loss_by_tag.get(dataset_tag)
            if l16_source is not None:
                representative_epochs = len(l16_source['epoch_loss'])
                l16_budget_k = int(round(l16_source['optimizer_updates_recorded'] / 1000.0))
                budget_note = f'L16: {l16_budget_k}k'
            else:
                representative_epochs = next(
                    (epochs for arch, _, epochs in panel_sources if arch == 'dit_l16'), 0
                )
                budget_note = 'L16: verified 200k only'
            ax.set_title(
                f'$N_{{2D}}=2^{{{dataset_exponent}}}$\n{budget_note}; {representative_epochs:,} epochs',
                fontsize=17, pad=12,
            )
            ax.set_xlabel('Optimizer updates (thousands)', fontsize=15)
            ax.set_yscale('log')
            ax.grid(alpha=0.18, which='major')
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            if dataset_tag == 'd2p15' and panel_curves:
                inset = ax.inset_axes([0.46, 0.12, 0.50, 0.38])
                tail_values = []
                for arch, x_plot, y_plot in panel_curves:
                    inset.plot(x_plot, y_plot, lw=2.0, color=DIT_ARCH_COLORS.get(arch, '0.2'))
                    tail = y_plot[x_plot >= 20.0]
                    if len(tail):
                        tail_values.extend(tail.tolist())
                if tail_values:
                    y_min, y_max = float(np.nanmin(tail_values)), float(np.nanmax(tail_values))
                    padding = max(1e-6, 0.10 * (y_max - y_min))
                    inset.set_ylim(y_min - padding, y_max + padding)
                inset.set_xlim(20, max(float(np.nanmax(x)) for _, x, _ in panel_curves))
                inset.set_title('linear-scale zoom', fontsize=11, pad=3)
                inset.set_xlabel('updates (k)', fontsize=9)
                inset.tick_params(labelsize=8)
                inset.grid(alpha=0.16)
        axes[0].set_ylabel('Cycle-averaged training loss', fontsize=16)
        handles = [
            Line2D(
                [0], [0], color=DIT_ARCH_COLORS[a], lw=3.0,
                label=(arch_label(a) + (' (fresh 300k)' if a == 'dit_l16' else ' (200k)')),
            )
            for a in DIT_ARCH_ORDER
        ]
        fig.legend(handles=handles, loc='upper center', bbox_to_anchor=(0.5, 0.925), ncol=3, frameon=False, fontsize=14)
        fig.suptitle('DiT optimization: L8/L12 at 200k, fresh L16 at 300k', fontsize=21, fontweight='semibold', y=0.99)
        fig.text(
            0.5, 0.855,
            f'Fresh L16 histories accepted for {len(fresh_loss_by_tag)}/10 data sizes; x-axes show recorded optimizer updates.',
            ha='center', va='center', fontsize=14, color='0.30',
        )
        fig.subplots_adjust(left=0.07, right=0.99, bottom=0.12, top=0.70, wspace=0.10)
        out = QUICKCHECK_DIR / 'nf_generalize_fig2_dit_training_curves.png'
        QUICKCHECK_DIR.mkdir(parents=True, exist_ok=True)
        fig.savefig(out, bbox_inches='tight')
        plt.show()
        print('wrote', out)

        fig, ax = plt.subplots(figsize=(11.5, 6.4), constrained_layout=True)
        for arch in DIT_ARCH_ORDER:
            sub = loss_plot_summary_df[loss_plot_summary_df['arch'].astype(str) == arch].sort_values('dataset_size')
            if sub.empty:
                continue
            ax.plot(
                np.log2(sub['dataset_size'].astype(float)), sub['tail_median_epoch_loss'],
                marker=DIT_ARCH_MARKERS.get(arch, 'o'), ms=8.5, lw=2.8,
                color=DIT_ARCH_COLORS.get(arch, '0.2'), label=arch_label(arch),
            )
        exponents = sorted(np.log2(loss_df['dataset_size'].dropna().astype(float)).astype(int).unique())
        ax.set_xticks(exponents)
        ax.set_xticklabels([f'$2^{{{p}}}$' for p in exponents])
        ax.set_yscale('log')
        ax.set_xlabel(r'Training set size $N_{2D}$', fontsize=16)
        ax.set_ylabel('Median epoch loss over final 5%', fontsize=16)
        ax.set_title('Tail loss at verified budgets: L8/L12 200k, fresh L16 300k', fontsize=19, pad=14)
        ax.grid(alpha=0.18, which='major')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.legend(frameon=False, ncol=3, fontsize=13)
        summary_out = QUICKCHECK_DIR / 'nf_generalize_fig2_dit_tail_loss_by_data_size.png'
        fig.savefig(summary_out, bbox_inches='tight')
        plt.show()
        print('wrote', summary_out)

        selected_summary = loss_plot_summary_df[
            loss_plot_summary_df['arch'].isin(DIT_ARCH_ORDER)
            & loss_plot_summary_df['dataset_tag'].isin(selected_tags)
        ][['arch_label', 'dataset_tag', 'dataset_size', 'epochs_completed',
           'optimizer_updates_recorded', 'tail_median_epoch_loss', 'loss_source']]
        display(selected_summary.sort_values(['dataset_size', 'arch_label']))
        display(Markdown(
            '**Interpretation:** each L16 panel uses the independent fresh 300k history only after '
            'the audit verifies its recorded update count. L8 and L12 remain at 200k. '
            'The inset is a linear-scale zoom of the rightmost panel after 20k updates. '
            'A low loss for a small dataset can reflect repeated exposure and memorization. '
            'The learning-rate traces are omitted because these runs share the same 4,000-update '
            'cosine-restart schedule; overlaying nine copies obscures the loss comparison.'
        ))

### Reading the optimization plots

- Each panel fixes the training-set size and compares model depth. The horizontal axis counts optimizer updates, not epochs.
- The vertical axis is the denoising objective on a logarithmic scale. The smoothed curves remove the 4,000-update restart oscillation so the long-run trend is visible.
- DiT-L16 can reach a low training loss at small and intermediate $N_{2D}$. That confirms optimization is occurring, but it does **not** establish good samples: the same checkpoints can still fail $P(k)$.
- The $2^{15}$ inset uses a linear vertical scale because the three curves are nearly coincident on the main logarithmic panel.

## Load Generated and Real Reference Slices

Each reference is loaded from the **training subset configured for that exact DiT run**, not from all available CAMELS maps. The config's source paths, `n_samples`, seed, and `zthin` define the subset. The notebook verifies that its exact slice count equals the manifest's $N_{2D}$.

Normalization uses the complete configured subset. For memory safety, general plotting may then take an even cap controlled by `DIT_MAX_REAL_REFERENCE_SLICES` (default 2,048; 0 means all). The nearest-training audit below refuses to run unless the complete configured subset is loaded.


In [ ]:
def npz_array_key(path: Path) -> str:
    with np.load(path) as data:
        return 'samples' if 'samples' in data.files else data.files[0]


def load_npz_array(path: Path) -> np.ndarray:
    with np.load(path) as data:
        key = 'samples' if 'samples' in data.files else data.files[0]
        arr = np.asarray(data[key], dtype=np.float32)
    if SIMDIFF_EVAL_AVAILABLE:
        return as_nchw(arr)
    if arr.ndim == 3:
        return arr[:, None, :, :]
    if arr.ndim == 4 and arr.shape[1] in (1, 3):
        return arr
    if arr.ndim == 4 and arr.shape[-1] in (1, 3):
        return np.moveaxis(arr, -1, 1)
    raise ValueError(f'Could not interpret generated array shape {arr.shape}')


def evenly_limit(arr: np.ndarray, limit: int | None) -> np.ndarray:
    arr = np.asarray(arr)
    if limit is None or len(arr) <= int(limit):
        return arr.copy()
    idx = np.linspace(0, len(arr) - 1, int(limit), dtype=int)
    return arr[idx].copy()


def config_path_for(row: pd.Series) -> Path:
    raw = str(row.get('config', '') or row.get('config_path', '') or '')
    if raw:
        path = Path(raw)
        return path if path.is_absolute() else PROJECT_DIR / path
    return PROJECT_DIR / 'local' / SWEEP_NAME / 'configs' / f"{row['run_name']}.yaml"


def real_reference_cache_key(config_path: Path) -> str:
    config = yaml.safe_load(config_path.read_text()) or {}
    return json.dumps(config.get('data', config), sort_keys=True, default=str)


loaded: dict[str, dict[str, Any]] = {}
load_rows = []
real_reference_cache: dict[str, np.ndarray] = {}
real_reference_kind = (
    'model training subset; normalized from complete configured training set; '
    'evenly capped only after normalization'
)
if manifest_df.empty:
    display(Markdown('No manifest available; skipping generated/real loading.'))
elif not SIMDIFF_EVAL_AVAILABLE:
    display(Markdown(f'`simdiff_eval` unavailable, so real-reference diagnostics are skipped: `{SIMDIFF_EVAL_ERROR}`'))
else:
    for _, row in manifest_df.sort_values(['arch', 'dataset_size']).iterrows():
        sample_path = Path(row['sample_path_resolved'])
        if not sample_path.exists():
            continue
        cfg_path = config_path_for(row)
        try:
            reference_info = configured_training_reference_info(cfg_path)
            configured_slices = int(reference_info['configured_slices'])
            expected_slices = int(row.get('dataset_size'))
            reference_matches_manifest = configured_slices == expected_slices
            if not reference_matches_manifest:
                raise RuntimeError(
                    f"REAL REFERENCE MISMATCH for {row.get('run_name')}: config selects "
                    f"{configured_slices} slices but manifest says {expected_slices}"
                )
            generated = evenly_limit(load_npz_array(sample_path), MAX_GENERATED)
            cache_key = real_reference_cache_key(cfg_path)
            if cache_key not in real_reference_cache:
                real_reference_cache[cache_key] = load_real_reference_from_config(
                    cfg_path,
                    max_slices=MAX_REAL_REFERENCE_SLICES,
                )
            real = real_reference_cache[cache_key]
            run_name = str(row.get('run_name'))
            loaded[run_name] = {
                'spec': row,
                'real': real,
                'generated': generated,
                'sample_path': sample_path,
                'config_path': cfg_path,
                'reference_info': reference_info,
                'real_reference_kind': real_reference_kind,
            }
            load_rows.append({
                'arch': row.get('arch'),
                'arch_label': row.get('arch_label'),
                'run_name': run_name,
                'dataset_tag': row.get('dataset_tag'),
                'dataset_size': int(row.get('dataset_size')),
                'n_real_exact_model_subset': configured_slices,
                'n_real_sources': int(reference_info['n_sources']),
                'n_real_raw_simulations': int(reference_info['configured_raw_samples']),
                'real_zthin': int(reference_info['zthin']),
                'n_real_used_for_plot': len(real),
                'reference_complete_for_plot': len(real) == configured_slices,
                'reference_matches_manifest': reference_matches_manifest,
                'reference_selection': reference_info['selection'],
                'real_reference_kind': real_reference_kind,
                'n_generated_loaded': len(generated),
                'sample_path': rel(sample_path),
                'config_path': rel(cfg_path),
            })
        except Exception as exc:
            load_rows.append({
                'arch': row.get('arch'),
                'arch_label': row.get('arch_label'),
                'run_name': row.get('run_name'),
                'dataset_tag': row.get('dataset_tag'),
                'dataset_size': row.get('dataset_size'),
                'n_real_configured': row.get('dataset_size'),
                'n_real_loaded': 0,
                'real_reference_kind': real_reference_kind,
                'n_generated_loaded': 0,
                'error': repr(exc),
                'sample_path': rel(sample_path),
                'config_path': rel(cfg_path),
            })

loaded_df = pd.DataFrame(load_rows).sort_values(['arch', 'dataset_size']) if load_rows else pd.DataFrame()
display(loaded_df)
print('loaded DiT sample rows:', len(loaded))

## Generated Image Grids Across Data Size

A quick visual check across the DiT data-size sweep. These are not nearest-neighbor diagnostics; they just show what the generated fields look like as `N_2D` changes.


In [ ]:
def choose_bundles(tags: list[str], arch: str | None = None) -> list[dict[str, Any]]:
    """Return exact requested bundles in tag order, without silent fallback."""
    requested = list(dict.fromkeys(str(tag) for tag in tags))
    by_tag = {
        str(bundle['spec'].get('dataset_tag')): bundle
        for bundle in loaded.values()
        if arch is None or str(bundle['spec'].get('arch')) == arch
    }
    missing = [tag for tag in requested if tag not in by_tag]
    if missing:
        display(Markdown(
            f"`{arch_label(arch) if arch else 'all architectures'}` requested tags are missing: "
            + ', '.join(f'`{tag}`' for tag in missing)
        ))
    return [by_tag[tag] for tag in requested if tag in by_tag]


def plot_dit_image_grid(
    sample_index: int = 0,
    tags: list[str] = ALL_DATA_TAGS,
    arch: str = IMAGE_ARCH,
    block_name: str = 'full_sweep',
) -> Path | None:
    if not loaded:
        display(Markdown('No loaded DiT samples available for image grid.'))
        return None
    bundles = choose_bundles(tags, arch=arch)
    if not bundles:
        display(Markdown(f'No exact bundles available for `{arch_label(arch)}` image grid.'))
        return None

    values = []
    for bundle in bundles:
        gen_idx = min(sample_index, len(bundle['generated']) - 1)
        values.extend([
            bundle['generated'][gen_idx, 0].ravel(),
            bundle['real'][0, 0].ravel(),
        ])
    flat = np.concatenate(values)
    vmin, vmax = np.nanquantile(flat, [0.005, 0.995])

    columns = 5
    blocks = int(np.ceil(len(bundles) / columns))
    fig, axes = plt.subplots(
        2 * blocks,
        columns,
        figsize=(3.05 * columns, 5.9 * blocks),
        squeeze=False,
        constrained_layout=True,
    )
    for index, bundle in enumerate(bundles):
        block, col = divmod(index, columns)
        generated_axis = axes[2 * block, col]
        reference_axis = axes[2 * block + 1, col]
        spec = bundle['spec']
        gen_idx = min(sample_index, len(bundle['generated']) - 1)
        generated_axis.imshow(
            bundle['generated'][gen_idx, 0], cmap='viridis', vmin=vmin, vmax=vmax
        )
        reference_axis.imshow(
            bundle['real'][0, 0], cmap='viridis', vmin=vmin, vmax=vmax
        )
        generated_axis.set_title(dataset_size_label(int(spec['dataset_size'])), pad=8)
        for axis in (generated_axis, reference_axis):
            axis.set_xticks([])
            axis.set_yticks([])
    for index in range(len(bundles), blocks * columns):
        block, col = divmod(index, columns)
        axes[2 * block, col].set_visible(False)
        axes[2 * block + 1, col].set_visible(False)
    for block in range(blocks):
        axes[2 * block, 0].set_ylabel('generated', fontsize=15, fontweight='bold')
        axes[2 * block + 1, 0].set_ylabel(
            'training subset', fontsize=15, fontweight='bold'
        )
    fig.suptitle(
        f'{arch_label(arch)} generated maps: {block_name.replace("_", " ")}',
        fontsize=21,
        fontweight='semibold',
    )
    QUICKCHECK_DIR.mkdir(parents=True, exist_ok=True)
    out = QUICKCHECK_DIR / (
        f'nf_generalize_fig2_{arch}_{block_name}_generated_image_grid.png'
    )
    fig.savefig(out, bbox_inches='tight', dpi=300)
    plt.show()
    print('wrote', out)
    return out


image_grid_paths = {}
for image_arch in DIT_ARCH_ORDER:
    image_grid_paths[image_arch] = plot_dit_image_grid(
        sample_index=int(os.environ.get('DIT_IMAGE_SAMPLE_INDEX', '0')),
        tags=ALL_DATA_TAGS,
        arch=image_arch,
        block_name='full_sweep',
    )
image_grid_path = image_grid_paths

### Reading the generated-map grid

This fixed-200k diagnostic includes every training-set size from $2^6$ through $2^{15}$. Read the first pair of rows as $2^6$ through $2^{10}$ and the second pair as $2^{11}$ through $2^{15}$. Within each pair, the upper row is generated and the lower row is one real reference displayed with the same color limits.

This is a qualitative failure check only. A plausible-looking field can still copy a training map, and a visibly different field can be out of distribution. The nearest-training, Fréchet, PDF, and $P(k)$ sections make those distinctions quantitative.

## Generated Samples Versus Nearest Training Slices

This is the requested visual copy check for the DiT runs. For each selected generated map, the notebook searches the **complete configured training subset for that exact model** in pixel mean-squared error, then shows the generated map, its closest training slice, and their absolute difference.

The summary covers DiT-L8, DiT-L12, and DiT-L16 at $N_{2D}=2^6,\ldots,2^{10}$. A second panel shows four DiT-L16 generated maps at $N_{2D}=2^8$ so the visibly noisy sample can be checked as a possible fluke. This pixel-space plot is a visual audit; PCA and SSCD remain the primary full-sample novelty metrics.


In [ ]:
DIT_NN_ARCHES = parse_csv_env(
    'DIT_NN_ARCHES', 'dit_l8,dit_base,dit_l16'
)
DIT_NN_TAGS = parse_csv_env('DIT_NN_TAGS', 'd2p06,d2p07,d2p08,d2p09,d2p10')
DIT_NN_MAX_GENERATED = int(os.environ.get('DIT_NN_MAX_GENERATED', '4'))
DIT_NN_SUMMARY_GENERATED_INDEX = int(os.environ.get('DIT_NN_SUMMARY_GENERATED_INDEX', '0'))
DIT_NN_AUDIT_ARCH = os.environ.get('DIT_NN_AUDIT_ARCH', 'dit_l16')
DIT_NN_AUDIT_TAG = os.environ.get('DIT_NN_AUDIT_TAG', 'd2p08')


def require_complete_training_reference(bundle: dict[str, Any]) -> int:
    configured_slices = int(bundle['reference_info']['configured_slices'])
    loaded_slices = len(bundle['real'])
    if loaded_slices != configured_slices:
        raise RuntimeError(
            'nearest-training search requires the complete configured subset: '
            f"run={bundle['spec']['run_name']} loaded={loaded_slices} "
            f'configured={configured_slices}. Increase DIT_MAX_REAL_REFERENCE_SLICES '
            'or restrict DIT_NN_TAGS to sizes at or below the cap.'
        )
    return configured_slices


def build_dit_nearest_training_audit(
    arches: list[str] = DIT_NN_ARCHES,
    tags: list[str] = DIT_NN_TAGS,
) -> tuple[pd.DataFrame, dict[str, dict[str, Any]]]:
    records = []
    by_run = {}
    for arch in arches:
        for bundle in choose_bundles(tags, arch=arch):
            row = bundle['spec']
            configured_slices = require_complete_training_reference(bundle)
            matches = nearest_training_matches(
                bundle['generated'],
                bundle['real'],
                max_generated=DIT_NN_MAX_GENERATED,
                max_training=None,
                training_chunk=256,
            )
            by_run[str(row['run_name'])] = {'bundle': bundle, 'matches': matches}
            for i, generated_index in enumerate(matches['generated_index']):
                records.append({
                    'arch': str(row['arch']),
                    'run_name': str(row['run_name']),
                    'dataset_tag': str(row['dataset_tag']),
                    'dataset_size': int(row['dataset_size']),
                    'generated_index': int(generated_index),
                    'nearest_training_index': int(matches['nearest_training_index'][i]),
                    'nearest_mse': float(matches['nearest_mse'][i]),
                    'nearest_rmse': float(matches['nearest_rmse'][i]),
                    'nearest_cosine': float(matches['nearest_cosine'][i]),
                    'n_training_searched': configured_slices,
                    'reference_is_complete': True,
                    'sample_path': rel(bundle['sample_path']),
                    'config_path': rel(bundle['config_path']),
                })
    audit = pd.DataFrame(records)
    if not audit.empty:
        audit = audit.sort_values(['arch', 'dataset_size', 'generated_index'])
        TABLE_DIR.mkdir(parents=True, exist_ok=True)
        out = TABLE_DIR / 'nf_generalize_fig2_dit_pixel_nearest_training.csv'
        audit.to_csv(out, index=False)
        print('wrote', out)
        display(audit)
    return audit, by_run


def nearest_plot_limits(items: list[tuple[np.ndarray, np.ndarray]]) -> tuple[float, float, float]:
    fields = np.concatenate([image.ravel() for pair in items for image in pair])
    differences = np.concatenate([np.abs(first - second).ravel() for first, second in items])
    return (
        float(np.nanquantile(fields, 0.005)),
        float(np.nanquantile(fields, 0.995)),
        float(np.nanquantile(differences, 0.995)),
    )


def plot_dit_nearest_training_summary(
    audit: pd.DataFrame,
    by_run: dict[str, dict[str, Any]],
    arch: str,
) -> Path | None:
    selected_arch = audit[audit['arch'] == arch]
    if selected_arch.empty:
        display(Markdown(f'No `{arch}` samples are available for the nearest-training summary.'))
        return None
    selected = selected_arch[
        selected_arch['generated_index'] == DIT_NN_SUMMARY_GENERATED_INDEX
    ].copy()
    if selected.empty:
        selected = selected_arch.groupby('run_name', as_index=False).head(1)
    selected = selected.sort_values('dataset_size')
    items = []
    for _, record in selected.iterrows():
        bundle = by_run[str(record['run_name'])]['bundle']
        generated = bundle['generated'][int(record['generated_index']), 0]
        training = bundle['real'][int(record['nearest_training_index']), 0]
        items.append((record, generated, training))
    vmin, vmax, diff_vmax = nearest_plot_limits([(gen, train) for _, gen, train in items])

    fig, axes = plt.subplots(
        3, len(items), figsize=(3.05 * len(items), 8.9),
        squeeze=False, constrained_layout=True,
    )
    for col, (record, generated, training) in enumerate(items):
        axes[0, col].imshow(generated, cmap='viridis', vmin=vmin, vmax=vmax)
        axes[1, col].imshow(training, cmap='viridis', vmin=vmin, vmax=vmax)
        axes[2, col].imshow(np.abs(generated - training), cmap='magma', vmin=0, vmax=diff_vmax)
        axes[0, col].set_title(dataset_size_label(int(record['dataset_size'])), fontweight='bold')
        axes[2, col].text(
            0.5, 0.02,
            f"MSE={record['nearest_mse']:.3g}; cos={record['nearest_cosine']:.3f}",
            transform=axes[2, col].transAxes, ha='center', va='bottom', fontsize=9,
            bbox=dict(facecolor='white', edgecolor='none', alpha=0.82, pad=2),
        )
        for ax in axes[:, col]:
            ax.set_xticks([])
            ax.set_yticks([])
    for row_index, label in enumerate(('generated', 'nearest training', 'absolute difference')):
        axes[row_index, 0].set_ylabel(label, fontsize=14, fontweight='bold')
    fig.suptitle(
        f'{arch_label(arch)} generated samples versus nearest training slices',
        fontsize=21, fontweight='bold',
    )
    out = QUICKCHECK_DIR / f'nf_generalize_fig2_{arch}_generated_vs_nearest_training.png'
    QUICKCHECK_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(out, bbox_inches='tight', dpi=300)
    plt.show()
    print('wrote', out)
    return out


def plot_dit_nearest_training_fluke_audit(
    audit: pd.DataFrame,
    by_run: dict[str, dict[str, Any]],
    arch: str = DIT_NN_AUDIT_ARCH,
    audit_tag: str = DIT_NN_AUDIT_TAG,
) -> Path | None:
    selected = audit[
        (audit['arch'] == arch) & (audit['dataset_tag'] == audit_tag)
    ].sort_values('generated_index')
    if selected.empty:
        display(Markdown(f'No DiT nearest-training records are available for `{audit_tag}`.'))
        return None
    run_name = str(selected.iloc[0]['run_name'])
    bundle = by_run[run_name]['bundle']
    items = []
    for _, record in selected.iterrows():
        generated = bundle['generated'][int(record['generated_index']), 0]
        training = bundle['real'][int(record['nearest_training_index']), 0]
        items.append((record, generated, training))
    vmin, vmax, diff_vmax = nearest_plot_limits([(gen, train) for _, gen, train in items])

    fig, axes = plt.subplots(
        len(items), 3, figsize=(9.6, 3.0 * len(items)),
        squeeze=False, constrained_layout=True,
    )
    for row_index, (record, generated, training) in enumerate(items):
        axes[row_index, 0].imshow(generated, cmap='viridis', vmin=vmin, vmax=vmax)
        axes[row_index, 1].imshow(training, cmap='viridis', vmin=vmin, vmax=vmax)
        axes[row_index, 2].imshow(np.abs(generated - training), cmap='magma', vmin=0, vmax=diff_vmax)
        axes[row_index, 0].set_ylabel(
            f"generated {int(record['generated_index'])}", fontsize=12, fontweight='bold'
        )
        axes[row_index, 2].text(
            0.5, 0.02,
            f"MSE={record['nearest_mse']:.3g}; cos={record['nearest_cosine']:.3f}",
            transform=axes[row_index, 2].transAxes, ha='center', va='bottom', fontsize=9,
            bbox=dict(facecolor='white', edgecolor='none', alpha=0.82, pad=2),
        )
        for ax in axes[row_index]:
            ax.set_xticks([])
            ax.set_yticks([])
    for col, title in enumerate(('generated', 'nearest training', 'absolute difference')):
        axes[0, col].set_title(title, fontsize=14, fontweight='bold')
    n_value = int(selected.iloc[0]['dataset_size'])
    fig.suptitle(
        f'{arch_label(arch)} {dataset_size_label(n_value)}: '
        f'{len(items)} generated-sample nearest-training audit',
        fontsize=19, fontweight='bold',
    )
    out = QUICKCHECK_DIR / (
        f'nf_generalize_fig2_{arch}_{audit_tag}_nearest_training_audit.png'
    )
    fig.savefig(out, bbox_inches='tight', dpi=300)
    plt.show()
    print('wrote', out)
    return out


dit_pixel_nn_df, dit_pixel_nn_by_run = build_dit_nearest_training_audit()
dit_pixel_nn_summary_paths = {
    arch: plot_dit_nearest_training_summary(dit_pixel_nn_df, dit_pixel_nn_by_run, arch)
    for arch in DIT_NN_ARCHES
}
dit_pixel_nn_fluke_path = plot_dit_nearest_training_fluke_audit(dit_pixel_nn_df, dit_pixel_nn_by_run)


### Reading the nearest-training panels

- **Generated:** the model output.
- **Nearest training:** the closest slice from that model's complete configured training subset.
- **Absolute difference:** what cannot be explained by copying that nearest slice.

A nearly blank difference panel, very small MSE, and cosine similarity near one are evidence of copying. A large difference only establishes novelty; it must still be paired with the in-distribution and physical-statistics checks.

## SSCD Distribution Distance: Are Novel Samples In Distribution?

Nearest-neighbor novelty can assign a high score to a visibly bad sample simply because it is far from every training image. This section adds the requested distribution-level check.

For each run, it loads the SSCD embeddings already cached by the full nearest-neighbor analysis, compares generated embeddings with **heldout real** embeddings, and reports a Fréchet feature distance. Because finite samples have a nonzero Fréchet distance even when both sets are real, the generated-to-heldout value is normalized by a same-size **real-vs-real split baseline**.

Read the ratio as follows:

- near 1: generated-to-real separation is comparable to finite real-split variability;
- substantially above 1: generated samples are farther from the real distribution, so a high novelty score may be invalid;
- below 1: not automatically better, because copying or low diversity can also reduce a distribution distance.

This is an in-distribution diagnostic, not a replacement for nearest-training plots, the one-point PDF, or $P(k)$.


In [ ]:
SSCD_CACHE_DIR = RESULTS_DIR / 'cache' / 'sscd_full_nn'
DIT_DISTRIBUTION_ARCHES = parse_csv_env(
    'DIT_DISTRIBUTION_ARCHES', 'dit_l8,dit_base,dit_l16'
)
DIT_DISTRIBUTION_TAGS = parse_csv_env(
    'DIT_DISTRIBUTION_TAGS',
    'd2p06,d2p07,d2p08,d2p09,d2p10,d2p11,d2p12,d2p13,d2p14,d2p15',
)
DIT_DISTRIBUTION_COMPONENTS = int(os.environ.get('DIT_DISTRIBUTION_COMPONENTS', '64'))
DIT_DISTRIBUTION_SEED = int(os.environ.get('DIT_DISTRIBUTION_SEED', str(SEED)))


def find_sscd_embedding_cache(run_name: str, kind: str) -> Path | None:
    pattern = f'{run_name}_{kind}_{SAMPLE_LABEL}_seed{SEED}_*.pt'
    matches = sorted(
        SSCD_CACHE_DIR.glob(pattern),
        key=lambda path: (path.stat().st_mtime, path.name),
    )
    return matches[-1] if matches else None


def load_sscd_embedding_cache(path: Path) -> np.ndarray:
    import torch

    try:
        payload = torch.load(path, map_location='cpu', weights_only=True)
    except TypeError:
        payload = torch.load(path, map_location='cpu')
    tensor = payload['embeddings'] if isinstance(payload, dict) else payload
    features = np.asarray(tensor.detach().cpu(), dtype=np.float64)
    if features.ndim != 2 or len(features) < 4:
        raise ValueError(f'invalid SSCD embedding cache {path}: shape={features.shape}')
    return features


def project_to_real_pca(
    real_features: np.ndarray,
    generated_features: np.ndarray,
    max_components: int,
) -> tuple[np.ndarray, np.ndarray, int]:
    center = real_features.mean(axis=0, keepdims=True)
    centered_real = real_features - center
    centered_generated = generated_features - center
    max_rank = min(
        int(max_components),
        centered_real.shape[0] - 2,
        centered_real.shape[1],
    )
    if max_rank < 1:
        raise ValueError('not enough heldout real embeddings for PCA projection')
    _, _, right_vectors = np.linalg.svd(centered_real, full_matrices=False)
    basis = right_vectors[:max_rank].T
    return centered_real @ basis, centered_generated @ basis, max_rank


def evaluate_sscd_distribution_distance() -> pd.DataFrame:
    rows = []
    if manifest_df.empty:
        display(Markdown('No manifest is available for the SSCD distribution-distance audit.'))
        return pd.DataFrame()

    selected = manifest_df[
        manifest_df['arch'].isin(DIT_DISTRIBUTION_ARCHES)
        & manifest_df['dataset_tag'].isin(DIT_DISTRIBUTION_TAGS)
    ].sort_values(['arch', 'dataset_size'])
    for _, run in selected.iterrows():
        run_name = str(run['run_name'])
        heldout_path = find_sscd_embedding_cache(run_name, 'heldout')
        generated_path = find_sscd_embedding_cache(run_name, 'generated')
        if heldout_path is None or generated_path is None:
            rows.append({
                'arch': str(run['arch']),
                'arch_label': arch_label(run['arch']),
                'run_name': run_name,
                'dataset_tag': str(run['dataset_tag']),
                'dataset_size': int(run['dataset_size']),
                'status': 'missing SSCD cache',
                'heldout_cache': rel(heldout_path),
                'generated_cache': rel(generated_path),
            })
            continue

        heldout = load_sscd_embedding_cache(heldout_path)
        generated = load_sscd_embedding_cache(generated_path)
        heldout_projected, generated_projected, rank = project_to_real_pca(
            heldout,
            generated,
            DIT_DISTRIBUTION_COMPONENTS,
        )
        n_eval = min(len(generated_projected), len(heldout_projected) // 2)
        if n_eval < 2:
            raise ValueError(f'not enough equal-size samples for {run_name}')

        rng = np.random.default_rng(DIT_DISTRIBUTION_SEED)
        real_indices = rng.permutation(len(heldout_projected))[: 2 * n_eval]
        generated_indices = rng.permutation(len(generated_projected))[:n_eval]
        real_first = heldout_projected[real_indices[:n_eval]]
        real_second = heldout_projected[real_indices[n_eval:]]
        generated_eval = generated_projected[generated_indices]

        real_baseline = real_split_frechet_baseline(
            np.concatenate([real_first, real_second], axis=0),
            seed=DIT_DISTRIBUTION_SEED,
        )
        generated_distance = frechet_feature_distance(generated_eval, real_first)
        baseline_distance = float(real_baseline['distance'])
        ratio = generated_distance / max(baseline_distance, 1e-12)
        rows.append({
            'arch': str(run['arch']),
            'arch_label': arch_label(run['arch']),
            'run_name': run_name,
            'dataset_tag': str(run['dataset_tag']),
            'dataset_size': int(run['dataset_size']),
            'status': 'ok',
            'feature_space': 'SSCD projected on heldout-real PCA',
            'pca_rank': int(rank),
            'n_generated_eval': int(n_eval),
            'n_heldout_eval': int(n_eval),
            'generated_to_heldout_frechet': float(generated_distance),
            'real_split_frechet': baseline_distance,
            'generated_to_heldout_over_real_split': float(ratio),
            'heldout_cache': rel(heldout_path),
            'generated_cache': rel(generated_path),
        })

    frame = pd.DataFrame(rows)
    TABLE_DIR.mkdir(parents=True, exist_ok=True)
    out = TABLE_DIR / 'nf_generalize_fig2_dit_sscd_frechet_distribution_distance.csv'
    frame.to_csv(out, index=False)
    print('wrote', out)
    display(frame)
    return frame


def plot_sscd_distribution_distance(frame: pd.DataFrame) -> Path | None:
    required_columns = {
        'status',
        'generated_to_heldout_over_real_split',
        'dataset_size',
        'arch',
    }
    if frame.empty or not required_columns.issubset(frame.columns):
        display(Markdown(
            '**SSCD distribution-distance plot skipped:** cached heldout and generated '
            'SSCD embeddings were not found. Run the SSCD full-nearest-neighbor analysis first.'
        ))
        return None
    valid = frame[
        (frame['status'] == 'ok')
        & pd.to_numeric(
            frame['generated_to_heldout_over_real_split'], errors='coerce'
        ).notna()
    ].copy()
    if valid.empty:
        display(Markdown(
            '**SSCD distribution-distance plot skipped:** cached heldout and generated '
            'SSCD embeddings were not found. Run the SSCD full-nearest-neighbor analysis first.'
        ))
        return None

    fig, ax = plt.subplots(figsize=(11.8, 6.8), constrained_layout=True)
    for arch in DIT_ARCH_ORDER:
        sub = valid[valid['arch'] == arch].sort_values('dataset_size')
        if sub.empty:
            continue
        ax.plot(
            np.log2(sub['dataset_size'].astype(float)),
            sub['generated_to_heldout_over_real_split'].astype(float),
            color=DIT_ARCH_COLORS[arch],
            marker=DIT_ARCH_MARKERS[arch],
            lw=2.8,
            ms=8,
            label=arch_label(arch),
        )
    exponents = sorted(np.log2(valid['dataset_size'].astype(float)).astype(int).unique())
    ax.set_xticks(exponents)
    ax.set_xticklabels([f'$2^{{{exponent}}}$' for exponent in exponents])
    ax.axhline(1.0, color='0.25', lw=1.8, ls='--', label='real-vs-real baseline')
    ax.set_yscale('log')
    ax.set_xlabel(r'Training images $N_{2D}$')
    ax.set_ylabel('Generated-to-heldout / real-split Fréchet distance')
    ax.set_title('SSCD distribution distance: novelty does not imply validity', pad=14)
    ax.grid(alpha=0.18, which='both')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.legend(frameon=False, ncol=2)
    out = QUICKCHECK_DIR / 'nf_generalize_fig2_dit_sscd_frechet_distribution_distance.png'
    QUICKCHECK_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(out, bbox_inches='tight', dpi=300)
    plt.show()
    print('wrote', out)
    return out


dit_sscd_distribution_df = evaluate_sscd_distribution_distance()
dit_sscd_distribution_plot = plot_sscd_distribution_distance(dit_sscd_distribution_df)


### Reading the SSCD Fréchet-distance plot

The plotted ratio compares generated-to-heldout distance with a real-vs-real finite-sample baseline. A ratio near one means the generated distribution is about as close to heldout data as two real subsets are to each other. A much larger ratio flags outputs that may be novel simply because they are outside the real distribution.

This is the FID-style check requested in the project discussion, using SSCD features rather than ImageNet Inception features because the data are single-channel HI maps.

## One-point Distribution and Power-Spectrum Agreement

These figures test physical agreement, not memorization.

- The **black curve is computed from every slice in the exact training subset used by that model**. It is not built from the complete CAMELS dataset and is not a capped plotting sample. The calculation streams batches from disk so the $2^{15}$ reference remains exact without exhausting notebook memory.
- The one-point distribution compares pixel values but ignores where those values occur.
- The power-spectrum ratio tests spatial structure. A ratio of one is exact agreement; values above or below one mean too much or too little power at that scale.

The PDF and $P(k)$ panels are separated into larger figures so their labels and scale-dependent failures remain readable.


In [ ]:
PHYSICAL_HIST_EDGES = np.linspace(-1.0, 1.0, 141, dtype=np.float64)
REAL_REFERENCE_RAW_BATCH_SIZE = int(os.environ.get('DIT_REAL_REFERENCE_RAW_BATCH_SIZE', '4'))
physical_curve_cache: dict[tuple[str, str], dict[str, Any]] = {}
real_physical_cache: dict[str, dict[str, Any]] = {}


def _radial_power_geometry(shape: tuple[int, int], nbins: int) -> tuple[np.ndarray, list[np.ndarray]]:
    height, width = shape
    ky = np.fft.fftfreq(height) * height
    kx = np.fft.fftfreq(width) * width
    kkx, kky = np.meshgrid(kx, ky)
    kvals = np.sqrt(kkx**2 + kky**2)
    valid = kvals > 0
    edges = np.linspace(kvals[valid].min(), kvals[valid].max(), nbins + 1)
    centers = 0.5 * (edges[:-1] + edges[1:])
    masks = [
        (kvals >= edges[index]) & (kvals < edges[index + 1])
        for index in range(nbins)
    ]
    return centers, masks


def _aggregate_physical_batches(batches, *, nbins: int) -> dict[str, Any]:
    """Aggregate a PDF and mean P(k) without materializing every image."""
    histogram_counts = np.zeros(len(PHYSICAL_HIST_EDGES) - 1, dtype=np.int64)
    power_sum = np.zeros(nbins, dtype=np.float64)
    image_count = 0
    pixel_count = 0
    kbins = None
    masks = None
    for batch in batches:
        batch = as_nchw(np.asarray(batch, dtype=np.float32))
        if not len(batch):
            continue
        fields = np.asarray(batch[:, 0], dtype=np.float64)
        histogram_counts += np.histogram(fields.ravel(), bins=PHYSICAL_HIST_EDGES)[0]
        pixel_count += int(fields.size)
        if masks is None:
            kbins, masks = _radial_power_geometry(tuple(fields.shape[-2:]), nbins)
        centered = fields - fields.mean(axis=(-2, -1), keepdims=True)
        fft = np.fft.fftn(centered, axes=(-2, -1))
        power = (fft * fft.conj()).real / (fields.shape[-2] * fields.shape[-1])
        for index, mask in enumerate(masks):
            if mask.any():
                power_sum[index] += float(np.sum(np.mean(power[:, mask], axis=1)))
        image_count += len(fields)
    if image_count == 0 or kbins is None:
        raise RuntimeError('No images were available for physical-statistics aggregation.')
    widths = np.diff(PHYSICAL_HIST_EDGES)
    in_range = int(histogram_counts.sum())
    density = histogram_counts / np.clip(in_range * widths, 1, None)
    return {
        'hist': density,
        'hist_counts': histogram_counts,
        'hist_edges': PHYSICAL_HIST_EDGES.copy(),
        'kbins': kbins,
        'mean_pk': power_sum / image_count,
        'n_images': int(image_count),
        'pixel_coverage': float(in_range / pixel_count),
    }


def _band_log_error(ratio: np.ndarray, start: float, stop: float) -> float:
    ratio = np.asarray(ratio, dtype=np.float64)
    finite_indices = np.flatnonzero(np.isfinite(ratio) & (ratio > 0))
    if not len(finite_indices):
        return np.nan
    lo = int(np.floor(start * len(finite_indices)))
    hi = int(np.floor(stop * len(finite_indices)))
    if stop >= 1.0:
        hi = len(finite_indices)
    selected = finite_indices[lo:max(lo + 1, hi)]
    return float(np.mean(np.abs(np.log10(ratio[selected]))))


def _physical_curve_for_bundle(bundle: dict[str, Any]) -> dict[str, Any]:
    spec = bundle['spec']
    arch = str(spec['arch'])
    tag = str(spec['dataset_tag'])
    key = (arch, tag)
    if key in physical_curve_cache:
        return physical_curve_cache[key]

    reference_key = real_reference_cache_key(bundle['config_path'])
    if reference_key not in real_physical_cache:
        real_physical_cache[reference_key] = _aggregate_physical_batches(
            iter_real_reference_batches_from_config(
                bundle['config_path'], raw_batch_size=REAL_REFERENCE_RAW_BATCH_SIZE
            ),
            nbins=PK_NBINS,
        )
    real_stats = real_physical_cache[reference_key]
    expected_real = int(bundle['reference_info']['configured_slices'])
    if int(real_stats['n_images']) != expected_real:
        raise RuntimeError(
            f"Exact real-reference count mismatch for {spec['run_name']}: "
            f"aggregated {real_stats['n_images']} but config selects {expected_real}."
        )
    generated_stats = _aggregate_physical_batches(
        [bundle['generated']], nbins=PK_NBINS
    )
    ratio = generated_stats['mean_pk'] / np.clip(real_stats['mean_pk'], 1e-30, None)
    widths = np.diff(real_stats['hist_edges'])
    curve = {
        'arch': arch,
        'arch_label': arch_label(arch),
        'dataset_tag': tag,
        'dataset_size': int(spec['dataset_size']),
        'run_name': str(spec['run_name']),
        'real_hist': real_stats['hist'],
        'generated_hist': generated_stats['hist'],
        'hist_edges': real_stats['hist_edges'],
        'kbins': real_stats['kbins'],
        'pk_ratio': ratio,
        'n_real_exact_model_subset': int(real_stats['n_images']),
        'n_generated': int(generated_stats['n_images']),
        'real_pixel_coverage': float(real_stats['pixel_coverage']),
        'generated_pixel_coverage': float(generated_stats['pixel_coverage']),
        'onepoint_hist_l1': float(
            np.sum(np.abs(generated_stats['hist'] - real_stats['hist']) * widths)
        ),
        'pk_log_ratio_mae': _band_log_error(ratio, 0.0, 1.0),
        'pk_low_log_ratio_mae': _band_log_error(ratio, 0.0, 1.0 / 3.0),
        'pk_mid_log_ratio_mae': _band_log_error(ratio, 1.0 / 3.0, 2.0 / 3.0),
        'pk_high_log_ratio_mae': _band_log_error(ratio, 2.0 / 3.0, 1.0),
        'pk_ratio_median': float(np.nanmedian(ratio)),
        'pk_ratio_min': float(np.nanmin(ratio)),
        'pk_ratio_max': float(np.nanmax(ratio)),
        'max_abs_pk_ratio_minus_1': float(np.nanmax(np.abs(ratio - 1.0))),
        'sample_path': rel(bundle['sample_path']),
        'config_path': rel(bundle['config_path']),
    }
    physical_curve_cache[key] = curve
    return curve


def build_dit_physical_summary() -> pd.DataFrame:
    rows = []
    for arch in DIT_ARCH_ORDER:
        for bundle in choose_bundles(ALL_DATA_TAGS, arch=arch):
            curve = _physical_curve_for_bundle(bundle)
            rows.append({
                name: value for name, value in curve.items()
                if name not in {
                    'real_hist', 'generated_hist', 'hist_edges', 'kbins', 'pk_ratio'
                }
            })
    summary = pd.DataFrame(rows)
    if summary.empty:
        return summary
    summary = summary.sort_values(['arch', 'dataset_size']).reset_index(drop=True)
    TABLE_DIR.mkdir(parents=True, exist_ok=True)
    summary_path = TABLE_DIR / 'nf_generalize_fig2_dit_physical_summary.csv'
    summary.to_csv(summary_path, index=False)
    for arch, sub in summary.groupby('arch', sort=False):
        sub.to_csv(TABLE_DIR / f'nf_generalize_fig2_{arch}_fidelity_summary.csv', index=False)
    print('wrote', summary_path)
    return summary


dit_physical_summary_df = build_dit_physical_summary()
display(dit_physical_summary_df)


def plot_dit_onepoint_pk(
    tags: list[str] = ALL_DATA_TAGS,
    arch: str = DETAIL_ARCH,
    block_name: str = 'full_sweep',
) -> dict[str, Path] | None:
    bundles = choose_bundles(tags, arch=arch)
    if not bundles:
        return None
    curves = [_physical_curve_for_bundle(bundle) for bundle in bundles]
    color = DIT_ARCH_COLORS[arch]
    centers = 0.5 * (
        curves[0]['hist_edges'][:-1] + curves[0]['hist_edges'][1:]
    )

    ncols = 5
    nrows = int(np.ceil(len(curves) / ncols))
    fig_pdf, pdf_axes = plt.subplots(
        nrows, ncols, figsize=(3.55 * ncols, 4.0 * nrows), squeeze=False
    )
    for axis, curve in zip(pdf_axes.ravel(), curves):
        axis.plot(centers, curve['real_hist'], color='black', lw=2.4)
        axis.plot(centers, curve['generated_hist'], color=color, lw=2.2)
        axis.set_yscale('log')
        axis.set_title(dataset_size_label(curve['dataset_size']), pad=8)
        axis.set_xlabel('Normalized field value')
        axis.grid(axis='y', alpha=0.14)
        axis.spines['top'].set_visible(False)
        axis.spines['right'].set_visible(False)
    for axis in pdf_axes.ravel()[len(curves):]:
        axis.set_visible(False)
    for axis in pdf_axes[:, 0]:
        axis.set_ylabel('Pixel PDF')
    fig_pdf.suptitle(
        f'{arch_label(arch)} one-point distributions: {block_name.replace("_", " ")}',
        fontsize=20,
        fontweight='semibold',
        y=0.995,
    )
    fig_pdf.legend(
        handles=[
            Line2D([0], [0], color='black', lw=2.4, label='exact model training subset'),
            Line2D([0], [0], color=color, lw=2.2, label='generated'),
        ],
        loc='upper center', bbox_to_anchor=(0.5, 0.945), ncol=2, frameon=False,
    )
    fig_pdf.subplots_adjust(
        left=0.065, right=0.99, bottom=0.08, top=0.88, hspace=0.38, wspace=0.28
    )
    QUICKCHECK_DIR.mkdir(parents=True, exist_ok=True)
    pdf_out = QUICKCHECK_DIR / f'nf_generalize_fig2_{arch}_{block_name}_onepoint.png'
    fig_pdf.savefig(pdf_out, bbox_inches='tight', dpi=300)
    plt.show()
    print('wrote', pdf_out)

    ratios = np.concatenate([
        curve['pk_ratio'][np.isfinite(curve['pk_ratio'])] for curve in curves
    ])
    shared_upper = max(2.0, float(np.nanquantile(ratios, 0.99)) * 1.08)
    fig_pk, pk_axes = plt.subplots(
        nrows, ncols, figsize=(3.55 * ncols, 3.8 * nrows), squeeze=False
    )
    for axis, curve in zip(pk_axes.ravel(), curves):
        axis.plot(curve['kbins'], curve['pk_ratio'], color=color, marker='o', ms=4, lw=2)
        axis.axhline(1.0, color='black', ls='--', lw=1.4)
        axis.set_ylim(0, shared_upper)
        axis.set_title(dataset_size_label(curve['dataset_size']), pad=8)
        axis.set_xlabel(r'$k$ bin')
        axis.grid(axis='y', alpha=0.14)
        axis.spines['top'].set_visible(False)
        axis.spines['right'].set_visible(False)
    for axis in pk_axes.ravel()[len(curves):]:
        axis.set_visible(False)
    for axis in pk_axes[:, 0]:
        axis.set_ylabel(r'$P_{\rm generated}(k)/P_{\rm real}(k)$')
    fig_pk.suptitle(
        f'{arch_label(arch)} power-spectrum ratios: {block_name.replace("_", " ")}',
        fontsize=20,
        fontweight='semibold',
        y=0.995,
    )
    fig_pk.text(
        0.5, 0.945, 'One is exact agreement; every panel uses the same vertical scale.',
        ha='center', fontsize=12.5, color='0.3',
    )
    fig_pk.subplots_adjust(
        left=0.065, right=0.99, bottom=0.08, top=0.87, hspace=0.36, wspace=0.28
    )
    pk_out = QUICKCHECK_DIR / f'nf_generalize_fig2_{arch}_{block_name}_pk_ratio.png'
    fig_pk.savefig(pk_out, bbox_inches='tight', dpi=300)
    plt.show()
    print('wrote', pk_out)

    return {'onepoint': pdf_out, 'pk_ratio': pk_out}


fidelity_plot_paths = {}
for fidelity_arch in DIT_ARCH_ORDER:
    fidelity_plot_paths[fidelity_arch] = plot_dit_onepoint_pk(
        tags=ALL_DATA_TAGS,
        arch=fidelity_arch,
        block_name='full_sweep',
    )
fidelity_plot_path = fidelity_plot_paths


def plot_physical_error_summaries(summary: pd.DataFrame) -> dict[str, Path] | None:
    if summary.empty:
        return None
    outputs = {}
    fig, axes = plt.subplots(1, 2, figsize=(14.5, 5.3))
    for arch in DIT_ARCH_ORDER:
        sub = summary[summary['arch'] == arch].sort_values('dataset_size')
        x = np.log2(sub['dataset_size']).astype(int)
        style = dict(
            color=DIT_ARCH_COLORS[arch], marker=DIT_ARCH_MARKERS[arch], lw=2.5,
            ms=7, label=arch_label(arch),
        )
        axes[0].plot(x, sub['onepoint_hist_l1'], **style)
        axes[1].plot(x, sub['pk_log_ratio_mae'], **style)
    for axis, title, ylabel in zip(
        axes,
        ['One-point distribution error', 'Power-spectrum error'],
        [r'$L_1$ distance', r'mean $|\log_{10}(P_{gen}/P_{real})|$'],
    ):
        axis.set_xticks(range(6, 16), [rf'$2^{{{i}}}$' for i in range(6, 16)])
        axis.set_xlabel(r'Training images $N_{2D}$')
        axis.set_ylabel(ylabel)
        axis.set_title(title, pad=9)
        axis.grid(axis='y', alpha=0.16)
        axis.spines['top'].set_visible(False)
        axis.spines['right'].set_visible(False)
    fig.suptitle('Physical-statistics error across all DiT training sizes', fontsize=21, fontweight='semibold')
    fig.legend(loc='upper center', bbox_to_anchor=(0.5, 0.90), ncol=3, frameon=False)
    fig.subplots_adjust(left=0.08, right=0.98, bottom=0.15, top=0.75, wspace=0.23)
    out = QUICKCHECK_DIR / 'nf_generalize_fig2_dit_physical_error_by_data_size.png'
    fig.savefig(out, bbox_inches='tight', dpi=300)
    plt.show()
    outputs['total_error'] = out

    fig_band, band_axes = plt.subplots(1, 3, figsize=(17.2, 5.0), sharey=True)
    columns = [
        ('pk_low_log_ratio_mae', 'Low $k$'),
        ('pk_mid_log_ratio_mae', 'Middle $k$'),
        ('pk_high_log_ratio_mae', 'High $k$'),
    ]
    for axis, (column, title) in zip(band_axes, columns):
        for arch in DIT_ARCH_ORDER:
            sub = summary[summary['arch'] == arch].sort_values('dataset_size')
            axis.plot(
                np.log2(sub['dataset_size']).astype(int), sub[column],
                color=DIT_ARCH_COLORS[arch], marker=DIT_ARCH_MARKERS[arch],
                lw=2.3, ms=7, label=arch_label(arch),
            )
        axis.set_xticks(range(6, 16), [rf'$2^{{{i}}}$' for i in range(6, 16)])
        axis.set_xlabel(r'Training images $N_{2D}$')
        axis.set_title(title, pad=9)
        axis.grid(axis='y', alpha=0.16)
        axis.spines['top'].set_visible(False)
        axis.spines['right'].set_visible(False)
    band_axes[0].set_ylabel(r'mean $|\log_{10}(P_{gen}/P_{real})|$')
    fig_band.suptitle('Power-spectrum error by scale', fontsize=21, fontweight='semibold')
    fig_band.legend(loc='upper center', bbox_to_anchor=(0.5, 0.89), ncol=3, frameon=False)
    fig_band.subplots_adjust(left=0.07, right=0.99, bottom=0.16, top=0.73, wspace=0.14)
    band_out = QUICKCHECK_DIR / 'nf_generalize_fig2_dit_pk_error_by_scale.png'
    fig_band.savefig(band_out, bbox_inches='tight', dpi=300)
    plt.show()
    outputs['scale_error'] = band_out
    return outputs


physical_error_plot_paths = plot_physical_error_summaries(dit_physical_summary_df)


def build_novelty_physical_table(summary: pd.DataFrame) -> pd.DataFrame:
    frames = []
    for feature, metrics in [('PCA', pca_metrics), ('SSCD', sscd_metrics)]:
        if metrics.empty or 'gen_gl_q95' not in metrics.columns:
            continue
        metric_rows = ensure_arch_columns(metrics)[
            ['arch', 'dataset_tag', 'dataset_size', 'gen_gl_q95']
        ].drop_duplicates(['arch', 'dataset_tag'])
        joined = summary.merge(
            metric_rows, on=['arch', 'dataset_tag', 'dataset_size'], how='inner'
        )
        joined['feature'] = feature
        frames.append(joined)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


novelty_physical_df = build_novelty_physical_table(dit_physical_summary_df)
if not novelty_physical_df.empty:
    fig_joint, joint_axes = plt.subplots(1, 2, figsize=(14.5, 5.5), sharey=True)
    for axis, feature in zip(joint_axes, ['PCA', 'SSCD']):
        sub_feature = novelty_physical_df[novelty_physical_df['feature'] == feature]
        for arch in DIT_ARCH_ORDER:
            sub = sub_feature[sub_feature['arch'] == arch]
            axis.scatter(
                sub['gen_gl_q95'], sub['pk_log_ratio_mae'],
                color=DIT_ARCH_COLORS[arch], marker=DIT_ARCH_MARKERS[arch],
                s=75, label=arch_label(arch), alpha=0.9,
            )
        axis.axvline(0.5, color='0.4', ls=':', lw=1.4)
        axis.set_xlabel(f'{feature} q95 novelty score')
        axis.set_title(f'{feature} embedding', pad=9)
        axis.grid(alpha=0.15)
        axis.spines['top'].set_visible(False)
        axis.spines['right'].set_visible(False)
    joint_axes[0].set_ylabel(r'mean $|\log_{10}(P_{gen}/P_{real})|$')
    fig_joint.suptitle('DiT novelty versus physical-statistics error', fontsize=21, fontweight='semibold')
    fig_joint.text(
        0.5, 0.89,
        'Useful samples lie toward high novelty and low physical error; novelty alone is insufficient.',
        ha='center', fontsize=12.5, color='0.3',
    )
    handles, labels = joint_axes[0].get_legend_handles_labels()
    fig_joint.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 0.82), ncol=3, frameon=False)
    fig_joint.subplots_adjust(left=0.08, right=0.98, bottom=0.14, top=0.70, wspace=0.16)
    novelty_physical_out = QUICKCHECK_DIR / 'nf_generalize_fig2_dit_novelty_vs_physical_error.png'
    fig_joint.savefig(novelty_physical_out, bbox_inches='tight', dpi=300)
    plt.show()
    print('wrote', novelty_physical_out)

### Reading the physical-statistics figures

These panels use the original fixed-200k samples. For the one-point PDF, overlap with the black curve means the marginal pixel distribution is correct. For $P(k)$, agreement means the ratio stays close to one across all $k$ bins. Departures at high $k$ indicate incorrect small-scale structure even when the one-point distribution looks convincing.

The key DiT-L16 failure is that some low-data samples look novel in PCA/SSCD while their power-spectrum ratio differs from one by factors of several.

## Focused DiT Small-Data Sanity Checks

The DiT curves show suspicious small-data behavior, especially for the deeper models. This section compares the same $N_{2D}=2^6,2^7,2^8,2^9,2^{10}$ runs across DiT-L8, DiT-L12/base, and DiT-L16.

The goal is to separate three possibilities:
1. **real architecture behavior:** deeper DiTs genuinely leave the training-neighbor regime earlier;
2. **metric-specific artifact:** PCA and SSCD disagree, so the effect depends on the embedding;
3. **bad-generation artifact:** loss, images, one-point PDFs, or $P(k)$ look poor even when the generalization score is high.

In [ ]:
SANITY_ARCHES = parse_csv_env('DIT_SANITY_ARCHES', 'dit_l8,dit_base,dit_l16')
SANITY_TAGS = parse_csv_env('DIT_SANITY_TAGS', 'd2p06,d2p07,d2p08,d2p09,d2p10')
SANITY_SAMPLE_INDEX = int(os.environ.get('DIT_SANITY_SAMPLE_INDEX', '0'))


def small_data_metric_comparison(
    arches: list[str] = SANITY_ARCHES,
    tags: list[str] = SANITY_TAGS,
) -> pd.DataFrame:
    frames = []
    for feature, metrics in [('PCA', pca_metrics), ('SSCD', sscd_metrics)]:
        if metrics.empty:
            continue
        tmp = ensure_arch_columns(metrics)
        if 'dataset_tag' not in tmp.columns:
            continue
        sub = tmp[tmp['arch'].astype(str).isin(arches) & tmp['dataset_tag'].astype(str).isin(tags)].copy()
        if sub.empty:
            continue
        keep = [
            'run_name', 'arch', 'arch_label', 'dataset_tag', 'dataset_size',
            'n_generated', 'n_train',
            'gen_gl_q90', 'gen_gl_q95', 'gen_gl_q99',
            'gen_copy_fraction_q90', 'gen_copy_fraction_q95', 'gen_copy_fraction_q99',
            'threshold_q90', 'threshold_q95', 'threshold_q99',
            'gen_nn_median', 'gen_nn_q95',
        ]
        sub = sub[[c for c in keep if c in sub.columns]]
        sub.insert(0, 'feature', feature)
        frames.append(sub)

    if not frames:
        display(Markdown('No PCA/SSCD metrics found for the small-data sanity check.'))
        return pd.DataFrame()

    comp = pd.concat(frames, ignore_index=True)
    comp['arch'] = comp['arch'].astype(str).replace({'dit': 'dit_base'})
    comp['arch_order'] = comp['arch'].map({a: i for i, a in enumerate(DIT_ARCH_ORDER)}).fillna(999)
    comp = comp.sort_values(['arch_order', 'dataset_size', 'feature']).drop(columns=['arch_order'])
    display(comp)

    TABLE_DIR.mkdir(parents=True, exist_ok=True)
    out = TABLE_DIR / 'nf_generalize_fig2_dit_small_data_pca_sscd_comparison.csv'
    comp.to_csv(out, index=False)
    print('wrote', out)
    return comp


small_data_metric_comparison_df = small_data_metric_comparison()

if not small_data_metric_comparison_df.empty and 'gen_gl_q95' in small_data_metric_comparison_df.columns:
    valid_arches = [a for a in SANITY_ARCHES if a in set(small_data_metric_comparison_df['arch'].astype(str))]
    fig, axes = plt.subplots(1, len(valid_arches), figsize=(5.0 * max(1, len(valid_arches)), 4.8), sharey=True, constrained_layout=True)
    if len(valid_arches) == 1:
        axes = [axes]
    for ax, arch in zip(axes, valid_arches):
        arch_df = small_data_metric_comparison_df[small_data_metric_comparison_df['arch'].astype(str) == arch].copy()
        for feature, color, marker in [('PCA', '#0072B2', 'o'), ('SSCD', '#D55E00', 's')]:
            sub = arch_df[arch_df['feature'] == feature].sort_values('dataset_size')
            if sub.empty:
                continue
            ax.plot(
                sub['dataset_size'], sub['gen_gl_q95'],
                marker=marker, ms=7.8, lw=2.5, color=color,
                label=f'{feature} q95',
            )
        ticks = sorted(arch_df['dataset_size'].dropna().astype(int).unique())
        ax.axhline(0.5, color='0.35', lw=1.4, ls=':')
        ax.set_xscale('log', base=2)
        ax.set_xticks(ticks)
        ax.set_xticklabels([dataset_size_label(t) for t in ticks])
        ax.set_ylim(-0.04, 1.04)
        ax.set_title(arch_label(arch))
        ax.set_xlabel(r'Training set size $N_{2D}$')
        ax.grid(alpha=0.22)
        for spine in ['top', 'right']:
            ax.spines[spine].set_visible(False)
    axes[0].set_ylabel('Generalization score')
    handles, labels = axes[-1].get_legend_handles_labels()
    if handles:
        fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 1.06), ncol=2, frameon=False)
    fig.suptitle('Small-data sanity check: PCA vs SSCD generalization', y=1.16)
    out = QUICKCHECK_DIR / 'nf_generalize_fig2_dit_small_data_pca_sscd_q95_by_depth.png'
    QUICKCHECK_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(out, bbox_inches='tight')
    plt.show()
    print('wrote', out)
else:
    display(Markdown('No `gen_gl_q95` column available for focused PCA-vs-SSCD plot.'))

### Reading the small-data novelty comparison

Each panel fixes one DiT depth. Blue and orange show the same generated samples in two representation spaces. Agreement between PCA and SSCD makes the novelty conclusion more robust; disagreement means the result depends on the embedding.

L8 and L12 move from low to high novelty as data increase. L16 is already assigned moderate novelty at several small $N_{2D}$ values, which is suspicious because the corresponding image and $P(k)$ checks are poor.

In [ ]:
small_data_image_paths = {}
for arch in SANITY_ARCHES:
    display(Markdown(f'### {arch_label(arch)} generated maps for $2^6$--$2^{{10}}$'))
    small_data_image_paths[arch] = plot_dit_image_grid(
        sample_index=SANITY_SAMPLE_INDEX,
        tags=SANITY_TAGS,
        arch=arch,
    )

### What to look for across depths

Compare the same training-set-size column across L8, L12, and L16. The DiT-L16 $2^8$ example has visibly noisy or patch-like structure that does not resemble a successful cosmological field. The multi-sample nearest-training audit checks whether that image is a single fluke or a repeated failure mode.

In [ ]:
small_data_fidelity_paths = {}
for arch in SANITY_ARCHES:
    display(Markdown(f'### {arch_label(arch)} one-point PDF and $P(k)$ for $2^6$--$2^{{10}}$'))
    small_data_fidelity_paths[arch] = plot_dit_onepoint_pk(tags=SANITY_TAGS, arch=arch)

### Physical interpretation of the depth comparison

The one-point PDF can remain deceptively close even when spatial structure is wrong. The lower $P(k)$ panels are therefore the stronger warning here. A curve well above one has excess power; a curve below one is missing power. L16 at small and intermediate data sizes does not show a clean monotonic improvement.

In [ ]:
def plot_small_data_loss_curves(
    arch: str,
    tags: list[str] = SANITY_TAGS,
) -> Path | None:
    if manifest_df.empty or 'loss_by_run' not in globals():
        display(Markdown('No manifest/loss cache available for focused loss curves.'))
        return None

    plot_df = manifest_df[
        (manifest_df['arch'].astype(str) == arch)
        & manifest_df['dataset_tag'].astype(str).isin(tags)
    ].copy().sort_values('dataset_size')
    if plot_df.empty:
        display(Markdown(f'No manifest rows found for `{arch_label(arch)}` focused loss curves.'))
        return None

    if 'loss_df' in globals() and isinstance(loss_df, pd.DataFrame) and not loss_df.empty:
        focus_loss = loss_df[
            (loss_df['arch'].astype(str) == arch)
            & loss_df['dataset_tag'].astype(str).isin(tags)
        ].copy()
        focus_cols = [c for c in [
            'arch_label', 'dataset_tag', 'dataset_size', 'n_epoch_loss',
            'final_epoch_loss', 'best_epoch_loss', 'n_batch_loss', 'final_batch_loss',
            'gradient_accumulation_steps', 'metrics_path',
        ] if c in focus_loss.columns]
        if focus_cols:
            display(focus_loss.sort_values('dataset_size')[focus_cols])

    if not any(
        len(np.asarray(loss_by_run.get(str(row['run_name']), {}).get('epoch_loss', [])))
        or len(np.asarray(loss_by_run.get(str(row['run_name']), {}).get('batch_loss', [])))
        for _, row in plot_df.iterrows()
    ):
        display(Markdown(f'No loss arrays found for `{arch_label(arch)}` focused loss curves.'))
        return None

    palette = ['#3B0F70', '#365C8D', '#1F968B', '#73D055', '#FDE725']
    with plt.rc_context({
        'font.family': 'sans-serif',
        'font.sans-serif': ['DejaVu Sans'],
        'mathtext.fontset': 'dejavusans',
        'font.size': 15,
        'axes.labelsize': 18,
        'axes.titlesize': 20,
        'xtick.labelsize': 14,
        'ytick.labelsize': 14,
        'legend.fontsize': 14,
    }):
        fig, axes = plt.subplots(1, 2, figsize=(14.8, 5.8))
        fig.subplots_adjust(left=0.075, right=0.985, bottom=0.20, top=0.67, wspace=0.16)
        for color, (_, row) in zip(palette, plot_df.iterrows()):
            run_name = str(row.get('run_name'))
            info = loss_by_run.get(run_name, {})
            label = dataset_size_label(int(row['dataset_size']))
            steps_per_epoch = max(1, int(row.get('steps_per_epoch', 1) or 1))
            grad_accum = max(1, int(row.get('gradient_accumulation_steps', 1) or 1))

            epoch_loss = np.asarray(info.get('epoch_loss', []), dtype=float)
            if len(epoch_loss):
                optimizer_updates = np.arange(len(epoch_loss), dtype=float) * steps_per_epoch
                x, y = downsample_xy(optimizer_updates, epoch_loss, max_points=900)
                axes[0].plot(x, y, lw=2.2, color=color, label=label)

            batch_loss = np.asarray(info.get('batch_loss', []), dtype=float)
            if len(batch_loss):
                micro_updates = np.arange(len(batch_loss), dtype=float)
                # Convert recorded micro-batches to optimizer updates: micro_updates / grad_accum.
                x = micro_updates / grad_accum
                y = batch_loss.copy()
                window = max(1, len(y) // 900)
                if window > 1:
                    kernel = np.ones(window, dtype=float) / window
                    y = np.convolve(y, kernel, mode='valid')
                    x = x[:len(y)] + 0.5 * (window - 1) / grad_accum
                x, y = downsample_xy(x, y, max_points=900)
                axes[1].plot(x, y, lw=2.0, color=color, label=label)

        axes[0].set_title('Epoch-mean denoising loss', pad=10)
        axes[0].set_ylabel('Training loss')
        axes[1].set_title('Smoothed batch denoising loss', pad=10)
        for ax in axes:
            ax.set_xlabel('Optimizer update')
            ax.set_yscale('log')
            ax.grid(False)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            ax.tick_params(width=1.1, length=5)

        handles, labels = axes[0].get_legend_handles_labels()
        if handles:
            fig.legend(
                handles, labels, title=r'Training images $N_{2D}$',
                loc='upper center', bbox_to_anchor=(0.5, 0.86),
                ncol=len(labels), frameon=False,
            )
        fig.suptitle(f'{arch_label(arch)} optimization history', fontsize=24, y=0.97)
        fig.text(
            0.5, 0.035,
            r'Sawtooth structure follows cosine learning-rate warm restarts ($T_0=4000$ updates).',
            ha='center', va='bottom', fontsize=13.5, color='0.35',
        )

        out = QUICKCHECK_DIR / f'nf_generalize_fig2_{arch}_small_data_loss_curves.png'
        QUICKCHECK_DIR.mkdir(parents=True, exist_ok=True)
        fig.savefig(out, bbox_inches='tight', dpi=300)
        plt.show()
        print('wrote', out)
        return out


small_data_loss_paths = {}
for arch in SANITY_ARCHES:
    display(Markdown(f'### {arch_label(arch)} loss curves for $2^6$--$2^{{10}}$'))
    small_data_loss_paths[arch] = plot_small_data_loss_curves(arch=arch)


### Why the loss curves do not clear L16

All three depths reduce the denoising objective. L16 often reaches the smallest loss because the deeper network can fit the finite training set more aggressively. Since its generated fields can still have incorrect $P(k)$, this is evidence that loss alone is an insufficient checkpoint-selection criterion.

## DiT-L16 validity audit

This audit separates **optimization**, **novelty**, and **physical fidelity**. A high q95 nearest-neighbor score only says that a generated map is not unusually close to the training set in that embedding. It does not show that the generated field follows the target distribution.


In [ ]:
def metric_value_at_n(df: pd.DataFrame, arch: str, n: int, column: str) -> float:
    if df.empty or column not in df.columns:
        return np.nan
    tmp = ensure_arch_columns(df)
    sub = tmp[(tmp['arch'].astype(str) == arch) & (tmp['dataset_size'].astype(float) == float(n))]
    return float(sub.iloc[0][column]) if not sub.empty and pd.notna(sub.iloc[0][column]) else np.nan


def load_yaml_if_exists(path: Path) -> dict[str, Any]:
    if not path.exists():
        return {}
    with path.open() as handle:
        return yaml.safe_load(handle) or {}


def build_l16_validity_audit() -> pd.DataFrame:
    arch = 'dit_l16'
    fidelity_path = TABLE_DIR / 'nf_generalize_fig2_dit_l16_fidelity_summary.csv'
    fidelity = read_csv_if_exists(fidelity_path)
    l16_manifest = manifest_df[
        (manifest_df['arch'].astype(str) == arch)
        & manifest_df['dataset_tag'].astype(str).isin(SANITY_TAGS)
    ].copy().sort_values('dataset_size')
    rows = []

    for _, spec in l16_manifest.iterrows():
        n = int(spec['dataset_size'])
        run_name = str(spec['run_name'])
        config_path = config_path_for(spec)
        config = load_yaml_if_exists(config_path)
        model_cfg = config.get('model', {})
        model_kwargs = model_cfg.get('kwargs', {})
        noise_kwargs = config.get('noise_scheduler', {}).get('kwargs', {})
        train_cfg = config.get('train', {})
        data_cfg = config.get('data', {})
        bundle = loaded.get(run_name, {})
        generated = np.asarray(bundle.get('generated', []))

        configuration_ok = bool(
            model_cfg.get('class') == 'DiTTransformer2DModel'
            and int(model_kwargs.get('num_layers', -1)) == 16
            and int(model_kwargs.get('num_attention_heads', -1)) == 12
            and int(model_kwargs.get('attention_head_dim', -1)) == 64
            and noise_kwargs.get('prediction_type') == 'v_prediction'
            and int(data_cfg.get('constant_label', -1)) == 0
            and int(train_cfg.get('gradient_accumulation_steps', -1)) == 4
        )
        sample_ok = bool(generated.ndim == 4 and len(generated) == 512 and np.isfinite(generated).all())

        loss_sub = loss_df[
            (loss_df['arch'].astype(str) == arch)
            & (loss_df['dataset_size'].astype(float) == float(n))
        ] if 'loss_df' in globals() and not loss_df.empty else pd.DataFrame()
        final_loss = float(loss_sub.iloc[0]['final_epoch_loss']) if not loss_sub.empty else np.nan
        best_loss = float(loss_sub.iloc[0]['best_epoch_loss']) if not loss_sub.empty else np.nan

        fidelity_sub = fidelity[fidelity['dataset_size'].astype(float) == float(n)] if not fidelity.empty else pd.DataFrame()
        pk_median = float(fidelity_sub.iloc[0]['pk_ratio_median']) if not fidelity_sub.empty else np.nan
        pk_max_deviation = float(fidelity_sub.iloc[0]['max_abs_pk_ratio_minus_1']) if not fidelity_sub.empty else np.nan
        pca_q95 = metric_value_at_n(pca_metrics, arch, n, 'gen_gl_q95')
        sscd_q95 = metric_value_at_n(sscd_metrics, arch, n, 'gen_gl_q95')
        novelty_score = float(np.nanmax([pca_q95, sscd_q95])) if np.isfinite([pca_q95, sscd_q95]).any() else np.nan

        if np.isfinite(pk_max_deviation) and pk_max_deviation > 0.5:
            status = 'novel_but_physically_invalid' if novelty_score >= 0.5 else 'physically_invalid'
        elif np.isfinite(pk_max_deviation) and pk_max_deviation > 0.25:
            status = 'fidelity_caution'
        else:
            status = 'fidelity_consistent'

        rows.append({
            'dataset_size': n,
            'configuration_ok': configuration_ok,
            'sample_ok': sample_ok,
            'final_epoch_loss': final_loss,
            'best_epoch_loss': best_loss,
            'pca_gen_gl_q95': pca_q95,
            'sscd_gen_gl_q95': sscd_q95,
            'pca_sscd_gap': abs(pca_q95 - sscd_q95),
            'pk_ratio_median': pk_median,
            'max_abs_pk_ratio_minus_1': pk_max_deviation,
            'interpretation': status,
        })

    audit_df = pd.DataFrame(rows)
    display(audit_df.style.format({
        'final_epoch_loss': '{:.3g}',
        'best_epoch_loss': '{:.3g}',
        'pca_gen_gl_q95': '{:.3f}',
        'sscd_gen_gl_q95': '{:.3f}',
        'pca_sscd_gap': '{:.3f}',
        'pk_ratio_median': '{:.3f}',
        'max_abs_pk_ratio_minus_1': '{:.3f}',
    }))
    out = TABLE_DIR / 'nf_generalize_fig2_dit_l16_validity_audit.csv'
    audit_df.to_csv(out, index=False)
    print('wrote', out)
    return audit_df


l16_validity_audit = build_l16_validity_audit()


if not l16_validity_audit.empty:
    fig, axes = plt.subplots(1, 2, figsize=(13.8, 5.4), sharey=True)
    for ax, (column, title) in zip(
        axes,
        [
            ('pca_gen_gl_q95', 'PCA novelty versus physical error'),
            ('sscd_gen_gl_q95', 'SSCD novelty versus physical error'),
        ],
    ):
        plot_data = l16_validity_audit.dropna(
            subset=[column, 'max_abs_pk_ratio_minus_1']
        )
        ax.scatter(
            plot_data[column],
            plot_data['max_abs_pk_ratio_minus_1'],
            s=95,
            c=np.log2(plot_data['dataset_size']),
            cmap='viridis',
            edgecolor='white',
            linewidth=0.9,
            zorder=3,
        )
        for _, point in plot_data.iterrows():
            exponent = int(round(np.log2(point['dataset_size'])))
            ax.annotate(
                rf'$2^{{{exponent}}}$',
                (point[column], point['max_abs_pk_ratio_minus_1']),
                xytext=(7, 6),
                textcoords='offset points',
                fontsize=11,
            )
        ax.axvline(0.5, color='0.35', ls=':', lw=1.4)
        ax.axhline(0.5, color='0.55', ls='--', lw=1.2)
        ax.set_xlabel('q95 novelty score')
        ax.set_title(title, fontsize=16, pad=10)
        ax.grid(alpha=0.15)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
    axes[0].set_ylabel(r'Maximum $|P_{\mathrm{gen}}/P_{\mathrm{real}} - 1|$')
    fig.suptitle(
        'DiT-L16 novelty must be checked against physical agreement',
        fontsize=20,
        fontweight='semibold',
        y=0.98,
    )
    fig.text(
        0.5,
        0.91,
        'Upper-right points are far from training neighbors but also have a large power-spectrum error.',
        ha='center',
        fontsize=12.5,
        color='0.32',
    )
    fig.subplots_adjust(left=0.08, right=0.98, bottom=0.13, top=0.79, wspace=0.16)
    l16_joint_out = QUICKCHECK_DIR / 'nf_generalize_fig2_dit_l16_novelty_vs_pk_error.png'
    fig.savefig(l16_joint_out, bbox_inches='tight', dpi=300)
    plt.show()
    print('wrote', l16_joint_out)


### Reading the joint novelty-versus-error plot

Moving right means the samples are farther from their nearest training neighbors. Moving down means the power spectrum is closer to the training reference. The desired region is therefore **right and low**.

The problematic L16 points are right and high: they are novel according to PCA or SSCD but physically inconsistent. This is why they must not be counted as evidence that the deeper model generalizes with less data.

### What the audit means

- The L16 runs have the intended 16-layer architecture, null class label, v-prediction scheduler, accumulation setting, and finite 512-sample outputs. That makes a simple routing or wrong-file bug unlikely.
- The denoising loss decreases cleanly and is often lower than for L8/L12. The optimizer is fitting its training objective; low diffusion loss is not a sufficient model-selection metric.
- At small $N_{2D}$, L16 can receive a high nearest-neighbor novelty score while its $P(k)$ is wrong by factors of several. Those cases are **novel but physically invalid**, not early generalization.
- The most likely interpretation is that the deeper model is more data- and hyperparameter-sensitive. Confirm it with another seed and a learning-rate/regularization ablation before making a capacity claim.


## DiT Generalization Curves

The q95 score measures nearest-neighbor novelty relative to a real-data baseline. A score near zero means generated fields are unusually close to training slices. **High score means unlike the training set; it does not guarantee physical fidelity.** Interpret these curves together with the image, one-point, and $P(k)$ checks above.


In [ ]:
def format_power_ticks(ax, values):
    vals = sorted({int(v) for v in values if pd.notna(v) and v > 0})
    if not vals:
        return
    ax.set_xscale('log', base=2)
    ax.set_xticks(vals)
    ax.set_xticklabels([rf'$2^{{{int(round(math.log2(v)))}}}$' for v in vals])


def plot_dit_generalization_curves(metrics_by_feature: dict[str, pd.DataFrame], quantile: str = 'q95') -> Path | None:
    gl_col = f'gen_gl_{quantile}'
    fig, axes = plt.subplots(1, 2, figsize=(13.0, 5.3), sharey=True, constrained_layout=True)
    plotted = False

    for ax, (feature_name, df) in zip(axes, metrics_by_feature.items()):
        all_x = []
        if df.empty or gl_col not in df.columns or 'dataset_size' not in df.columns:
            ax.set_visible(False)
            continue
        df = ensure_arch_columns(df)
        for arch in DIT_ARCH_ORDER:
            sub = df[df['arch'].astype(str) == arch].dropna(subset=['dataset_size', gl_col]).sort_values('dataset_size')
            if sub.empty:
                continue
            all_x.extend(sub['dataset_size'].astype(float).tolist())
            ax.plot(
                sub['dataset_size'], sub[gl_col],
                marker=DIT_ARCH_MARKERS.get(arch, 'o'), ms=8, lw=3,
                color=DIT_ARCH_COLORS.get(arch, '0.2'), label=arch_label(arch),
            )
            plotted = True
        format_power_ticks(ax, all_x)
        ax.axhline(0.5, color='0.35', lw=1.5, ls=':', label='0.5 marker')
        ax.set_ylim(-0.04, 1.04)
        ax.set_xlabel(r'Training set size $N_{2D}$')
        ax.set_title(f'{feature_name}: DiT depth sweep at 200k updates ({quantile})')
        ax.grid(True, alpha=0.22)
        ax.legend(frameon=False, loc='lower right')
        for spine in ['top', 'right']:
            ax.spines[spine].set_visible(False)
    axes[0].set_ylabel('Generalization score')

    if not plotted:
        display(Markdown(f'No `{gl_col}` columns found to plot.'))
        plt.close(fig)
        return None

    QUICKCHECK_DIR.mkdir(parents=True, exist_ok=True)
    out = QUICKCHECK_DIR / f'nf_generalize_fig2_dit_depth_gl_curves_{quantile}.png'
    fig.savefig(out, bbox_inches='tight')
    plt.show()
    print('wrote', out)
    return out

combined_curve = plot_dit_generalization_curves({'PCA': pca_metrics, 'SSCD': sscd_metrics}, quantile='q95')

### Reading the DiT generalization curves

The horizontal 0.5 line is a descriptive midpoint, not a universal physical threshold. Curves that move from near zero to near one show a transition away from training-neighbor behavior as data increase.

L8 and L12 show the expected qualitative transition. The nonmonotonic L16 curve should be read as a failed validity check, not as a surprising reversal of the capacity relationship.

## Transition Summary

`N50` is the interpolated training-set size where the generalization score crosses 0.5. The table below uses the original fixed 200k-update sweep. This is a compact comparison, but it should be read with the full curve and the later L16 continuation experiment because a single midpoint can hide optimization sensitivity, changes in slope, or physically invalid samples.


In [ ]:
def interpolate_crossing(df: pd.DataFrame, ycol: str, threshold: float = 0.5) -> dict[str, Any]:
    if df.empty or ycol not in df.columns or 'dataset_size' not in df.columns:
        return {'status': 'missing', 'n_cross': np.nan, 'log2_n_cross': np.nan}
    sub = df[['dataset_size', ycol]].dropna().sort_values('dataset_size')
    sub = sub[sub['dataset_size'] > 0]
    if sub.empty:
        return {'status': 'missing', 'n_cross': np.nan, 'log2_n_cross': np.nan}

    x = np.log2(sub['dataset_size'].astype(float).to_numpy())
    y = sub[ycol].astype(float).to_numpy()
    if y[0] >= threshold:
        return {'status': 'left_censored', 'n_cross': 2 ** x[0], 'log2_n_cross': x[0]}
    if y[-1] < threshold:
        return {'status': 'right_censored', 'n_cross': 2 ** x[-1], 'log2_n_cross': x[-1]}

    for i in range(len(y) - 1):
        y0, y1 = y[i], y[i + 1]
        if (y0 <= threshold <= y1) or (y1 <= threshold <= y0):
            if y1 == y0:
                xc = x[i]
            else:
                frac = (threshold - y0) / (y1 - y0)
                xc = x[i] + frac * (x[i + 1] - x[i])
            return {'status': 'interpolated', 'n_cross': 2 ** xc, 'log2_n_cross': xc}
    return {'status': 'not_found', 'n_cross': np.nan, 'log2_n_cross': np.nan}


rows = []
for feature_name, df in [('PCA', pca_metrics), ('SSCD', sscd_metrics)]:
    df = ensure_arch_columns(df)
    for arch in DIT_ARCH_ORDER:
        sub = df[df['arch'].astype(str) == arch]
        for q in ['q90', 'q95', 'q99']:
            col = f'gen_gl_{q}'
            result = interpolate_crossing(sub, col, threshold=0.5)
            rows.append({
                'feature': feature_name,
                'arch': arch,
                'arch_label': arch_label(arch),
                'score_col': col,
                'threshold': 0.5,
                **result,
            })

transition_df = pd.DataFrame(rows)
display(transition_df.sort_values(['feature', 'arch', 'score_col']))

if len(transition_df):
    out = TABLE_DIR / 'nf_generalize_fig2_dit_transition_summary.csv'
    TABLE_DIR.mkdir(parents=True, exist_ok=True)
    transition_df.to_csv(out, index=False)
    print('wrote', out)

## Complete 200k-update depth comparison

This is the main architecture comparison. Every DiT depth uses the same 200k optimizer-update budget and is shown over the complete $2^6$ to $2^{15}$ data-size sweep. The UNet curves are fixed-budget references. The later DiT-L16 continuation covers only $2^6$ to $2^{10}$, so it is kept separate and is not used as a depth-scaling curve.


In [ ]:
UNET_RESULTS_DIR = PROJECT_DIR / 'results' / 'nf_generalize_fig2'
UNET_TABLE_DIR = UNET_RESULTS_DIR / 'tables'

unet_pca = add_generalization_columns(read_csv_if_exists(UNET_TABLE_DIR / 'nf_generalize_fig2_pca_full_nn_metrics.csv'))
unet_sscd = add_generalization_columns(read_csv_if_exists(UNET_TABLE_DIR / 'nf_generalize_fig2_sscd_full_nn_metrics.csv'))


def plot_dit_vs_unet_combined(
    dit_by_feature: dict[str, pd.DataFrame],
    unet_by_feature: dict[str, pd.DataFrame],
    quantile: str = 'q95',
) -> Path | None:
    gl_col = f'gen_gl_{quantile}'
    feature_order = [('PCA', 'PCA embedding'), ('SSCD', 'SSCD embedding')]
    if not any(not df.empty and gl_col in df.columns for df in dit_by_feature.values()):
        display(Markdown(f'No DiT `{gl_col}` data available.'))
        return None

    dit_colors = {'dit_l8': '#009E73', 'dit_base': '#0072B2', 'dit_l16': '#CC79A7'}
    unet_colors = {'u64': '#B8B8B8', 'u128': '#858585', 'u256': '#505050'}
    unet_dashes = {'u64': (0, (5, 3)), 'u128': (0, (2, 2)), 'u256': (0, (8, 3))}

    with plt.rc_context({
        'font.family': 'sans-serif',
        'font.sans-serif': ['DejaVu Sans'],
        'mathtext.fontset': 'dejavusans',
        'font.size': 16,
        'axes.labelsize': 20,
        'axes.titlesize': 22,
        'xtick.labelsize': 16,
        'ytick.labelsize': 16,
        'legend.fontsize': 14,
    }):
        fig, axes = plt.subplots(1, 2, figsize=(17.0, 6.5), sharey=True)
        fig.subplots_adjust(left=0.075, right=0.985, bottom=0.15, top=0.70, wspace=0.08)

        for ax, (feature_key, panel_title) in zip(axes, feature_order):
            dit_df = ensure_arch_columns(dit_by_feature.get(feature_key, pd.DataFrame()))
            unet_df = ensure_arch_columns(unet_by_feature.get(feature_key, pd.DataFrame()))
            all_x = []

            if not unet_df.empty and gl_col in unet_df.columns:
                for arch in UNET_ARCH_ORDER:
                    sub = unet_df[unet_df['arch'].astype(str) == arch].dropna(
                        subset=['dataset_size', gl_col]
                    ).sort_values('dataset_size')
                    if sub.empty:
                        continue
                    all_x.extend(sub['dataset_size'].astype(float).tolist())
                    ax.plot(
                        sub['dataset_size'], sub[gl_col],
                        color=unet_colors[arch], linestyle=unet_dashes[arch],
                        marker=UNET_ARCH_MARKERS[arch], markerfacecolor='white',
                        markeredgewidth=1.4, lw=1.9, ms=7.0, alpha=0.95,
                        zorder=1,
                    )

            if not dit_df.empty and gl_col in dit_df.columns:
                for arch in DIT_ARCH_ORDER:
                    sub = dit_df[dit_df['arch'].astype(str) == arch].dropna(
                        subset=['dataset_size', gl_col]
                    ).sort_values('dataset_size')
                    if sub.empty:
                        continue
                    all_x.extend(sub['dataset_size'].astype(float).tolist())
                    ax.plot(
                        sub['dataset_size'], sub[gl_col],
                        color=dit_colors[arch], marker=DIT_ARCH_MARKERS[arch],
                        markeredgecolor='white', markeredgewidth=0.8,
                        lw=3.5, ms=9.5, zorder=3,
                    )

            ax.axhline(0.5, color='0.35', lw=1.4, ls=':', zorder=0)
            format_power_ticks(ax, all_x)
            ax.set_ylim(-0.035, 1.035)
            ax.set_xlabel(r'Training images $N_{2D}$', labelpad=8)
            ax.set_title(panel_title, pad=12, fontweight='semibold')
            ax.grid(False)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            ax.tick_params(width=1.1, length=5)

        axes[0].set_ylabel('q95 novelty score', labelpad=8)
        axes[1].tick_params(labelleft=False)
        axes[0].text(66, 0.515, '0.5 reference', fontsize=12.5, color='0.35', va='bottom')

        legend_handles = []
        legend_labels = []
        for arch in UNET_ARCH_ORDER:
            legend_handles.append(Line2D(
                [0], [0], color=unet_colors[arch], linestyle=unet_dashes[arch],
                marker=UNET_ARCH_MARKERS[arch], markerfacecolor='white',
                markeredgewidth=1.3, lw=1.9, ms=7.0,
            ))
            legend_labels.append(arch_label(arch))
        for arch in DIT_ARCH_ORDER:
            legend_handles.append(Line2D(
                [0], [0], color=dit_colors[arch], marker=DIT_ARCH_MARKERS[arch],
                markeredgecolor='white', markeredgewidth=0.8, lw=3.5, ms=9.0,
            ))
            legend_labels.append(arch_label(arch))

        fig.suptitle('DiT depth sweep at fixed 200k updates', fontsize=27, y=0.97, fontweight='semibold')
        fig.text(
            0.5, 0.895,
            'UNet references and all DiT depths use 200k optimizer updates. High novelty does not guarantee physical fidelity.',
            ha='center', va='center', fontsize=16.5, color='0.28',
        )
        fig.legend(
            legend_handles, legend_labels,
            loc='upper center', bbox_to_anchor=(0.5, 0.835),
            ncol=6, frameon=False, handlelength=2.2, columnspacing=1.5,
        )

        out = QUICKCHECK_DIR / f'nf_generalize_fig2_dit_depth_vs_unet_pca_sscd_{quantile}.png'
        QUICKCHECK_DIR.mkdir(parents=True, exist_ok=True)
        fig.savefig(out, bbox_inches='tight', dpi=300)
        plt.show()
        print('wrote', out)
        return out


combined_dit_unet_comparison = plot_dit_vs_unet_combined(
    {'PCA': pca_metrics, 'SSCD': sscd_metrics},
    {'PCA': unet_pca, 'SSCD': unet_sscd},
    quantile='q95',
)


### Main fixed-budget comparison

This is the cleanest architecture comparison because every displayed DiT depth uses the same 200k-update budget. The L8-to-L12 shift is qualitatively consistent with a deeper model needing more data before leaving the copy-like regime.

The L16 points do not extend that trend cleanly because several small-data outputs are physically invalid. The clean 300k L16 rerun is evaluated separately rather than silently replacing points in this fixed-budget figure.

## Appendix: Exploratory Capacity Check

This is a compact summary of the original fixed-200k sweep. The x-axis is parameter count and the y-axis is the interpolated data size where q95 novelty crosses 0.5.

The points are deliberately **not connected or fit with a line**. DiT-L16 has physically invalid small-data samples, so its apparent crossing cannot support a capacity-scaling claim. Use this figure only to locate hypotheses for a clean rerun.


In [ ]:
# Parameter counts.
# UNet values are exact from the existing sweep. DiT values are approximate for width 768,
# patch 8, and depths 8/12/16; replace with print_model_param_count.py values if needed.
MODEL_CAPACITY = pd.DataFrame([
    {'model': 'UNet-64',        'family': 'UNet', 'arch_key': 'u64',      'model_params': 26_621_057,  'source': 'exact'},
    {'model': 'UNet-128',       'family': 'UNet', 'arch_key': 'u128',     'model_params': 140_539_521, 'source': 'exact'},
    {'model': 'UNet-256',       'family': 'UNet', 'arch_key': 'u256',     'model_params': 196_059_905, 'source': 'exact'},
    {'model': 'DiT-L8',         'family': 'DiT',  'arch_key': 'dit_l8',   'model_params': 95_000_000,  'source': 'approx'},
    {'model': 'DiT-L12 / base', 'family': 'DiT',  'arch_key': 'dit_base', 'model_params': 138_290_000, 'source': 'approx'},
    {'model': 'DiT-L16',        'family': 'DiT',  'arch_key': 'dit_l16',  'model_params': 182_000_000, 'source': 'approx'},
])
MODEL_CAPACITY['model_params_m'] = MODEL_CAPACITY['model_params'] / 1e6


def n50_for_model(feature_name: str, model: str, family: str, model_params: int, df: pd.DataFrame, q: str = 'q95') -> dict[str, Any]:
    col = f'gen_gl_{q}'
    cross = interpolate_crossing(df, col, threshold=0.5)
    return {
        'feature': feature_name,
        'model': model,
        'family': family,
        'model_params': model_params,
        'model_params_m': model_params / 1e6,
        'score_col': col,
        **cross,
    }


def capacity_transition_table(q: str = 'q95') -> pd.DataFrame:
    rows = []
    feature_frames = {
        'PCA': (pca_metrics, unet_pca),
        'SSCD': (sscd_metrics, unet_sscd),
    }
    for feature_name, (dit_df, unet_df) in feature_frames.items():
        dit_tmp = ensure_arch_columns(dit_df)
        for arch in DIT_ARCH_ORDER:
            cap = MODEL_CAPACITY[MODEL_CAPACITY['arch_key'] == arch]
            if cap.empty:
                continue
            cap = cap.iloc[0]
            sub = dit_tmp[dit_tmp['arch'].astype(str) == arch]
            rows.append(n50_for_model(feature_name, cap['model'], cap['family'], int(cap['model_params']), sub, q=q))

        if unet_df.empty:
            continue
        unet_tmp = ensure_arch_columns(unet_df)
        for arch in UNET_ARCH_ORDER:
            cap = MODEL_CAPACITY[MODEL_CAPACITY['arch_key'] == arch].iloc[0]
            sub = unet_tmp[unet_tmp['arch'].astype(str) == arch]
            rows.append(n50_for_model(feature_name, cap['model'], cap['family'], int(cap['model_params']), sub, q=q))
    out = pd.DataFrame(rows)
    return out.sort_values(['feature', 'family', 'model_params']).reset_index(drop=True)


capacity_n50 = capacity_transition_table(q='q95')
display(capacity_n50[['feature', 'model', 'family', 'model_params_m', 'status', 'n_cross', 'log2_n_cross']])

# Pairwise comparison against nearby UNet capacities.
ratio_rows = []
for feature_name in ['PCA', 'SSCD']:
    sub = capacity_n50[capacity_n50['feature'] == feature_name].set_index('model')
    for dit_model in ['DiT-L8', 'DiT-L12 / base', 'DiT-L16']:
        if dit_model not in sub.index:
            continue
        for baseline in ['UNet-128', 'UNet-256']:
            if baseline not in sub.index:
                continue
            ratio_rows.append({
                'feature': feature_name,
                'comparison': f'{dit_model} / {baseline}',
                'param_ratio': sub.loc[dit_model, 'model_params'] / sub.loc[baseline, 'model_params'],
                'n50_ratio': sub.loc[dit_model, 'n_cross'] / sub.loc[baseline, 'n_cross'],
                'dit_log2_n50': sub.loc[dit_model, 'log2_n_cross'],
                'baseline_log2_n50': sub.loc[baseline, 'log2_n_cross'],
            })
capacity_ratios = pd.DataFrame(ratio_rows)
display(capacity_ratios)

fig, axes = plt.subplots(1, 2, figsize=(14.2, 5.6), sharey=True)
label_offsets = {
    'UNet-64': (8, 7),
    'UNet-128': (8, -17),
    'UNet-256': (-58, 8),
    'DiT-L8': (8, 8),
    'DiT-L12 / base': (-86, 10),
    'DiT-L16': (-58, -18),
}
family_style = {
    'UNet': {'color': '#6E6E6E', 'marker': 'o', 'label': 'UNet width'},
    'DiT': {'color': '#0072B2', 'marker': 'D', 'label': 'DiT depth'},
}
for ax, feature_name in zip(axes, ['PCA', 'SSCD']):
    sub = capacity_n50[
        (capacity_n50['feature'] == feature_name)
        & capacity_n50['n_cross'].notna()
    ].copy()
    if sub.empty:
        ax.set_visible(False)
        continue
    for family, style in family_style.items():
        family_data = sub[sub['family'] == family]
        ax.scatter(
            family_data['model_params'],
            family_data['n_cross'],
            color=style['color'],
            marker=style['marker'],
            s=90,
            edgecolor='white',
            linewidth=0.9,
            label=style['label'],
            zorder=3,
        )
        for _, row in family_data.iterrows():
            offset = label_offsets.get(row['model'], (7, 7))
            annotation_color = '#8B1A1A' if row['model'] == 'DiT-L16' else '0.18'
            ax.annotate(
                row['model'],
                (row['model_params'], row['n_cross']),
                xytext=offset,
                textcoords='offset points',
                fontsize=10.5,
                color=annotation_color,
            )
    ax.set_xscale('log')
    ax.set_yscale('log', base=2)
    ax.set_xlabel('Trainable parameters')
    ax.set_title(f'{feature_name} q95 crossing', fontsize=17, pad=10)
    ax.grid(axis='y', alpha=0.16)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
axes[0].set_ylabel(r'$N_{50}$ training images')
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc='upper center',
    bbox_to_anchor=(0.5, 0.88),
    ncol=2,
    frameon=False,
)
fig.suptitle(
    'Exploratory capacity diagnostic at fixed 200k updates',
    fontsize=21,
    fontweight='semibold',
    y=0.985,
)
fig.text(
    0.5,
    0.92,
    'Points are shown without a fitted or connecting line. DiT-L16 is not valid evidence for a scaling law.',
    ha='center',
    fontsize=12.5,
    color='0.32',
)
fig.subplots_adjust(left=0.09, right=0.98, bottom=0.14, top=0.74, wspace=0.16)

out = QUICKCHECK_DIR / 'nf_generalize_fig2_dit_depth_capacity_n50_q95.png'
fig.savefig(out, bbox_inches='tight', dpi=300)
plt.show()
print('wrote', out)

if not capacity_ratios.empty:
    lines = ['### Fixed-200k capacity interpretation']
    for _, row in capacity_ratios.iterrows():
        if not np.isfinite(row['n50_ratio']):
            continue
        lines.append(
            f"- {row['feature']}: {row['comparison']} has {row['param_ratio']:.2f}x parameters "
            f"and {row['n50_ratio']:.2f}x the q95 N50."
        )
    lines.append('- This is an exploratory fixed-budget diagnostic, not a universal scaling law. The DiT-L16 point is not reliable capacity evidence while some high-novelty samples are physically invalid and its transition changes with continued optimization.')
    display(Markdown('\n'.join(lines)))

### Why this is only exploratory

$N_{50}$ compresses each full transition curve into one interpolated number. That is convenient, but it hides nonmonotonicity, slope, and physical failures. The absence of connecting lines is intentional: these six points do not yet establish a fitted scaling relation.

## Appendix: Saved Diagnostic Inventory

The notebook already creates the main figures above. Re-displaying every saved PNG made the notebook long and repeated the same evidence. This section now lists saved files and only displays them when explicitly requested.


In [ ]:
saved_diagnostic_specs = [
    ('Training-loss comparison', QUICKCHECK_DIR / 'nf_generalize_fig2_dit_training_curves.png'),
    ('PCA/SSCD depth curves', QUICKCHECK_DIR / 'nf_generalize_fig2_dit_depth_gl_curves_q95.png'),
    ('DiT depth versus UNet', QUICKCHECK_DIR / 'nf_generalize_fig2_dit_depth_vs_unet_pca_sscd_q95.png'),
    ('Exploratory capacity diagnostic', QUICKCHECK_DIR / 'nf_generalize_fig2_dit_depth_capacity_n50_q95.png'),
    ('PCA paper-style curve', QUICKCHECK_DIR / 'nf_generalize_fig2_dit_pca_full_nn_paper_style_gl_curves.png'),
    ('SSCD paper-style curve', QUICKCHECK_DIR / 'nf_generalize_fig2_dit_sscd_full_nn_paper_style_gl_curves.png'),
]
for arch in DIT_ARCH_ORDER:
    saved_diagnostic_specs.extend([
        (
            f'{arch_label(arch)} generated image grid',
            QUICKCHECK_DIR / f'nf_generalize_fig2_{arch}_full_sweep_generated_image_grid.png',
        ),
        (
            f'{arch_label(arch)} one-point distributions',
            QUICKCHECK_DIR / f'nf_generalize_fig2_{arch}_full_sweep_onepoint.png',
        ),
        (
            f'{arch_label(arch)} power-spectrum ratios',
            QUICKCHECK_DIR / f'nf_generalize_fig2_{arch}_full_sweep_pk_ratio.png',
        ),
    ])

saved_diagnostic_inventory = pd.DataFrame([
    {
        'diagnostic': title,
        'path': rel(path),
        'exists': path.exists(),
        'size_mb': path.stat().st_size / 1024**2 if path.exists() else np.nan,
    }
    for title, path in saved_diagnostic_specs
])
display(saved_diagnostic_inventory)

if os.environ.get('DIT_SHOW_SAVED_DIAGNOSTICS', '0') == '1':
    for title, path in saved_diagnostic_specs:
        display(Markdown(f'### {title}'))
        if path.exists():
            display(Image(filename=str(path), width=950))
        else:
            display(Markdown(f'Missing: `{rel(path)}`'))
else:
    display(Markdown(
        'Duplicate figures are hidden by default. Set '
        '`DIT_SHOW_SAVED_DIAGNOSTICS=1` before running the notebook to display them.'
    ))

## Sample File Sanity Check

This inspects array keys and shapes without doing the expensive nearest-neighbor analysis again.


In [ ]:
def inspect_npz(path: Path) -> dict[str, Any]:
    if not path.exists():
        return {'exists': False}
    with np.load(path) as data:
        keys = list(data.files)
        first = keys[0] if keys else None
        arr = data[first] if first else None
        return {
            'exists': True,
            'keys': ', '.join(keys[:8]),
            'first_key': first,
            'shape': tuple(arr.shape) if arr is not None else None,
            'dtype': str(arr.dtype) if arr is not None else None,
            'size_mb': path.stat().st_size / 1024**2,
        }

if manifest_df.empty:
    display(Markdown('No manifest available, so sample files were not inspected.'))
else:
    rows = []
    for _, row in manifest_df.sort_values(['arch', 'dataset_size']).iterrows():
        path = row['sample_path_resolved']
        info = inspect_npz(path)
        rows.append({
            'arch_label': row.get('arch_label'),
            'dataset_tag': row.get('dataset_tag'),
            'dataset_size': row.get('dataset_size'),
            'path': rel(path),
            **info,
        })
    sample_inspect_df = pd.DataFrame(rows)
    display(sample_inspect_df)

## Controlled DiT-L16 continuation: low-data diagnostic, not a scaling curve

This section inspects DiT-L16 at $N_{2D}=2^6,\ldots,2^{10}$ across the original 200k sample and the saved 225k, 250k, 275k, and 300k samples. It does **not** cover $2^{11}$ to $2^{15}$, so it cannot establish whether the full depth transition moves right with additional optimization.

### Resume-state warning

The currently saved 225k to 300k samples were generated by an older class-safe resume loader that restored the DiT weights but recreated the optimizer and learning-rate scheduler state at each 25k stage. AdamW moments and scheduler progress were therefore reset. These files are useful for diagnosing failure modes, but they are not a controlled longer-training trajectory and are excluded from the scaling comparison above.

The corrected loader restores `optimizer.pkl`, `lr_scheduler.pkl`, and `noise_scheduler.pkl` and fails if those files are missing. Rerun the continuation from the original 200k checkpoints before using later checkpoints to make a training-dynamics claim.

Every physical-statistics comparison below reloads the **complete configured training reference**. A run is rejected if the number of real slices does not equal its manifest `dataset_size`. Novelty must be read together with generated images, the one-point PDF, and $P(k)$: off-distribution noise can be far from every training image and therefore receive a high novelty score.


In [ ]:
CONTINUE_MANIFEST_PATH = PROJECT_DIR / 'local' / 'nf_generalize_fig2_dit_l16_continue' / 'manifest.json'
CONTINUE_ANALYSIS_MANIFEST_PATH = CONTINUE_MANIFEST_PATH.parent / 'analysis_manifest.json'
CONTINUE_CHECKPOINTS = [
    (200, 'dpm50'),
    (225, 'dpm50_cont_225k'),
    (250, 'dpm50_cont_250k'),
    (275, 'dpm50_cont_275k'),
    (300, 'dpm50_cont_300k'),
]
CONTINUE_TAGS = ['d2p06', 'd2p07', 'd2p08', 'd2p09', 'd2p10']
CONTINUE_DETAIL_TAG = os.environ.get('DIT_CONTINUE_DETAIL_TAG', 'd2p08')

continue_manifest_obj = read_json(CONTINUE_MANIFEST_PATH)
continue_analysis_obj = read_json(CONTINUE_ANALYSIS_MANIFEST_PATH)
if continue_manifest_obj is None:
    display(Markdown(f'Continuation manifest not created yet: `{rel(CONTINUE_MANIFEST_PATH)}`'))
    continue_manifest_df = pd.DataFrame()
else:
    continue_manifest_df = pd.DataFrame(continue_manifest_obj)
    display(continue_manifest_df[[
        'continue_stage', 'run_name', 'dataset_tag', 'dataset_size',
        'target_total_updates', 'expected_checkpoint', 'sample_label',
    ]].sort_values(['continue_stage', 'dataset_size']))

if continue_analysis_obj is None:
    continue_base_df = pd.DataFrame()
else:
    continue_base_df = ensure_arch_columns(pd.DataFrame(continue_analysis_obj))
    continue_base_df = continue_base_df[
        (continue_base_df['arch'].astype(str) == 'dit_l16')
        & continue_base_df['dataset_tag'].isin(CONTINUE_TAGS)
    ].sort_values('dataset_size')


def continuation_sample_path(run_name: str, label: str) -> Path:
    return SAMPLE_DIR / f'{run_name}_seed{SEED}_{label}.npz'


def continuation_table_path(feature: str, updates_k: int) -> Path:
    return TABLE_DIR / f'nf_generalize_fig2_dit_l16_cont_{updates_k}k_{feature.lower()}_full_nn_metrics.csv'


sample_audit_rows = []
for _, row in continue_base_df.iterrows():
    for updates_k, label in CONTINUE_CHECKPOINTS:
        sample_path = continuation_sample_path(str(row['run_name']), label)
        sample_audit_rows.append({
            'dataset_tag': row['dataset_tag'],
            'dataset_size': int(row['dataset_size']),
            'updates_k': updates_k,
            'sample_label': label,
            'sample_exists': sample_path.exists(),
            'sample_path': rel(sample_path),
        })
continuation_sample_audit_df = pd.DataFrame(sample_audit_rows)
display(continuation_sample_audit_df)


In [ ]:
continuation_fidelity_rows = []
continuation_curve_cache = {}
continuation_image_cache = {}

if continue_base_df.empty or not SIMDIFF_EVAL_AVAILABLE:
    display(Markdown('Continuation physical-statistics check is waiting for its manifest or `simdiff_eval`.'))
else:
    for _, row in continue_base_df.iterrows():
        run_name = str(row['run_name'])
        dataset_tag = str(row['dataset_tag'])
        dataset_size = int(row['dataset_size'])
        cfg_path = config_path_for(row)
        real = as_nchw(load_real_from_config(cfg_path, max_raw_samples=None))
        if len(real) != dataset_size:
            raise RuntimeError(
                'FULL TRAINING REFERENCE MISMATCH: '
                f'{run_name} loaded {len(real)} real slices from {cfg_path}; expected {dataset_size}.'
            )

        real_hist = field_histogram(real, bins=140)
        edges = np.asarray(real_hist['bin_edges'], dtype=float)
        centers = 0.5 * (edges[:-1] + edges[1:])
        widths = np.diff(edges)
        real_density = np.asarray(real_hist['hist'], dtype=float)
        pk_real, kbins = batch_power_spectra(real, nbins=PK_NBINS)
        mean_pk_real = np.clip(np.nanmean(pk_real, axis=0), 1e-30, None)

        for updates_k, label in CONTINUE_CHECKPOINTS:
            sample_path = continuation_sample_path(run_name, label)
            if not sample_path.exists():
                continue
            generated = evenly_limit(load_npz_array(sample_path), MAX_GENERATED)
            generated_density, _ = np.histogram(generated.ravel(), bins=edges, density=True)
            hist_l1 = float(np.sum(np.abs(generated_density - real_density) * widths))
            pk_generated, _ = batch_power_spectra(generated, nbins=PK_NBINS)
            ratio = np.nanmean(pk_generated, axis=0) / mean_pk_real
            finite = np.isfinite(ratio) & (ratio > 0)
            pk_log10_mae = float(np.mean(np.abs(np.log10(ratio[finite])))) if finite.any() else np.nan
            thirds = np.array_split(np.where(finite)[0], 3) if finite.any() else [[], [], []]
            band_ratio = [float(np.nanmean(ratio[idx])) if len(idx) else np.nan for idx in thirds]

            resolved_checkpoint = 'missing (legacy 200k sample)'
            with np.load(sample_path, allow_pickle=False) as data:
                if 'resolved_checkpoint' in data.files:
                    resolved_checkpoint = str(np.asarray(data['resolved_checkpoint']).item())

            continuation_fidelity_rows.append({
                'run_name': run_name,
                'dataset_tag': dataset_tag,
                'dataset_size': dataset_size,
                'updates_k': updates_k,
                'sample_label': label,
                'n_real': len(real),
                'n_generated': len(generated),
                'real_reference_kind': 'complete configured training set',
                'real_config_path': str(cfg_path),
                'resolved_checkpoint': resolved_checkpoint,
                'hist_l1': hist_l1,
                'pk_log10_mae': pk_log10_mae,
                'pk_ratio_low_k': band_ratio[0],
                'pk_ratio_mid_k': band_ratio[1],
                'pk_ratio_high_k': band_ratio[2],
            })
            continuation_curve_cache[(dataset_tag, updates_k)] = {
                'centers': centers.copy(),
                'real_density': real_density.copy(),
                'generated_density': generated_density.copy(),
                'kbins': np.asarray(kbins).copy(),
                'pk_ratio': np.asarray(ratio).copy(),
            }
            continuation_image_cache[(dataset_tag, updates_k)] = generated[:4, 0].copy()

continuation_fidelity_df = pd.DataFrame(continuation_fidelity_rows)
if continuation_fidelity_df.empty:
    display(Markdown('No continuation samples are available yet. Rerun this section after the staged sampler jobs finish.'))
else:
    display(continuation_fidelity_df.sort_values(['dataset_size', 'updates_k']))
    fidelity_out = TABLE_DIR / 'nf_generalize_fig2_dit_l16_continuation_fidelity.csv'
    continuation_fidelity_df.to_csv(fidelity_out, index=False)
    print('wrote', fidelity_out)

    fig, axes = plt.subplots(1, 2, figsize=(13.2, 5.0), constrained_layout=True)
    colors = plt.cm.viridis(np.linspace(0.08, 0.90, len(CONTINUE_TAGS)))
    for color, tag in zip(colors, CONTINUE_TAGS):
        sub = continuation_fidelity_df[continuation_fidelity_df['dataset_tag'] == tag].sort_values('updates_k')
        if sub.empty:
            continue
        label = dataset_size_label(int(sub['dataset_size'].iloc[0]))
        axes[0].plot(sub['updates_k'], sub['hist_l1'], marker='o', lw=2.5, color=color, label=label)
        axes[1].plot(sub['updates_k'], sub['pk_log10_mae'], marker='o', lw=2.5, color=color, label=label)
    axes[0].set_title('One-point PDF error')
    axes[0].set_ylabel(r'$L_1$ distance (lower is better)')
    axes[1].set_title('Power-spectrum error')
    axes[1].set_ylabel(r'mean $|\log_{10}(P_{gen}/P_{real})|$ (lower is better)')
    for ax in axes:
        ax.set_xlabel('Optimizer updates (thousands)')
        ax.set_xticks([x for x, _ in CONTINUE_CHECKPOINTS])
        ax.grid(alpha=0.18)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
    axes[1].legend(title=r'$N_{2D}$', frameon=False, ncol=1)
    fig.suptitle('Does additional optimization repair DiT-L16 small-data fidelity?', y=1.03)
    out = QUICKCHECK_DIR / 'nf_generalize_fig2_dit_l16_continuation_fidelity_trajectories.png'
    fig.savefig(out, bbox_inches='tight')
    plt.show()
    print('wrote', out)


In [ ]:
def four_sample_montage(images: np.ndarray) -> np.ndarray:
    images = np.asarray(images)
    if images.ndim != 3 or len(images) < 4:
        raise ValueError(f'Expected at least four 2D samples, found shape {images.shape}.')
    return np.block([[images[0], images[1]], [images[2], images[3]]])


if continuation_image_cache:
    available_updates = [
        updates_k
        for updates_k, _ in CONTINUE_CHECKPOINTS
        if any((tag, updates_k) in continuation_image_cache for tag in CONTINUE_TAGS)
    ]
    available_tags = [
        tag
        for tag in CONTINUE_TAGS
        if any((tag, updates_k) in continuation_image_cache for updates_k in available_updates)
    ]
    values = np.concatenate([
        continuation_image_cache[(tag, updates_k)].ravel()
        for updates_k in available_updates
        for tag in available_tags
        if (tag, updates_k) in continuation_image_cache
    ])
    vmin, vmax = np.quantile(values, [0.005, 0.995])
    fig, axes = plt.subplots(
        len(available_updates),
        len(available_tags),
        figsize=(3.2 * len(available_tags), 3.0 * len(available_updates)),
        squeeze=False,
        constrained_layout=True,
    )
    for row_index, updates_k in enumerate(available_updates):
        for column_index, tag in enumerate(available_tags):
            ax = axes[row_index, column_index]
            images = continuation_image_cache.get((tag, updates_k))
            if images is not None:
                ax.imshow(four_sample_montage(images), cmap='viridis', vmin=vmin, vmax=vmax)
            ax.set_xticks([])
            ax.set_yticks([])
            if row_index == 0:
                ax.set_title(dataset_size_label(dataset_size_from_tag(tag)), fontsize=15)
            if column_index == 0:
                ax.set_ylabel(f'{updates_k}k', fontsize=14, fontweight='semibold')
    fig.suptitle(
        'DiT-L16 low-data checkpoint audit: four generated maps per checkpoint',
        y=1.02,
        fontsize=20,
    )
    out = QUICKCHECK_DIR / 'nf_generalize_fig2_dit_l16_continuation_image_grid.png'
    fig.savefig(out, bbox_inches='tight', dpi=240)
    plt.show()
    print('wrote', out)

if continuation_curve_cache:
    detail_updates = [
        updates_k
        for updates_k, _ in CONTINUE_CHECKPOINTS
        if (CONTINUE_DETAIL_TAG, updates_k) in continuation_curve_cache
    ]
    if detail_updates:
        cmap = plt.cm.plasma(np.linspace(0.12, 0.88, len(detail_updates)))
        first = continuation_curve_cache[(CONTINUE_DETAIL_TAG, detail_updates[0])]
        fig, axes = plt.subplots(1, 2, figsize=(13.2, 5.0), constrained_layout=True)
        axes[0].plot(
            first['centers'],
            first['real_density'],
            color='black',
            lw=2.7,
            label='complete training reference',
        )
        axes[1].axhline(1.0, color='black', ls='--', lw=1.5, label='ideal ratio')
        for color, updates_k in zip(cmap, detail_updates):
            curves = continuation_curve_cache[(CONTINUE_DETAIL_TAG, updates_k)]
            axes[0].plot(
                curves['centers'],
                curves['generated_density'],
                color=color,
                lw=2.1,
                label=f'{updates_k}k',
            )
            axes[1].plot(
                curves['kbins'],
                curves['pk_ratio'],
                color=color,
                marker='o',
                ms=4.0,
                lw=2.0,
                label=f'{updates_k}k',
            )
        axes[0].set_yscale('log')
        axes[0].set_xlabel('Normalized field value')
        axes[0].set_ylabel('Pixel PDF')
        axes[0].set_title('One-point distribution')
        axes[1].set_xlabel(r'$k$ bin')
        axes[1].set_ylabel(r'$P_{generated}(k)/P_{real}(k)$')
        axes[1].set_title('Power-spectrum fidelity')
        for ax in axes:
            ax.grid(False)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            ax.legend(frameon=False)
        n_label = dataset_size_label(dataset_size_from_tag(CONTINUE_DETAIL_TAG))
        fig.suptitle(
            f'Legacy state-reset continuation at $N_{{2D}}$ = {n_label}',
            y=1.03,
        )
        out = QUICKCHECK_DIR / f'nf_generalize_fig2_dit_l16_{CONTINUE_DETAIL_TAG}_continuation_pdf_pk.png'
        fig.savefig(out, bbox_inches='tight')
        plt.show()
        print('wrote', out)


In [ ]:
continuation_table_audit_rows = []
novelty_parts = []
for updates_k, _label in CONTINUE_CHECKPOINTS:
    for feature in ('pca', 'sscd'):
        table_path = continuation_table_path(feature, updates_k)
        continuation_table_audit_rows.append({
            'feature': feature.upper(),
            'updates_k': updates_k,
            'exists': table_path.exists(),
            'table_path': rel(table_path),
        })
        if not table_path.exists():
            continue
        table = add_generalization_columns(pd.read_csv(table_path))
        table = ensure_arch_columns(table)
        table = table[table['arch'].astype(str) == 'dit_l16'].copy()
        table['feature'] = feature.upper()
        table['updates_k'] = updates_k
        table['analysis_manifest'] = str(CONTINUE_ANALYSIS_MANIFEST_PATH)
        novelty_parts.append(table)

continuation_table_audit_df = pd.DataFrame(continuation_table_audit_rows)
display(continuation_table_audit_df)

continuation_novelty_df = pd.concat(novelty_parts, ignore_index=True) if novelty_parts else pd.DataFrame()
if continuation_novelty_df.empty:
    display(Markdown('PCA/SSCD continuation tables are not available yet. They are submitted after each exact-checkpoint sampler array.'))
else:
    display(continuation_novelty_df[[
        c for c in ['feature', 'updates_k', 'dataset_tag', 'dataset_size', 'gen_gl_q95', 'sample_path']
        if c in continuation_novelty_df.columns
    ]].sort_values(['feature', 'dataset_size', 'updates_k']))
    available_novelty_updates = sorted(continuation_novelty_df['updates_k'].dropna().astype(int).unique().tolist())
    if 'gen_gl_q95' in continuation_novelty_df.columns and len(available_novelty_updates) >= 2:
        fig, axes = plt.subplots(1, 2, figsize=(13.2, 5.0), sharey=True, constrained_layout=True)
        colors = plt.cm.viridis(np.linspace(0.08, 0.90, len(CONTINUE_TAGS)))
        for ax, feature in zip(axes, ('PCA', 'SSCD')):
            feature_df = continuation_novelty_df[continuation_novelty_df['feature'] == feature]
            for color, tag in zip(colors, CONTINUE_TAGS):
                sub = feature_df[feature_df['dataset_tag'] == tag].sort_values('updates_k')
                if sub.empty:
                    continue
                ax.plot(sub['updates_k'], sub['gen_gl_q95'], marker='o', lw=2.5, color=color,
                        label=dataset_size_label(int(sub['dataset_size'].iloc[0])))
            ax.axhline(0.5, color='0.35', ls=':', lw=1.4)
            ax.set_title(feature)
            ax.set_xlabel('Optimizer updates (thousands)')
            ax.set_xticks([x for x, _ in CONTINUE_CHECKPOINTS])
            ax.set_ylim(-0.04, 1.04)
            ax.grid(alpha=0.18)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
        axes[0].set_ylabel('q95 novelty score')
        axes[1].legend(title=r'$N_{2D}$', frameon=False)
        fig.suptitle('Does DiT-L16 novelty change with additional optimization?', y=1.03)
        out = QUICKCHECK_DIR / 'nf_generalize_fig2_dit_l16_continuation_pca_sscd.png'
        fig.savefig(out, bbox_inches='tight')
        plt.show()
        print('wrote', out)

        # Main transition view: one curve per checkpoint over training-set size.
        fig, axes = plt.subplots(1, 2, figsize=(13.2, 5.0), sharey=True, constrained_layout=True)
        update_colors = plt.cm.plasma(np.linspace(0.12, 0.88, len(available_novelty_updates)))
        dataset_sizes = [dataset_size_from_tag(tag) for tag in CONTINUE_TAGS]
        for ax, feature in zip(axes, ('PCA', 'SSCD')):
            feature_df = continuation_novelty_df[continuation_novelty_df['feature'] == feature]
            for color, updates_k in zip(update_colors, available_novelty_updates):
                sub = feature_df[feature_df['updates_k'] == updates_k].sort_values('dataset_size')
                if sub.empty:
                    continue
                ax.plot(
                    sub['dataset_size'], sub['gen_gl_q95'],
                    marker='o', ms=7.0, lw=2.5, color=color, label=f'{updates_k}k',
                )
            ax.axhline(0.5, color='0.35', ls=':', lw=1.4)
            ax.set_xscale('log', base=2)
            ax.set_xticks(dataset_sizes)
            ax.set_xticklabels([dataset_size_label(n) for n in dataset_sizes])
            ax.set_xlabel(r'Training set size $N_{2D}$')
            ax.set_title(feature)
            ax.set_ylim(-0.04, 1.04)
            ax.grid(alpha=0.18)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
        axes[0].set_ylabel('q95 novelty score')
        axes[1].legend(title='Optimizer updates', frameon=False, ncol=1)
        fig.suptitle('Low-data DiT-L16 checkpoint diagnostic (not a scaling curve)', y=1.03)
        out = QUICKCHECK_DIR / 'nf_generalize_fig2_dit_l16_continuation_by_data_size.png'
        fig.savefig(out, bbox_inches='tight')
        plt.show()
        print('wrote', out)

        comparison_rows = []
        for feature in ('PCA', 'SSCD'):
            feature_df = continuation_novelty_df[continuation_novelty_df['feature'] == feature]
            early = feature_df[feature_df['updates_k'] == 200][
                ['dataset_tag', 'dataset_size', 'gen_gl_q95']
            ].rename(columns={'gen_gl_q95': 'q95_200k'})
            late = feature_df[feature_df['updates_k'] == 300][
                ['dataset_tag', 'dataset_size', 'gen_gl_q95']
            ].rename(columns={'gen_gl_q95': 'q95_300k'})
            paired = early.merge(late, on=['dataset_tag', 'dataset_size'], how='outer')
            paired['feature'] = feature
            paired['delta_300k_minus_200k'] = paired['q95_300k'] - paired['q95_200k']
            comparison_rows.append(paired)
        l16_200k_vs_300k = pd.concat(comparison_rows, ignore_index=True).sort_values(
            ['feature', 'dataset_size']
        )
        display(Markdown('### Legacy low-data checkpoint comparison'))
        display(l16_200k_vs_300k)

        if not continuation_fidelity_df.empty:
            colors = plt.cm.viridis(np.linspace(0.08, 0.90, len(CONTINUE_TAGS)))
            fig, axes = plt.subplots(2, 2, figsize=(14.5, 9.2), constrained_layout=True)
            panel_specs = [
                ('PCA', 'gen_gl_q95', 'PCA q95 novelty', 'higher means farther from training images'),
                ('SSCD', 'gen_gl_q95', 'SSCD q95 novelty', 'higher means farther from training images'),
                (None, 'hist_l1', 'One-point PDF error', 'lower is better'),
                (None, 'pk_log10_mae', r'Power-spectrum error', 'lower is better'),
            ]
            for ax, (feature, metric, title, subtitle) in zip(axes.ravel(), panel_specs):
                for color, tag in zip(colors, CONTINUE_TAGS):
                    if feature is None:
                        source_df = continuation_fidelity_df
                    else:
                        source_df = continuation_novelty_df[
                            continuation_novelty_df['feature'] == feature
                        ]
                    sub = source_df[source_df['dataset_tag'] == tag].sort_values('updates_k')
                    if sub.empty or metric not in sub.columns:
                        continue
                    ax.plot(
                        sub['updates_k'],
                        sub[metric],
                        color=color,
                        marker='o',
                        ms=6.5,
                        lw=2.5,
                        label=dataset_size_label(int(sub['dataset_size'].iloc[0])),
                    )
                if metric == 'gen_gl_q95':
                    ax.axhline(0.5, color='0.35', ls=':', lw=1.3)
                    ax.set_ylim(-0.04, 1.04)
                ax.set_title(title, fontsize=18, pad=16)
                ax.text(
                    0.5,
                    1.01,
                    subtitle,
                    transform=ax.transAxes,
                    ha='center',
                    va='bottom',
                    fontsize=11.5,
                    color='0.35',
                )
                ax.set_xlabel('Optimizer updates (thousands)')
                ax.set_xticks([updates for updates, _ in CONTINUE_CHECKPOINTS])
                ax.grid(axis='y', alpha=0.16)
                ax.spines['top'].set_visible(False)
                ax.spines['right'].set_visible(False)
            axes[0, 0].set_ylabel('q95 novelty score')
            axes[1, 0].set_ylabel(r'$L_1$ distance')
            axes[1, 1].set_ylabel(r'mean $|\log_{10}(P_{gen}/P_{real})|$')
            handles, labels = axes[0, 0].get_legend_handles_labels()
            fig.legend(
                handles,
                labels,
                title=r'Training images $N_{2D}$',
                loc='upper center',
                bbox_to_anchor=(0.5, 0.93),
                ncol=len(labels),
                frameon=False,
            )
            fig.suptitle(
                'Low-data DiT-L16 checkpoint diagnostic (not a scaling curve)',
                fontsize=24,
                y=1.01,
            )
            fig.text(
                0.5,
                0.955,
                'Legacy files reset optimizer and scheduler state at each 25k stage; '
                'use this figure to diagnose failures, not longer-training behavior.',
                ha='center',
                fontsize=12.5,
                color='#8B1A1A',
            )
            out = QUICKCHECK_DIR / 'nf_generalize_fig2_dit_l16_legacy_checkpoint_audit.png'
            fig.savefig(out, bbox_inches='tight', dpi=300)
            plt.show()
            print('wrote', out)

        display(Markdown(
            '**Interpretation:** this low-data experiment cannot show a complete scaling relation. '
            'The nonmonotonic changes are also contaminated by optimizer and scheduler resets in the '
            'legacy resume loader. The complete fixed-budget depth comparison remains the 200k sweep '
            'shown above. Rerun the continuation with restored state before interpreting any shift.'
        ))
    elif 'gen_gl_q95' in continuation_novelty_df.columns:
        display(Markdown(
            '**Generalization trajectory not plotted:** found analyzed updates '
            f'`{available_novelty_updates}`. At least two analyzed checkpoints are required; '
            'check the table audit above for missing PCA/SSCD files.'
        ))


### Reading the legacy continuation experiment

Rows and curves compare 200k, 225k, 250k, 275k, and 300k checkpoints for the low-data L16 runs. This experiment is useful for locating when failures appear, but the legacy loader reset optimizer and scheduler state between stages. It is therefore a checkpoint diagnostic, not a controlled statement about the effect of additional training.

Warnings are kept in this markdown rather than placed over the figure.

## Fresh DiT-L16 300k replacement sweep

This section uses the clean replacement experiment, not the failed staged
continuation. Ten DiT-L16 models start from new seed-123 initializations and
train directly to 300k optimizer updates, one for every training-set size from
$2^6$ through $2^{15}$.

DiT-L8 and DiT-L12 use their original 200k runs. DiT-L16 uses the clean 300k
replacement sweep. This is therefore a depth comparison at unequal training
budgets, labeled explicitly in the figure. No failed continuation or old L16
table is substituted. The q95 novelty score tests proximity to training
examples; **q95 novelty does not guarantee physical fidelity**.


In [ ]:
FRESH_SWEEP_NAME = 'nf_generalize_fig2_dit_l16_fresh300k_v2'
FRESH_EXPECTED_POWERS = list(range(6, 16))
FRESH_EXPECTED_TAGS = [f'd2p{power:02d}' for power in FRESH_EXPECTED_POWERS]
FRESH_EXPECTED_SIZES = [2 ** power for power in FRESH_EXPECTED_POWERS]


def fresh_300k_v2_metric_path(feature: str) -> Path:
    return TABLE_DIR / (
        f'{FRESH_SWEEP_NAME}_{feature.lower()}_full_nn_metrics.csv'
    )


def audit_fresh_300k_v2_table(feature: str) -> tuple[pd.DataFrame, dict[str, Any]]:
    path = fresh_300k_v2_metric_path(feature)
    audit_row: dict[str, Any] = {
        'feature': feature,
        'path': rel(path),
        'exists': path.exists(),
        'rows': 0,
        'missing_tags': list(FRESH_EXPECTED_TAGS),
        'duplicate_tags': [],
        'complete': False,
    }
    if not path.exists():
        return pd.DataFrame(), audit_row

    table = ensure_arch_columns(add_generalization_columns(pd.read_csv(path)))
    table = table[table['arch'].astype(str) == 'dit_l16'].copy()
    table = table.sort_values('dataset_size')
    tags = table['dataset_tag'].dropna().astype(str)
    missing = sorted(set(FRESH_EXPECTED_TAGS) - set(tags))
    extra = sorted(set(tags) - set(FRESH_EXPECTED_TAGS))
    duplicates = sorted(tags[tags.duplicated(keep=False)].unique().tolist())
    complete = (
        len(table) == 10
        and not missing
        and not extra
        and not duplicates
        and 'gen_gl_q95' in table.columns
        and table['gen_gl_q95'].notna().all()
    )
    audit_row.update({
        'rows': len(table),
        'missing_tags': missing,
        'extra_tags': extra,
        'duplicate_tags': duplicates,
        'complete': bool(complete),
    })
    return table, audit_row


fresh_300k_v2_metrics: dict[str, pd.DataFrame] = {}
fresh_300k_v2_audit_rows: list[dict[str, Any]] = []
for feature in ('PCA', 'SSCD'):
    table, audit_row = audit_fresh_300k_v2_table(feature)
    fresh_300k_v2_metrics[feature] = table
    fresh_300k_v2_audit_rows.append(audit_row)

fresh_300k_v2_audit_df = pd.DataFrame(fresh_300k_v2_audit_rows)
display(Markdown('### Clean replacement table audit'))
display(fresh_300k_v2_audit_df)

fresh_300k_v2_complete = (
    len(fresh_300k_v2_audit_df) == 2
    and bool(fresh_300k_v2_audit_df['complete'].all())
)
if fresh_300k_v2_complete:
    display(Markdown(
        '**Fresh 300k v2 audit passed.** All ten data sizes are present exactly '
        'once in both PCA and SSCD.'
    ))
else:
    display(Markdown(
        '**Fresh 300k v2 audit incomplete: not drawing the replacement L16 '
        'curve.** Both tables must contain all ten data sizes exactly once.'
    ))


In [ ]:
FRESH_DEPTH_COLORS = {
    'dit_l8': '#009E73',
    'dit_base': '#0072B2',
    'dit_l16': '#B33C86',
}
FRESH_DEPTH_MARKERS = {'dit_l8': 'P', 'dit_base': 'D', 'dit_l16': 'X'}
FRESH_DEPTH_LABELS = {
    'dit_l8': 'DiT-L8 200k',
    'dit_base': 'DiT-L12 / base 200k',
    'dit_l16': 'DiT-L16 300k',
}


def require_fresh_complete_curve(
    table: pd.DataFrame,
    arch: str,
    *,
    context: str,
) -> pd.DataFrame:
    sub = table[table['arch'].astype(str) == arch].copy()
    sub = sub.sort_values('dataset_size')
    tags = sub['dataset_tag'].dropna().astype(str)
    missing = sorted(set(FRESH_EXPECTED_TAGS) - set(tags))
    extra = sorted(set(tags) - set(FRESH_EXPECTED_TAGS))
    if (
        len(sub) != 10
        or missing
        or extra
        or tags.duplicated().any()
        or 'gen_gl_q95' not in sub.columns
        or sub['gen_gl_q95'].isna().any()
    ):
        raise ValueError(
            f'{context} is incomplete: rows={len(sub)}, '
            f'missing={missing}, extra={extra}'
        )
    return sub


def fresh_depth_sources(feature: str) -> dict[str, pd.DataFrame]:
    baseline = pca_metrics if feature == 'PCA' else sscd_metrics
    return {
        'dit_l8': require_fresh_complete_curve(
            baseline, 'dit_l8', context=f'{feature} DiT-L8 200k'
        ),
        'dit_base': require_fresh_complete_curve(
            baseline, 'dit_base', context=f'{feature} DiT-L12 200k'
        ),
        'dit_l16': require_fresh_complete_curve(
            fresh_300k_v2_metrics[feature],
            'dit_l16',
            context=f'{feature} fresh DiT-L16 300k v2',
        ),
    }


def plot_fresh_300k_v2_depth_comparison(
    *,
    output_name: str,
    zoom_max_power: int | None = None,
) -> Path:
    fig, axes = plt.subplots(
        1, 2, figsize=(16.2, 6.2), sharey=True, constrained_layout=True
    )
    for ax, feature in zip(axes, ('PCA', 'SSCD')):
        for arch, sub in fresh_depth_sources(feature).items():
            ax.plot(
                sub['dataset_size'],
                sub['gen_gl_q95'],
                color=FRESH_DEPTH_COLORS[arch],
                marker=FRESH_DEPTH_MARKERS[arch],
                ms=9,
                lw=3,
                label=FRESH_DEPTH_LABELS[arch],
            )
        ax.axhline(0.5, color='0.35', ls=':', lw=1.6)
        shown_powers = [
            power for power in FRESH_EXPECTED_POWERS
            if zoom_max_power is None or power <= zoom_max_power
        ]
        ax.set_xscale('log', base=2)
        ax.set_xticks([2 ** power for power in shown_powers])
        ax.set_xticklabels([rf'$2^{{{power}}}$' for power in shown_powers])
        if zoom_max_power is not None:
            ax.set_xlim(2 ** 5.75, 2 ** (zoom_max_power + 0.25))
        ax.set_ylim(-0.04, 1.04)
        ax.set_xlabel(r'Training images $N_{2D}$')
        ax.set_title(f'{feature} q95 novelty', fontsize=19, pad=12)
        ax.grid(axis='y', alpha=0.18)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
    axes[0].set_ylabel('q95 novelty score')
    handles, labels = axes[1].get_legend_handles_labels()
    fig.legend(
        handles, labels, loc='upper center', bbox_to_anchor=(0.5, 0.91),
        ncol=3, frameon=False
    )
    title = 'Clean DiT depth comparison'
    if zoom_max_power is not None:
        title += ': transition region'
    fig.suptitle(title, fontsize=23, fontweight='semibold', y=1.01)
    fig.text(
        0.5, 0.945,
        'L8 and L12 use 200k updates; fresh L16 uses 300k updates.',
        ha='center', color='0.35', fontsize=12.5
    )
    out = QUICKCHECK_DIR / output_name
    fig.savefig(out, bbox_inches='tight', dpi=300)
    plt.show()
    print('wrote', out)
    return out


fresh_300k_v2_outputs = []
if fresh_300k_v2_complete:
    fresh_300k_v2_outputs.append(plot_fresh_300k_v2_depth_comparison(
        output_name='nf_generalize_fig2_dit_l16_fresh300k_v2_depth_comparison_full.png'
    ))
    fresh_300k_v2_outputs.append(plot_fresh_300k_v2_depth_comparison(
        output_name='nf_generalize_fig2_dit_l16_fresh300k_v2_depth_comparison_zoom.png',
        zoom_max_power=11,
    ))


## Fresh 300k DiT-L16 Samples and Physical Statistics

The novelty curves above use the fresh 300k PCA and SSCD tables. This section
loads the matching **fresh 300k sample files themselves** for every training-set
size from $2^6$ through $2^{15}$. It never falls back to the legacy 200k
`dpm50` files.

The audit reports the resolved checkpoint, scheduler, step count, and seed
stored inside each NPZ before drawing a figure. Every black one-point and
$P(k)$ reference is streamed from the exact training subset selected by that
fresh run's frozen configuration.

The current files use DPM-Solver with 50 sampling steps. A bad image from one
checkpoint does not by itself establish a sampler failure. The final audit
searches for alternate samples from the same checkpoint. A controlled sampler
test requires the same checkpoint and initial noise with DPM50, higher-step
DPM, and DDPM500.

In [ ]:
FRESH_MANIFEST_PATH = PROJECT_DIR / 'local' / FRESH_SWEEP_NAME / 'manifest.json'
FRESH_SAMPLE_ROOT = PROJECT_DIR / 'results' / FRESH_SWEEP_NAME / 'samples'
FRESH_EXPECTED_SAMPLE_LABEL = 'dpm50_fresh300k_v2'


def _npz_scalar(payload, key: str, default=None):
    if key not in payload.files:
        return default
    value = np.asarray(payload[key])
    if value.size != 1:
        return value.tolist()
    return value.reshape(()).item()


def _resolve_project_path(value: str | Path) -> Path:
    path = Path(str(value))
    return path if path.is_absolute() else PROJECT_DIR / path


def _fresh_sample_path(row: dict[str, Any]) -> Path:
    raw = str(row['sample_path']).format(
        seed=SEED,
        sample_label=str(row['sample_label']),
    )
    return _resolve_project_path(raw)


if not FRESH_MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f'Missing fresh 300k manifest: {FRESH_MANIFEST_PATH}. '
        'The fresh diagnostics will not substitute legacy samples.'
    )

fresh_manifest_rows = sorted(
    json.loads(FRESH_MANIFEST_PATH.read_text()),
    key=lambda row: int(row['dataset_size']),
)
if [str(row['dataset_tag']) for row in fresh_manifest_rows] != FRESH_EXPECTED_TAGS:
    raise RuntimeError('Fresh manifest does not contain exactly d2p06 through d2p15 in order.')

fresh_300k_bundles: dict[str, dict[str, Any]] = {}
fresh_sample_audit_rows = []
for row in fresh_manifest_rows:
    tag = str(row['dataset_tag'])
    sample_label = str(row['sample_label'])
    if sample_label != FRESH_EXPECTED_SAMPLE_LABEL:
        raise RuntimeError(
            f'Fresh {tag} manifest has sample label {sample_label!r}; '
            f'expected {FRESH_EXPECTED_SAMPLE_LABEL!r}.'
        )
    if int(row['target_total_updates']) != 300_000:
        raise RuntimeError(
            f'Fresh {tag} targets {row["target_total_updates"]} updates, not 300000.'
        )
    sample_path = _fresh_sample_path(row)
    config_path = _resolve_project_path(row['config'])
    if not sample_path.exists():
        raise FileNotFoundError(
            f'Missing fresh 300k sample for {tag}: {sample_path}. '
            'Legacy 200k samples are intentionally not used as a fallback.'
        )
    if not config_path.exists():
        raise FileNotFoundError(f'Missing frozen fresh config for {tag}: {config_path}')

    generated = as_nchw(np.asarray(load_npz_array(sample_path), dtype=np.float32))
    with np.load(sample_path, allow_pickle=False) as payload:
        scheduler = str(_npz_scalar(payload, 'scheduler', 'missing'))
        num_steps = int(_npz_scalar(payload, 'num_steps', -1))
        seed = int(_npz_scalar(payload, 'seed', -1))
        requested_checkpoint = str(_npz_scalar(payload, 'requested_checkpoint', 'missing'))
        resolved_checkpoint = str(_npz_scalar(payload, 'resolved_checkpoint', 'missing'))
        stored_config = str(_npz_scalar(payload, 'config_path', 'missing'))

    expected_checkpoint = str(Path(row['expected_checkpoint']).resolve())
    resolved_matches = str(Path(resolved_checkpoint).resolve()) == expected_checkpoint
    requested_matches = str(Path(requested_checkpoint).resolve()) == expected_checkpoint
    stored_config_matches = str(Path(stored_config).resolve()) == str(config_path.resolve())
    reference_info = configured_training_reference_info(config_path)
    audit = {
        'dataset_tag': tag,
        'dataset_size': int(row['dataset_size']),
        'target_updates': int(row['target_total_updates']),
        'sample_label': sample_label,
        'n_generated': len(generated),
        'exact_training_slices': int(reference_info['configured_slices']),
        'scheduler': scheduler,
        'num_steps': num_steps,
        'seed': seed,
        'requested_checkpoint_matches_manifest': requested_matches,
        'resolved_checkpoint_matches_manifest': resolved_matches,
        'stored_config_matches_manifest': stored_config_matches,
        'sample_path': rel(sample_path),
        'config_path': rel(config_path),
        'stored_config_path': stored_config,
        'requested_checkpoint': requested_checkpoint,
        'resolved_checkpoint': resolved_checkpoint,
    }
    fresh_sample_audit_rows.append(audit)
    fresh_300k_bundles[tag] = {
        'spec': row,
        'generated': generated,
        'sample_path': sample_path,
        'config_path': config_path,
        'reference_info': reference_info,
        'sample_metadata': audit,
    }

fresh_300k_sample_audit_df = pd.DataFrame(fresh_sample_audit_rows)
display(Markdown('### Fresh-sample provenance audit'))
display(fresh_300k_sample_audit_df)
fresh_300k_sample_audit_pass = bool(
    len(fresh_300k_sample_audit_df) == 10
    and (fresh_300k_sample_audit_df['target_updates'] == 300_000).all()
    and (fresh_300k_sample_audit_df['sample_label'] == FRESH_EXPECTED_SAMPLE_LABEL).all()
    and (fresh_300k_sample_audit_df['n_generated'] == 512).all()
    and fresh_300k_sample_audit_df['requested_checkpoint_matches_manifest'].all()
    and fresh_300k_sample_audit_df['resolved_checkpoint_matches_manifest'].all()
    and fresh_300k_sample_audit_df['stored_config_matches_manifest'].all()
    and fresh_300k_sample_audit_df['scheduler'].str.contains(
        'DPMSolverMultistepScheduler', regex=False
    ).all()
    and (fresh_300k_sample_audit_df['num_steps'] == 50).all()
    and (fresh_300k_sample_audit_df['seed'] == SEED).all()
)
if not fresh_300k_sample_audit_pass:
    raise RuntimeError(
        'Fresh 300k sample provenance audit failed. Figures are withheld rather '
        'than mixing checkpoints or sampler settings.'
    )


def plot_fresh_300k_generated_full_sweep(samples_per_size: int = 4) -> Path:
    bundles = [fresh_300k_bundles[tag] for tag in FRESH_EXPECTED_TAGS]
    samples_per_size = max(1, int(samples_per_size))
    if any(len(bundle['generated']) < samples_per_size for bundle in bundles):
        raise RuntimeError(
            f'Requested {samples_per_size} generated maps per data size, but at '
            'least one fresh sample file contains fewer maps.'
        )
    values = np.concatenate([
        bundle['generated'][:samples_per_size, 0].ravel()
        for bundle in bundles
    ])
    vmin, vmax = np.nanquantile(values, [0.005, 0.995])
    fig, axes = plt.subplots(
        2 * samples_per_size,
        5,
        figsize=(16.2, 3.0 * 2 * samples_per_size),
        constrained_layout=True,
    )
    for block, block_bundles in enumerate((bundles[:5], bundles[5:])):
        for col, bundle in enumerate(block_bundles):
            for sample_index in range(samples_per_size):
                axis = axes[block * samples_per_size + sample_index, col]
                axis.imshow(
                    bundle['generated'][sample_index, 0],
                    cmap='viridis',
                    vmin=vmin,
                    vmax=vmax,
                )
                axis.set_xticks([])
                axis.set_yticks([])
                if sample_index == 0:
                    axis.set_title(
                        dataset_size_label(int(bundle['spec']['dataset_size'])),
                        pad=7,
                    )
        for sample_index in range(samples_per_size):
            axes[block * samples_per_size + sample_index, 0].set_ylabel(
                f'sample {sample_index + 1}',
                fontsize=12.5,
                fontweight='semibold',
            )
    fig.suptitle(
        'Fresh DiT-L16 generated maps across the full 300k sweep',
        fontsize=22,
        fontweight='semibold',
    )
    fig.text(
        0.5, 0.94, 'All panels: DPM-Solver, 50 steps, seed 123',
        ha='center', fontsize=13, color='0.3',
    )
    out = QUICKCHECK_DIR / 'nf_generalize_fig2_dit_l16_fresh300k_v2_generated_full_sweep.png'
    fig.savefig(out, bbox_inches='tight', dpi=300)
    plt.show()
    print('wrote', out)
    return out


fresh_300k_generated_grid = plot_fresh_300k_generated_full_sweep(
    samples_per_size=int(os.environ.get('DIT_FRESH_IMAGES_PER_SIZE', '4'))
)


fresh_300k_real_stats: dict[str, dict[str, Any]] = {}
fresh_300k_physical_curves: dict[str, dict[str, Any]] = {}
for tag in FRESH_EXPECTED_TAGS:
    bundle = fresh_300k_bundles[tag]
    real_stats = _aggregate_physical_batches(
        iter_real_reference_batches_from_config(
            bundle['config_path'], raw_batch_size=REAL_REFERENCE_RAW_BATCH_SIZE
        ),
        nbins=PK_NBINS,
    )
    expected_count = int(bundle['reference_info']['configured_slices'])
    if int(real_stats['n_images']) != expected_count:
        raise RuntimeError(
            f'Fresh {tag} real-reference mismatch: {real_stats["n_images"]} '
            f'aggregated versus {expected_count} configured.'
        )
    generated_stats = _aggregate_physical_batches(
        [bundle['generated']], nbins=PK_NBINS
    )
    ratio = generated_stats['mean_pk'] / np.clip(real_stats['mean_pk'], 1e-30, None)
    fresh_300k_real_stats[tag] = real_stats
    fresh_300k_physical_curves[tag] = {
        'dataset_size': int(bundle['spec']['dataset_size']),
        'hist_edges': real_stats['hist_edges'],
        'real_hist': real_stats['hist'],
        'generated_hist': generated_stats['hist'],
        'kbins': real_stats['kbins'],
        'pk_ratio': ratio,
        'pk_log2_ratio': np.log2(np.clip(ratio, 1e-12, None)),
        'n_real': int(real_stats['n_images']),
        'n_generated': int(generated_stats['n_images']),
    }


def plot_fresh_300k_onepoint_full_sweep() -> Path:
    curves = [fresh_300k_physical_curves[tag] for tag in FRESH_EXPECTED_TAGS]
    centers = 0.5 * (curves[0]['hist_edges'][:-1] + curves[0]['hist_edges'][1:])
    fig, axes = plt.subplots(2, 5, figsize=(17.4, 8.0))
    for axis, curve in zip(axes.ravel(), curves):
        axis.plot(centers, curve['real_hist'], color='black', lw=2.3)
        axis.plot(centers, curve['generated_hist'], color='#B33C86', lw=2.2)
        axis.set_yscale('log')
        axis.set_title(dataset_size_label(curve['dataset_size']), pad=8)
        axis.set_xlabel('Normalized field value')
        axis.grid(axis='y', alpha=0.14)
        axis.spines['top'].set_visible(False)
        axis.spines['right'].set_visible(False)
    for axis in axes[:, 0]:
        axis.set_ylabel('Pixel PDF')
    fig.suptitle(
        'Fresh DiT-L16 one-point distributions at 300k updates',
        fontsize=22, fontweight='semibold', y=0.995,
    )
    fig.legend(
        handles=[
            Line2D([0], [0], color='black', lw=2.3, label='exact model training subset'),
            Line2D([0], [0], color='#B33C86', lw=2.2, label='generated'),
        ],
        loc='upper center', bbox_to_anchor=(0.5, 0.95), ncol=2, frameon=False,
    )
    fig.subplots_adjust(left=0.06, right=0.99, bottom=0.08, top=0.87, hspace=0.38, wspace=0.28)
    out = QUICKCHECK_DIR / 'nf_generalize_fig2_dit_l16_fresh300k_v2_onepoint_full_sweep.png'
    fig.savefig(out, bbox_inches='tight', dpi=300)
    plt.show()
    print('wrote', out)
    return out


def plot_fresh_300k_pk_full_sweep() -> dict[str, Path]:
    curves = [fresh_300k_physical_curves[tag] for tag in FRESH_EXPECTED_TAGS]
    finite = np.concatenate([
        curve['pk_ratio'][np.isfinite(curve['pk_ratio'])] for curve in curves
    ])
    shared_upper = max(1.25, float(np.nanmax(finite)) * 1.04)
    fig, axes = plt.subplots(2, 5, figsize=(17.4, 7.6))
    for axis, curve in zip(axes.ravel(), curves):
        axis.plot(curve['kbins'], curve['pk_ratio'], color='#B33C86', marker='o', ms=3.8, lw=2)
        axis.axhline(1.0, color='black', ls='--', lw=1.4)
        axis.set_ylim(0, shared_upper)
        axis.set_title(dataset_size_label(curve['dataset_size']), pad=8)
        axis.set_xlabel(r'$k$ bin')
        axis.grid(axis='y', alpha=0.14)
        axis.spines['top'].set_visible(False)
        axis.spines['right'].set_visible(False)
    for axis in axes[:, 0]:
        axis.set_ylabel(r'$P_{\rm generated}(k)/P_{\rm real}(k)$')
    fig.suptitle(
        'Fresh DiT-L16 power-spectrum ratios at 300k updates',
        fontsize=22, fontweight='semibold', y=0.995,
    )
    fig.text(
        0.5, 0.95, 'One is exact agreement; all ten panels use the same vertical scale.',
        ha='center', fontsize=12.5, color='0.3',
    )
    fig.subplots_adjust(left=0.06, right=0.99, bottom=0.08, top=0.87, hspace=0.36, wspace=0.28)
    ratio_out = QUICKCHECK_DIR / 'nf_generalize_fig2_dit_l16_fresh300k_v2_pk_ratio_full_sweep.png'
    fig.savefig(ratio_out, bbox_inches='tight', dpi=300)
    plt.show()
    print('wrote', ratio_out)

    matrix = np.vstack([curve['pk_log2_ratio'] for curve in curves])
    finite_log = matrix[np.isfinite(matrix)]
    limit = max(1.0, float(np.ceil(np.nanmax(np.abs(finite_log)) * 2) / 2))
    fig_heat, axis = plt.subplots(figsize=(13.8, 5.8))
    image = axis.imshow(
        matrix,
        aspect='auto',
        origin='upper',
        cmap='RdBu_r',
        vmin=-limit,
        vmax=limit,
        extent=[curves[0]['kbins'][0], curves[0]['kbins'][-1], 9.5, -0.5],
    )
    axis.set_yticks(range(10), [dataset_size_label(2 ** power) for power in range(6, 16)])
    axis.set_xlabel(r'$k$ bin')
    axis.set_ylabel(r'Training images $N_{2D}$')
    axis.set_title('Scale-resolved power-spectrum error', fontsize=21, fontweight='semibold', pad=12)
    colorbar = fig_heat.colorbar(image, ax=axis, pad=0.02)
    colorbar.set_label(r'$\log_2[P_{\rm generated}(k)/P_{\rm real}(k)]$')
    axis.text(
        0.5, -0.16, 'Zero is exact; +1 means twice the power and -1 means half the power.',
        transform=axis.transAxes, ha='center', fontsize=12.5, color='0.3',
    )
    heat_out = QUICKCHECK_DIR / 'nf_generalize_fig2_dit_l16_fresh300k_v2_pk_log2_error.png'
    fig_heat.savefig(heat_out, bbox_inches='tight', dpi=300)
    plt.show()
    print('wrote', heat_out)
    return {'ratio': ratio_out, 'log2_error': heat_out}


fresh_300k_onepoint_path = plot_fresh_300k_onepoint_full_sweep()
fresh_300k_pk_paths = plot_fresh_300k_pk_full_sweep()


def streaming_nearest_training_match(bundle: dict[str, Any], generated_index: int = 0) -> dict[str, Any]:
    generated_index = min(int(generated_index), len(bundle['generated']) - 1)
    generated = np.asarray(bundle['generated'][generated_index], dtype=np.float32)
    generated_flat = generated.reshape(-1)
    generated_norm = float(np.linalg.norm(generated_flat))
    best_mse = np.inf
    best_image = None
    best_index = -1
    offset = 0
    for batch in iter_real_reference_batches_from_config(
        bundle['config_path'], raw_batch_size=REAL_REFERENCE_RAW_BATCH_SIZE
    ):
        training = as_nchw(np.asarray(batch, dtype=np.float32))
        training_flat = training.reshape(len(training), -1)
        mse = np.mean((training_flat - generated_flat[None]) ** 2, axis=1)
        local = int(np.argmin(mse))
        if float(mse[local]) < best_mse:
            best_mse = float(mse[local])
            best_index = offset + local
            best_image = training[local].copy()
        offset += len(training)
    expected = int(bundle['reference_info']['configured_slices'])
    if offset != expected or best_image is None:
        raise RuntimeError(
            f'Nearest search scanned {offset} training slices; expected {expected}.'
        )
    best_flat = best_image.reshape(-1)
    denominator = generated_norm * float(np.linalg.norm(best_flat))
    cosine = float(np.dot(generated_flat, best_flat) / denominator) if denominator else 0.0
    return {
        'generated': generated,
        'nearest': best_image,
        'difference': np.abs(generated - best_image),
        'nearest_index': best_index,
        'mse': best_mse,
        'cosine': cosine,
        'training_slices_scanned': offset,
    }


def plot_fresh_300k_nearest_full_sweep(generated_index: int = 0) -> Path:
    matches = {
        tag: streaming_nearest_training_match(fresh_300k_bundles[tag], generated_index)
        for tag in FRESH_EXPECTED_TAGS
    }
    fig, axes = plt.subplots(6, 5, figsize=(16.4, 19.0), constrained_layout=True)
    for block, tags in enumerate((FRESH_EXPECTED_TAGS[:5], FRESH_EXPECTED_TAGS[5:])):
        for col, tag in enumerate(tags):
            match = matches[tag]
            value_stack = np.concatenate([
                match['generated'].ravel(), match['nearest'].ravel()
            ])
            vmin, vmax = np.nanquantile(value_stack, [0.005, 0.995])
            difference_max = max(float(np.nanquantile(match['difference'], 0.995)), 1e-8)
            generated_axis = axes[3 * block, col]
            nearest_axis = axes[3 * block + 1, col]
            difference_axis = axes[3 * block + 2, col]
            generated_axis.imshow(match['generated'][0], cmap='viridis', vmin=vmin, vmax=vmax)
            nearest_axis.imshow(match['nearest'][0], cmap='viridis', vmin=vmin, vmax=vmax)
            difference_axis.imshow(match['difference'][0], cmap='magma', vmin=0, vmax=difference_max)
            generated_axis.set_title(dataset_size_label(dataset_size_from_tag(tag)), pad=7)
            difference_axis.text(
                0.03, 0.03,
                f'MSE={match["mse"]:.3g}; cos={match["cosine"]:.3f}',
                transform=difference_axis.transAxes,
                fontsize=9.5,
                color='white',
                bbox={'facecolor': 'black', 'alpha': 0.62, 'pad': 2},
            )
            for axis in (generated_axis, nearest_axis, difference_axis):
                axis.set_xticks([])
                axis.set_yticks([])
        axes[3 * block, 0].set_ylabel('generated', fontsize=14, fontweight='bold')
        axes[3 * block + 1, 0].set_ylabel('nearest training', fontsize=14, fontweight='bold')
        axes[3 * block + 2, 0].set_ylabel('absolute difference', fontsize=14, fontweight='bold')
    fig.suptitle(
        'Fresh DiT-L16 samples versus exact nearest training slices',
        fontsize=22, fontweight='semibold',
    )
    out = QUICKCHECK_DIR / 'nf_generalize_fig2_dit_l16_fresh300k_v2_nearest_full_sweep.png'
    fig.savefig(out, bbox_inches='tight', dpi=300)
    plt.show()
    print('wrote', out)
    return out


fresh_300k_nearest_path = plot_fresh_300k_nearest_full_sweep(
    generated_index=int(os.environ.get('DIT_FRESH_NN_SAMPLE_INDEX', '0'))
)


sampler_audit_rows = []
for row in fresh_manifest_rows:
    run_name = str(row['run_name'])
    expected_checkpoint = str(Path(row['expected_checkpoint']).resolve())
    for sample_path in sorted(FRESH_SAMPLE_ROOT.glob(f'{run_name}_seed{SEED}_*.npz')):
        with np.load(sample_path, allow_pickle=False) as payload:
            scheduler = str(_npz_scalar(payload, 'scheduler', 'missing'))
            num_steps = int(_npz_scalar(payload, 'num_steps', -1))
            resolved_checkpoint = str(_npz_scalar(payload, 'resolved_checkpoint', 'missing'))
        sampler_audit_rows.append({
            'dataset_tag': str(row['dataset_tag']),
            'dataset_size': int(row['dataset_size']),
            'scheduler': scheduler,
            'num_steps': num_steps,
            'same_expected_checkpoint': str(Path(resolved_checkpoint).resolve()) == expected_checkpoint,
            'sample_path': rel(sample_path),
        })

fresh_sampler_audit_df = pd.DataFrame(sampler_audit_rows).sort_values(
    ['dataset_size', 'scheduler', 'num_steps']
)
display(Markdown('### Sampler adequacy audit'))
display(fresh_sampler_audit_df)
controlled_sampler_counts = (
    fresh_sampler_audit_df[fresh_sampler_audit_df['same_expected_checkpoint']]
    .groupby('dataset_tag')[['scheduler', 'num_steps']]
    .apply(lambda group: len(group.drop_duplicates()))
)
if controlled_sampler_counts.max() < 2:
    display(Markdown(
        '**No controlled sampler comparison is available yet.** These files '
        'establish the behavior of DPM-Solver 50 only. To test whether 50 steps '
        'cause the artifacts, generate DPM100 or DPM200 and DDPM500 samples from '
        'the same resolved checkpoint and seed, then rerun this audit.'
    ))
else:
    display(Markdown(
        'At least one data size has multiple sampler settings from the same '
        'checkpoint. Compare images and physical errors only within those matched rows.'
    ))

### Interpretation

The full-range figure answers whether the clean L16 transition is monotonic
across all ten data sizes. The zoomed figure shows the transition region without
hiding the high-data runs. A rightward L16 transition would be consistent with a
larger data requirement, but the experiment measures that outcome; it does not
assume or enforce it. Read PCA and SSCD together and verify generated maps,
one-point distributions, and power spectra before treating high novelty as a
successful scientific model.


## Takeaways


In [ ]:
lines = ['### Notebook summary']
if fresh_300k_v2_complete:
    lines.append(
        '- **Fresh 300k v2 status: complete.** PCA and SSCD both contain '
        'all ten data sizes from $2^6$ through $2^{15}$.'
    )
else:
    lines.append(
        '- **Fresh 300k v2 status: incomplete.** The replacement L16 depth '
        'curve is withheld until PCA and SSCD both contain all ten data sizes.'
    )
lines.extend([
    '- The plotted L16 line comes only from the clean replacement sweep.',
    '- DiT-L8 and DiT-L12 use 200k updates; DiT-L16 uses 300k, so this is '
    'not an equal-compute comparison.',
    '- PCA and SSCD measure novelty relative to training examples. They do '
    'not establish physical validity by themselves.',
])
display(Markdown('\n'.join(lines)))


## Appendix: Conditional Calibration Input Audit

The DiT depth sweep above is **unconditional**: it measures memorization,
novelty, and physical statistics while changing architecture and training-set
size. This appendix audits a separate **conditional UNet** experiment used for
the cosmological-parameter recovery figure.

The conditional generator is given the complete six-dimensional CAMELS
parameter vector
$(\Omega_m,\sigma_8,A_{\rm SN1},A_{\rm AGN1},A_{\rm SN2},A_{\rm AGN2})$.
The code below verifies that both normalized and raw vectors have all six
columns and that every held-out condition has the requested number of generated
seeds. The poster's $\Omega_m$ panel is one projection of this full-vector
experiment; it is not evidence that the model was conditioned on $\Omega_m$
alone.

The 16th-to-84th percentile bars summarize variation across generated seeds at
a fixed requested condition. Their inclusion fraction is labeled
**seed-interval inclusion; not posterior coverage** because the seeds are not
samples from a Bayesian posterior over cosmological parameters.

In [ ]:
expected_parameter_count = 6
conditional_parameter_names = [
    'Omega_m', 'sigma_8', 'A_SN1', 'A_AGN1', 'A_SN2', 'A_AGN2'
]
conditional_root = PROJECT_DIR / 'results' / 'nf_conditional_bias_probe'
conditional_sample_root = conditional_root / 'samples'
conditional_manifest_path = PROJECT_DIR / 'local' / 'nf_conditional_bias_probe' / 'manifest.json'

conditional_manifest_rows = []
if conditional_manifest_path.exists():
    conditional_manifest_rows = json.loads(conditional_manifest_path.read_text())
manifest_by_run = {
    str(row.get('run_name')): row for row in conditional_manifest_rows
}

conditional_audit_rows = []
conditional_sample_paths = sorted(conditional_sample_root.glob('*.npz'))
for sample_path in conditional_sample_paths:
    try:
        with np.load(sample_path, allow_pickle=True) as payload:
            files = set(payload.files)
            required = {'samples', 'theta_norm_repeated', 'theta_raw', 'heldout_indices', 'samples_per_cosmology'}
            missing_keys = sorted(required - files)
            if missing_keys:
                raise KeyError('missing arrays: ' + ', '.join(missing_keys))
            samples = np.asarray(payload['samples'])
            theta_norm_repeated = np.asarray(payload['theta_norm_repeated'])
            theta_raw = np.asarray(payload['theta_raw'])
            heldout_indices = np.atleast_1d(np.asarray(payload['heldout_indices']))
            samples_per_cosmology = int(np.asarray(payload['samples_per_cosmology']).item())
            run_name_value = payload['run_name'].item() if 'run_name' in files else sample_path.stem
            run_name = str(run_name_value)
        manifest_row = manifest_by_run.get(run_name, {})
        heldout_manifest_ok = False
        training_heldout_disjoint = False
        if manifest_row:
            heldout_path = Path(str(manifest_row.get('heldout_indices_path', '')))
            pairs_path = Path(str(manifest_row.get('selected_pairs_path', '')))
            if not heldout_path.is_absolute():
                heldout_path = PROJECT_DIR / heldout_path
            if not pairs_path.is_absolute():
                pairs_path = PROJECT_DIR / pairs_path
            if heldout_path.exists() and pairs_path.exists():
                manifest_heldout = np.atleast_1d(np.loadtxt(heldout_path, dtype=np.int64))
                training_pairs = pd.read_csv(pairs_path)
                training_simulations = set(training_pairs['simulation_index'].astype(int))
                heldout_manifest_ok = bool(np.array_equal(heldout_indices, manifest_heldout))
                training_heldout_disjoint = bool(
                    training_simulations.isdisjoint(set(manifest_heldout.astype(int)))
                )
        norm_shape_ok = bool(
            theta_norm_repeated.ndim == 2
            and theta_norm_repeated.shape[1] == expected_parameter_count
        )
        raw_shape_ok = bool(
            theta_raw.ndim == 2
            and theta_raw.shape[1] == expected_parameter_count
        )
        condition_count_ok = bool(
            len(theta_norm_repeated) == len(theta_raw) == len(heldout_indices)
        )
        repetition_ok = bool(
            len(samples) == len(theta_raw) * samples_per_cosmology
        )
        manifest_order = manifest_row.get('param_names', [])
        order_ok = bool(
            not manifest_order or list(manifest_order) == conditional_parameter_names
        )
        conditional_audit_rows.append({
            'run_name': run_name,
            'sample_path': rel(sample_path),
            'theta_norm_shape': tuple(theta_norm_repeated.shape),
            'theta_raw_shape': tuple(theta_raw.shape),
            'n_heldout_conditions': len(theta_raw),
            'samples_per_condition': samples_per_cosmology,
            'n_generated': len(samples),
            'six_normalized_parameters': norm_shape_ok,
            'six_raw_parameters': raw_shape_ok,
            'heldout_alignment_ok': condition_count_ok,
            'sample_repetition_ok': repetition_ok,
            'parameter_order_ok': order_ok,
            'heldout_indices_match_manifest': heldout_manifest_ok,
            'training_and_heldout_simulations_disjoint': training_heldout_disjoint,
            'full_vector_audit_pass': bool(
                norm_shape_ok and raw_shape_ok and condition_count_ok
                and repetition_ok and order_ok and heldout_manifest_ok
                and training_heldout_disjoint
            ),
        })
    except Exception as exc:
        conditional_audit_rows.append({
            'run_name': sample_path.stem,
            'sample_path': rel(sample_path),
            'full_vector_audit_pass': False,
            'error': repr(exc),
        })

conditional_input_audit_df = pd.DataFrame(conditional_audit_rows)
if conditional_input_audit_df.empty:
    display(Markdown(
        'Conditional sample files are not present in this checkout. '
        'Run the conditional pipeline on Great Lakes, then rerun this appendix.'
    ))
else:
    display(conditional_input_audit_df)
    if not conditional_input_audit_df['full_vector_audit_pass'].fillna(False).all():
        display(Markdown('**Conditional input audit failed for at least one file; do not use its calibration result.**'))

calibration_candidates = sorted(
    conditional_root.glob('**/bias_probe_per_cosmology_points.csv'),
    key=lambda path: path.stat().st_mtime if path.exists() else 0,
    reverse=True,
)
conditional_calibration_points = pd.DataFrame()
conditional_calibration_path = None
for candidate in calibration_candidates:
    candidate_df = pd.read_csv(candidate)
    required_columns = {
        'parameter', 'theta_in', 'theta_rec_median', 'theta_rec_q16', 'theta_rec_q84'
    }
    if required_columns.issubset(candidate_df.columns):
        available_parameters = set(candidate_df['parameter'].astype(str))
        if set(conditional_parameter_names).issubset(available_parameters):
            conditional_calibration_points = candidate_df
            conditional_calibration_path = candidate
            break

if conditional_calibration_points.empty:
    display(Markdown(
        'No six-parameter calibration table is available yet. The provenance '
        'audit above remains valid, but no recovery panel is drawn.'
    ))
else:
    display(Markdown(f'Calibration source: `{rel(conditional_calibration_path)}`'))
    fig_conditional, conditional_axes = plt.subplots(2, 3, figsize=(15.6, 9.3))
    inclusion_rows = []
    regime_values = list(dict.fromkeys(
        conditional_calibration_points.get(
            'regime', pd.Series(['all'] * len(conditional_calibration_points))
        ).astype(str)
    ))
    regime_colors = {
        regime: color for regime, color in zip(
            regime_values, ['#D55E00', '#0072B2', '#009E73', '#CC79A7']
        )
    }
    for axis, parameter in zip(conditional_axes.ravel(), conditional_parameter_names):
        parameter_rows = conditional_calibration_points[
            conditional_calibration_points['parameter'].astype(str) == parameter
        ].copy()
        for regime, regime_rows in parameter_rows.groupby(
            parameter_rows.get('regime', pd.Series('all', index=parameter_rows.index)).astype(str),
            sort=False,
        ):
            x = regime_rows['theta_in'].to_numpy(dtype=float)
            median = regime_rows['theta_rec_median'].to_numpy(dtype=float)
            q16 = regime_rows['theta_rec_q16'].to_numpy(dtype=float)
            q84 = regime_rows['theta_rec_q84'].to_numpy(dtype=float)
            order = np.argsort(x)
            axis.errorbar(
                x[order], median[order],
                yerr=np.vstack([median[order] - q16[order], q84[order] - median[order]]),
                fmt='o', ms=4.5, capsize=2.5, color=regime_colors.get(regime, '0.3'),
                alpha=0.82, label=regime,
            )
            included = (x >= q16) & (x <= q84)
            inclusion_rows.append({
                'parameter': parameter,
                'regime': regime,
                'seed_interval_inclusion_fraction': float(np.mean(included)),
                'n_conditions': int(len(included)),
                'diagnostic': 'seed-interval inclusion; not posterior coverage',
            })
        if not parameter_rows.empty:
            limits = np.nanmin(parameter_rows[['theta_in', 'theta_rec_q16']].to_numpy()), np.nanmax(
                parameter_rows[['theta_in', 'theta_rec_q84']].to_numpy()
            )
            axis.plot(limits, limits, color='0.25', ls='--', lw=1.4)
            axis.set_xlim(limits)
            axis.set_ylim(limits)
        axis.set_title(parameter, pad=8)
        axis.set_xlabel('requested value')
        axis.set_ylabel('recovered value')
        axis.grid(alpha=0.15)
        axis.spines['top'].set_visible(False)
        axis.spines['right'].set_visible(False)
    handles, labels = conditional_axes[0, 0].get_legend_handles_labels()
    if handles:
        fig_conditional.legend(
            handles, labels, loc='upper center', bbox_to_anchor=(0.5, 0.93),
            ncol=max(1, len(labels)), frameon=False,
        )
    fig_conditional.suptitle(
        'Conditional recovery uses the full six-parameter input vector',
        fontsize=21, fontweight='semibold', y=0.985,
    )
    fig_conditional.subplots_adjust(
        left=0.07, right=0.98, bottom=0.08, top=0.84, hspace=0.38, wspace=0.28
    )
    conditional_plot_path = QUICKCHECK_DIR / 'nf_conditional_bias_probe_six_parameter_calibration.png'
    fig_conditional.savefig(conditional_plot_path, bbox_inches='tight', dpi=300)
    plt.show()
    print('wrote', conditional_plot_path)
    seed_interval_inclusion_df = pd.DataFrame(inclusion_rows)
    display(Markdown('### Seed-interval inclusion; not posterior coverage'))
    display(seed_interval_inclusion_df)

## Audited DiT-L16 Continuation: 300k to 500k Updates

This section tests whether the fresh DiT-L16 sweep simply needed more optimization. Every one of the ten models is continued from its clean 300k checkpoint to 340k, 380k, 420k, 460k, and 500k optimizer updates with full model, EMA, optimizer, scheduler, scaler, and random-number state restored.

The analysis keeps three questions separate:

1. Does the denoising loss continue to decrease?
2. Do PCA and SSCD novelty curves move with additional training?
3. Do the exact-subset one-point distribution, power spectrum, and patch-boundary diagnostics improve?

The final audit is mandatory. **Do not infer a scaling law** from this section unless every expected checkpoint, sample, metric table, sampler record, and physical-statistics output passes that audit.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

CONT_SWEEP = 'nf_generalize_fig2_dit_l16_continue500k_v2'
CONT_LOCAL_DIR = PROJECT_DIR / 'local' / CONT_SWEEP
CONT_SAMPLE_DIR = PROJECT_DIR / 'results' / CONT_SWEEP / 'samples'
CONT_TABLE_DIR = PROJECT_DIR / 'results' / 'nf_generalize_fig2_dit' / 'tables'
CONT_PHYSICS_DIR = PROJECT_DIR / 'results' / 'nf_generalize_fig2_dit' / 'physics'
CONT_QUICKCHECK_DIR = PROJECT_DIR / 'results' / CONT_SWEEP / 'quickcheck'
CONT_QUICKCHECK_DIR.mkdir(parents=True, exist_ok=True)

CONT_UPDATES_K = [300, 340, 380, 420, 460, 500]
CONT_TAGS = [f'd2p{i:02d}' for i in range(6, 16)]
CONT_FEATURES = ('PCA', 'SSCD')

audit_path = CONT_LOCAL_DIR / 'final_audit.json'
if not audit_path.is_file():
    raise FileNotFoundError(
        f'Missing mandatory final audit: {audit_path}. Run the sweep audit before this notebook.'
    )
continuation_audit = json.loads(audit_path.read_text())
if continuation_audit.get('status') != 'PASS':
    raise RuntimeError(
        'DiT-L16 continuation audit did not pass:\n' +
        json.dumps(continuation_audit, indent=2, sort_keys=True)
    )
display(Markdown('### Final artifact audit: **PASS**'))
display(pd.json_normalize(continuation_audit, sep='.'))

analysis_manifest_path = CONT_LOCAL_DIR / 'analysis_manifest.json'
continuation_manifest = pd.DataFrame(json.loads(analysis_manifest_path.read_text()))
continuation_manifest['analysis_updates'] = continuation_manifest['analysis_updates'].astype(int)
continuation_manifest['updates_k'] = continuation_manifest['analysis_updates'] // 1000
expected_pairs = {(tag, updates) for tag in CONT_TAGS for updates in CONT_UPDATES_K}
actual_pairs = set(zip(continuation_manifest['dataset_tag'], continuation_manifest['updates_k']))
if len(continuation_manifest) != 60 or actual_pairs != expected_pairs:
    raise RuntimeError('analysis_manifest.json does not contain all 60 dataset/checkpoint rows')

def continuation_row(tag: str, updates_k: int) -> pd.Series:
    rows = continuation_manifest[
        (continuation_manifest['dataset_tag'] == tag) &
        (continuation_manifest['updates_k'] == int(updates_k))
    ]
    if len(rows) != 1:
        raise RuntimeError(f'Expected one manifest row for {tag} at {updates_k}k; found {len(rows)}')
    return rows.iloc[0]

continuation_novelty_frames = []
for feature in CONT_FEATURES:
    for updates_k in CONT_UPDATES_K:
        path = CONT_TABLE_DIR / f'{CONT_SWEEP}_{updates_k}k_{feature.lower()}_full_nn_metrics.csv'
        frame = pd.read_csv(path)
        frame = add_generalization_columns(frame)
        if len(frame) != 10 or set(frame['dataset_tag']) != set(CONT_TAGS):
            raise RuntimeError(f'Incomplete novelty table: {path}')
        frame['feature'] = feature
        frame['updates_k'] = updates_k
        continuation_novelty_frames.append(frame)
continuation_novelty = pd.concat(continuation_novelty_frames, ignore_index=True)

physics_summary_path = CONT_TABLE_DIR / f'{CONT_SWEEP}_physics_summary.csv'
selected_bins_path = CONT_TABLE_DIR / f'{CONT_SWEEP}_pk_selected_bins.csv'
patch_table_path = CONT_TABLE_DIR / f'{CONT_SWEEP}_patch_boundaries.csv'
physics_curves_path = CONT_PHYSICS_DIR / f'{CONT_SWEEP}_curves.npz'
continuation_physics = pd.read_csv(physics_summary_path)
continuation_selected_bins = pd.read_csv(selected_bins_path)
continuation_patch = pd.read_csv(patch_table_path)
continuation_curves = np.load(physics_curves_path)

if len(continuation_physics) != 60:
    raise RuntimeError(f'Expected 60 physics rows; found {len(continuation_physics)}')
if len(continuation_selected_bins) != 180:
    raise RuntimeError(f'Expected 180 selected-k rows; found {len(continuation_selected_bins)}')

display(Markdown(
    f'Loaded **{len(continuation_manifest)}** manifest rows, '
    f'**{len(continuation_novelty)}** novelty rows, '
    f'**{len(continuation_physics)}** physical-summary rows, and '
    f'**{len(continuation_selected_bins)}** selected-k rows.'
))


### Training loss from 300k to 500k

Each panel is one training-set size. The curves show the cycle-averaged denoising objective over the continuation interval, with vertical guides at the six sampled checkpoints. A declining loss establishes that optimization continued; it does not by itself establish novelty or physical validity.


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 8.6), sharex=True, constrained_layout=True)
loss_audit_rows = []
for axis, tag in zip(axes.flat, CONT_TAGS):
    row = continuation_row(tag, 500)
    metrics, metrics_path = read_latest_metrics(row)
    epoch_loss = flatten_numeric(metrics.get('epoch_loss'))
    steps_per_epoch = int(row.get('steps_per_epoch', 1) or 1)
    update_axis, smoothed_loss = cycle_average_epoch_loss(
        epoch_loss, steps_per_epoch, restart_updates=4000
    )
    keep = (update_axis >= 295_000) & (update_axis <= 505_000)
    if not np.any(keep):
        raise RuntimeError(f'No 300k-500k loss history found for {tag}: {metrics_path}')
    axis.plot(update_axis[keep] / 1000, smoothed_loss[keep], color='#b83280', lw=2.2)
    for updates_k in CONT_UPDATES_K:
        axis.axvline(updates_k, color='0.78', lw=0.8, zorder=0)
    axis.set_yscale('log')
    axis.set_title(dataset_size_label(int(row['dataset_size'])), fontsize=15)
    axis.set_xlabel('Optimizer updates (thousands)')
    axis.grid(alpha=0.2)
    loss_audit_rows.append({
        'dataset_tag': tag,
        'dataset_size': int(row['dataset_size']),
        'history_last_update': float(np.nanmax(update_axis)),
        'loss_near_300k': float(smoothed_loss[keep][0]),
        'loss_near_500k': float(smoothed_loss[keep][-1]),
        'relative_change': float(smoothed_loss[keep][-1] / smoothed_loss[keep][0] - 1),
        'metrics_path': str(metrics_path),
    })
for axis in axes[:, 0]:
    axis.set_ylabel('Cycle-averaged denoising loss')
fig.suptitle('Fresh DiT-L16 optimization after the 300k checkpoint', fontsize=21, fontweight='semibold')
loss_path = CONT_QUICKCHECK_DIR / 'dit_l16_continue500k_loss.png'
fig.savefig(loss_path, dpi=250, bbox_inches='tight')
plt.show()
display(pd.DataFrame(loss_audit_rows))
print('wrote', loss_path)


### PCA and SSCD novelty

Each colored line is one optimizer checkpoint. The complete $2^6$ through $2^{15}$ sweep is retained, so any movement of the transition can be compared directly. The dotted line is a visual 0.5 reference, not a fitted phase boundary. High novelty can still describe an out-of-distribution or physically incorrect sample.


In [ ]:
checkpoint_colors = plt.cm.viridis(np.linspace(0.08, 0.92, len(CONT_UPDATES_K)))
fig, axes = plt.subplots(1, 2, figsize=(16.5, 6.4), sharey=True, constrained_layout=True)
for axis, feature in zip(axes, CONT_FEATURES):
    feature_frame = continuation_novelty[continuation_novelty['feature'] == feature]
    for color, updates_k in zip(checkpoint_colors, CONT_UPDATES_K):
        current = feature_frame[feature_frame['updates_k'] == updates_k].sort_values('dataset_size')
        axis.plot(
            np.log2(current['dataset_size']), current['gen_gl_q95'],
            marker='o', ms=5.5, lw=2.2, color=color, label=f'{updates_k}k'
        )
    axis.axhline(0.5, color='0.35', lw=1.2, ls=':')
    axis.set_title(f'{feature} q95 novelty', fontsize=18, fontweight='semibold')
    axis.set_xlabel(r'Training images $N_{2D}$')
    axis.set_xticks(range(6, 16), [rf'$2^{{{i}}}$' for i in range(6, 16)])
    axis.set_ylim(-0.03, 1.04)
    axis.grid(alpha=0.2)
axes[0].set_ylabel('q95 novelty score')
axes[1].legend(title='Optimizer updates', ncol=2, frameon=False, loc='lower right')
fig.suptitle('DiT-L16 novelty across the full continuation trajectory', fontsize=22, fontweight='semibold')
novelty_path = CONT_QUICKCHECK_DIR / 'dit_l16_continue500k_novelty.png'
fig.savefig(novelty_path, dpi=250, bbox_inches='tight')
plt.show()
print('wrote', novelty_path)


### One-point and power-spectrum trajectories

The black one-point and power-spectrum references are recomputed from the **exact training subset configured for each model**. The heatmaps below summarize all ten data sizes and all six checkpoints. Lower error is better. This makes it possible to see whether longer training repairs the intermediate-data failure or only changes novelty.


In [ ]:
def _metric_heatmap(axis, frame, column, title, colorbar_label):
    matrix = (
        frame.pivot(index='updates_k', columns='dataset_size', values=column)
        .reindex(index=CONT_UPDATES_K, columns=[2**i for i in range(6, 16)])
    )
    image = axis.imshow(matrix.to_numpy(dtype=float), aspect='auto', origin='lower', cmap='magma')
    axis.set_title(title, fontsize=17, fontweight='semibold')
    axis.set_xlabel(r'Training images $N_{2D}$')
    axis.set_ylabel('Optimizer updates')
    axis.set_xticks(range(10), [rf'$2^{{{i}}}$' for i in range(6, 16)])
    axis.set_yticks(range(6), [f'{value}k' for value in CONT_UPDATES_K])
    plt.colorbar(image, ax=axis, shrink=0.85, label=colorbar_label)

fig, axes = plt.subplots(1, 2, figsize=(17, 6.4), constrained_layout=True)
_metric_heatmap(axes[0], continuation_physics, 'hist_l1', 'One-point PDF error', r'$L_1$ error')
_metric_heatmap(axes[1], continuation_physics, 'pk_log10_mae', r'Power-spectrum error', r'mean $|\log_{10}(P_g/P_r)|$')
fig.suptitle('Physical agreement over data size and training time', fontsize=22, fontweight='semibold')
physical_heatmap_path = CONT_QUICKCHECK_DIR / 'dit_l16_continue500k_physical_heatmaps.png'
fig.savefig(physical_heatmap_path, dpi=250, bbox_inches='tight')
plt.show()
print('wrote', physical_heatmap_path)

hist_edges = continuation_curves['histogram_edges']
hist_centers = 0.5 * (hist_edges[:-1] + hist_edges[1:])

def plot_physical_checkpoint(updates_k: int):
    fig, axes = plt.subplots(4, 5, figsize=(19, 13.5), constrained_layout=True)
    for column, tag in enumerate(CONT_TAGS[:5]):
        for block, current_tag in enumerate((tag, CONT_TAGS[column + 5])):
            row = 2 * block
            key = f'{current_tag}_{updates_k}k'
            axes[row, column].plot(hist_centers, continuation_curves[f'{key}_real_hist_probability'], color='black', lw=2, label='exact training subset')
            axes[row, column].plot(hist_centers, continuation_curves[f'{key}_generated_hist_probability'], color='#b83280', lw=2, label='generated')
            axes[row, column].set_yscale('log')
            axes[row, column].set_title(dataset_size_label(2 ** (column + 6 + 5 * block)))
            axes[row, column].set_xlabel('Normalized field value')
            kbins = continuation_curves[f'{key}_kbins']
            axes[row + 1, column].plot(kbins, continuation_curves[f'{key}_pk_ratio'], color='#b83280', marker='o', ms=2.8, lw=1.8)
            axes[row + 1, column].axhline(1.0, color='black', lw=1.1, ls='--')
            axes[row + 1, column].set_ylim(0, 3.7)
            axes[row + 1, column].set_xlabel(r'$k$ bin')
    axes[0, 0].set_ylabel('Pixel probability')
    axes[1, 0].set_ylabel(r'$P_{generated}/P_{real}$')
    axes[2, 0].set_ylabel('Pixel probability')
    axes[3, 0].set_ylabel(r'$P_{generated}/P_{real}$')
    axes[0, 0].legend(frameon=False, fontsize=10)
    fig.suptitle(f'Fresh DiT-L16 exact-subset physical checks at {updates_k}k updates', fontsize=22, fontweight='semibold')
    path = CONT_QUICKCHECK_DIR / f'dit_l16_continue500k_physics_{updates_k}k.png'
    fig.savefig(path, dpi=230, bbox_inches='tight')
    plt.show()
    print('wrote', path)

plot_physical_checkpoint(300)
plot_physical_checkpoint(500)


### Scale-resolved uncertainty at k-bin 20, 40, and 60

The left row shows the mean generated-to-real power ratio with bootstrap uncertainty. The right row shows the variance across the 512 generated inference samples. A mean near one can conceal excessive sample-to-sample scatter, so both quantities are required.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10), constrained_layout=True)
for column, k_bin in enumerate((20, 40, 60)):
    selected = continuation_selected_bins[continuation_selected_bins['k_bin'] == k_bin]
    for color, updates_k in zip(checkpoint_colors, CONT_UPDATES_K):
        current = selected[selected['updates_k'] == updates_k].sort_values('dataset_size')
        x = np.log2(current['dataset_size'])
        mean = current['ratio_mean'].to_numpy(dtype=float)
        low = current['ratio_mean_ci_low'].to_numpy(dtype=float)
        high = current['ratio_mean_ci_high'].to_numpy(dtype=float)
        axes[0, column].plot(x, mean, marker='o', ms=4.5, color=color, lw=1.8, label=f'{updates_k}k')
        axes[0, column].fill_between(x, low, high, color=color, alpha=0.12)
        axes[1, column].plot(x, current['ratio_variance'], marker='o', ms=4.5, color=color, lw=1.8)
    axes[0, column].axhline(1.0, color='black', lw=1.1, ls='--')
    axes[0, column].set_title(f'k-bin {k_bin}', fontsize=16, fontweight='semibold')
    axes[1, column].set_xlabel(r'Training images $N_{2D}$')
    for axis in axes[:, column]:
        axis.set_xticks(range(6, 16), [rf'$2^{{{i}}}$' for i in range(6, 16)])
        axis.grid(alpha=0.2)
axes[0, 0].set_ylabel(r'Mean $P_{generated}/P_{real}$')
axes[1, 0].set_ylabel('Variance across generated samples')
axes[0, 2].legend(title='Updates', ncol=2, frameon=False)
fig.suptitle('Scale-resolved power-spectrum accuracy and inference variance', fontsize=22, fontweight='semibold')
selected_path = CONT_QUICKCHECK_DIR / 'dit_l16_continue500k_selected_kbins.png'
fig.savefig(selected_path, dpi=250, bbox_inches='tight')
plt.show()
print('wrote', selected_path)


### DPM-Solver 50 versus DDPM 500

This controlled comparison uses the same resolved checkpoint and seed for $2^8$ and $2^{11}$ at 300k and 500k. It tests whether the visible artifacts or power-spectrum errors are caused by the 50-step sampler. Scheduler metadata and the executed terminal step are displayed with the curves.


In [ ]:
def _npz_scalar(payload, key, default=None):
    if key not in payload.files:
        return default
    value = np.asarray(payload[key])
    return value.item() if value.size == 1 else value.tolist()

sampler_rows = []
fig, axes = plt.subplots(2, 2, figsize=(14, 10), constrained_layout=True)
for axis, (tag, updates_k) in zip(axes.flat, [(tag, updates) for tag in ('d2p08', 'd2p11') for updates in (300, 500)]):
    row = continuation_row(tag, updates_k)
    dpm_path = Path(row['sample_path'])
    if not dpm_path.is_absolute():
        dpm_path = PROJECT_DIR / dpm_path
    run_name = str(row['run_name'])
    ddpm_label = f'ddpm500_{"source_300k" if updates_k == 300 else "cont_500k"}'
    ddpm_path = CONT_SAMPLE_DIR / f'{run_name}_seed123_{ddpm_label}.npz'
    key = f'{tag}_{updates_k}k'
    real_pk = continuation_curves[f'{key}_real_pk_mean']
    kbins = continuation_curves[f'{key}_kbins']
    for path, label, color in ((dpm_path, 'DPM-Solver 50', '#0072B2'), (ddpm_path, 'DDPM 500', '#D55E00')):
        samples = load_npz_array(path)
        spectra, current_kbins = batch_power_spectra(samples, nbins=len(real_pk))
        if not np.allclose(current_kbins, kbins, equal_nan=True):
            raise RuntimeError(f'k-bin mismatch for {path}')
        ratio = np.nanmean(spectra, axis=0) / np.clip(real_pk, 1e-30, None)
        axis.plot(kbins, ratio, marker='o', ms=3, lw=2, color=color, label=label)
        with np.load(path, allow_pickle=False) as payload:
            sampler_rows.append({
                'dataset_tag': tag,
                'updates_k': updates_k,
                'sampler': label,
                'scheduler': _npz_scalar(payload, 'scheduler', 'missing'),
                'executed_inference_steps': _npz_scalar(payload, 'executed_inference_steps', 'missing'),
                'terminal_sigma': _npz_scalar(payload, 'terminal_sigma', 'missing'),
                'terminal_sigma_verifiable': _npz_scalar(payload, 'terminal_sigma_verifiable', 'missing'),
                'resolved_checkpoint': _npz_scalar(payload, 'resolved_checkpoint', 'missing'),
                'sample_path': str(path),
            })
    axis.axhline(1, color='black', lw=1.1, ls='--')
    axis.set_title(f'{dataset_size_label(int(row["dataset_size"]))}, {updates_k}k updates')
    axis.set_xlabel(r'$k$ bin')
    axis.set_ylabel(r'$P_{generated}/P_{real}$')
    axis.grid(alpha=0.2)
axes[0, 0].legend(frameon=False)
fig.suptitle('Controlled sampler check', fontsize=22, fontweight='semibold')
sampler_path = CONT_QUICKCHECK_DIR / 'dit_l16_continue500k_sampler_control.png'
fig.savefig(sampler_path, dpi=250, bbox_inches='tight')
plt.show()
sampler_audit = pd.DataFrame(sampler_rows)
display(sampler_audit)
print('wrote', sampler_path)


### Patch-boundary diagnostic

DiT-L8, DiT-L12, and DiT-L16 all use patch size 8. This diagnostic measures discontinuity at patch boundaries relative to interior neighboring pixels. The real-reference line and 200k L8/L12 curves provide scale controls; the L16 heatmap shows whether the checkerboard pattern changes from 300k to 500k.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(17, 6.2), constrained_layout=True)
line_specs = [
    ('real_reference', None, 'exact real subset', 'black', '-'),
    ('dit_l8', 200, 'DiT-L8, 200k', '#009E73', '--'),
    ('dit_base', 200, 'DiT-L12, 200k', '#0072B2', '--'),
    ('dit_l16', 300, 'DiT-L16, 300k', '#CC79A7', '-'),
    ('dit_l16', 500, 'DiT-L16, 500k', '#8E2A68', '-'),
]
for architecture, updates_k, label, color, linestyle in line_specs:
    current = continuation_patch[continuation_patch['architecture'] == architecture].copy()
    if updates_k is not None:
        current = current[current['updates_k'] == updates_k]
    current = current.sort_values('dataset_size')
    axes[0].plot(np.log2(current['dataset_size']), current['patch_boundary_ratio'], marker='o', lw=2, color=color, ls=linestyle, label=label)
axes[0].axhline(1, color='0.5', lw=1, ls=':')
axes[0].set_title('Architecture and real-reference comparison', fontsize=16, fontweight='semibold')
axes[0].set_xlabel(r'Training images $N_{2D}$')
axes[0].set_ylabel('Patch-boundary / interior discontinuity')
axes[0].set_xticks(range(6, 16), [rf'$2^{{{i}}}$' for i in range(6, 16)])
axes[0].legend(frameon=False, fontsize=10)
axes[0].grid(alpha=0.2)

l16_patch = continuation_patch[continuation_patch['architecture'] == 'dit_l16']
matrix = l16_patch.pivot(index='updates_k', columns='dataset_size', values='patch_boundary_ratio').reindex(index=CONT_UPDATES_K, columns=[2**i for i in range(6, 16)])
image = axes[1].imshow(matrix.to_numpy(dtype=float), aspect='auto', origin='lower', cmap='coolwarm')
axes[1].set_title('DiT-L16 continuation', fontsize=16, fontweight='semibold')
axes[1].set_xlabel(r'Training images $N_{2D}$')
axes[1].set_ylabel('Optimizer updates')
axes[1].set_xticks(range(10), [rf'$2^{{{i}}}$' for i in range(6, 16)])
axes[1].set_yticks(range(6), [f'{value}k' for value in CONT_UPDATES_K])
plt.colorbar(image, ax=axes[1], shrink=0.86, label='Boundary ratio')
fig.suptitle('Patch-boundary audit (patch size 8)', fontsize=22, fontweight='semibold')
patch_path = CONT_QUICKCHECK_DIR / 'dit_l16_continue500k_patch_boundary.png'
fig.savefig(patch_path, dpi=250, bbox_inches='tight')
plt.show()
print('wrote', patch_path)


### Generated samples versus nearest training slices

For the transition candidate $2^8$ and the higher-data control $2^{11}$, four generated samples are compared with their nearest training slice at 300k and 500k. A nearly blank absolute-difference map indicates copying. A large difference establishes novelty only; it must be read together with the physical-statistics and sampler panels above.


In [ ]:
nearest_summary_rows = []
for tag in ('d2p08', 'd2p11'):
    for updates_k in (300, 500):
        row = continuation_row(tag, updates_k)
        sample_path = Path(row['sample_path'])
        if not sample_path.is_absolute():
            sample_path = PROJECT_DIR / sample_path
        config_value = row.get('source_config') if updates_k == 300 else row.get('config')
        config_path = Path(str(config_value))
        if not config_path.is_absolute():
            config_path = PROJECT_DIR / config_path
        generated = load_npz_array(sample_path)
        training = load_real_reference_from_config(config_path, max_slices=None)
        matches = nearest_training_matches(
            generated, training, max_generated=4, max_training=None, training_chunk=256
        )
        generated_index = np.asarray(matches['generated_index'], dtype=int)
        nearest_index = np.asarray(matches['nearest_training_index'], dtype=int)
        selected_generated = generated[generated_index, 0]
        selected_training = training[nearest_index, 0]
        difference = np.abs(selected_generated - selected_training)

        fig, axes = plt.subplots(4, 3, figsize=(10.5, 13.5), constrained_layout=True)
        for index in range(4):
            combined = np.concatenate([selected_generated[index].ravel(), selected_training[index].ravel()])
            vmin, vmax = np.quantile(combined, [0.005, 0.995])
            dmax = max(float(np.quantile(difference[index], 0.995)), 1e-8)
            axes[index, 0].imshow(selected_generated[index], cmap='viridis', vmin=vmin, vmax=vmax)
            axes[index, 1].imshow(selected_training[index], cmap='viridis', vmin=vmin, vmax=vmax)
            axes[index, 2].imshow(difference[index], cmap='magma', vmin=0, vmax=dmax)
            axes[index, 0].set_ylabel(f'generated {generated_index[index]}', fontweight='bold')
            axes[index, 2].text(
                0.03, 0.03,
                f'MSE={matches["nearest_mse"][index]:.3g}; cos={matches["nearest_cosine"][index]:.3f}',
                transform=axes[index, 2].transAxes, color='white', fontsize=9,
                bbox={'facecolor': 'black', 'alpha': 0.62, 'pad': 2},
            )
            for axis in axes[index]:
                axis.set_xticks([])
                axis.set_yticks([])
            nearest_summary_rows.append({
                'dataset_tag': tag,
                'updates_k': updates_k,
                'generated_index': int(generated_index[index]),
                'nearest_training_index': int(nearest_index[index]),
                'nearest_mse': float(matches['nearest_mse'][index]),
                'nearest_cosine': float(matches['nearest_cosine'][index]),
            })
        for axis, title in zip(axes[0], ('generated', 'nearest training', 'absolute difference')):
            axis.set_title(title, fontsize=14, fontweight='semibold')
        fig.suptitle(f'DiT-L16 {dataset_size_label(int(row["dataset_size"]))}: nearest-training audit at {updates_k}k', fontsize=19, fontweight='semibold')
        path = CONT_QUICKCHECK_DIR / f'dit_l16_{tag}_{updates_k}k_nearest_training.png'
        fig.savefig(path, dpi=230, bbox_inches='tight')
        plt.show()
        print('wrote', path)

nearest_summary = pd.DataFrame(nearest_summary_rows)
display(nearest_summary)


### Interpretation checklist

- If loss falls while one-point and $P(k)$ errors remain flat, insufficient optimizer updates alone are not the explanation.
- If DPM-Solver 50 and DDPM 500 agree at the same checkpoint, sampler truncation is not the main cause of the artifacts.
- If patch-boundary ratios are elevated specifically for DiT-L16, investigate patch-token optimization or reconstruction rather than the CAMELS preprocessing.
- If the nearest-training difference is large but physical error is also large, the sample is novel but invalid; it must not be counted as successful generalization.
- Only a repeated, physically valid shift of the PCA and SSCD curves supports a depth-dependent transition. **Do not infer a scaling law** merely because the novelty score crosses 0.5.


## Great Lakes Rerun Command

From the repo root on Great Lakes:

```bash
cd /home/jiamingp/diffusion_models_repo
jupyter nbconvert --execute --to notebook --inplace notebooks/nf_generalize_fig2_dit_results.ipynb
```

If you are using the Jupyter web session, just open this notebook and run all cells. The heavy PCA/SSCD nearest-neighbor work should not rerun here; this notebook reads the completed CSV and PNG outputs.
